In [1]:
"""
STAGE 1: Data Preprocessing & COCO Conversion
==========================================================
Converts LabelMe annotations to COCO format
Preprocesses metadata (23 features, NO center, derived class labels)
Creates PATIENT-LEVEL, CENTER-AWARE stratified train/val/test splits

FIXES APPLIED:
✅ Patient-level splitting (prevents data leakage)
✅ Center-aware stratification (controls center bias)
✅ Label-free patient fingerprint (no tumor/benign/malignant)
✅ Includes normal images (tumor=0) with zero annotations

IMPORTANT NOTES:
1. Patient grouping is APPROXIMATED (no explicit patient IDs available)
2. This is PATIENT-LEVEL classification (not lesion or image level)
3. Center used ONLY for stratification (excluded from model inputs)
4. Normal images included for realistic class imbalance

Dataset: BTXRD Bone Tumor X-ray Dataset
"""

import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for Stage 1 preprocessing"""
    
    RAW_IMAGES_DIR = "/kaggle/input/datasets/sadib2026/btxrd-with-mask/btxrd_with_mask/images"
    RAW_MASKS_DIR = "/kaggle/input/btxrd-with-mask/btxrd_with_mask/masks"
    RAW_ANNOTATIONS_DIR = "/kaggle/input/datasets/sadib2026/btxrd-with-mask/btxrd_with_mask/Annotations"
    METADATA_FILE = "/kaggle/input/datasets/sadib2026/btxrd-with-mask/btxrd_with_mask/dataset.xlsx"
    
    OUTPUT_DIR = "preprocessed"
    
    # Splits (70% train, 15% val, 15% test)
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    RANDOM_SEED = 42
    
    # Metadata features: 23 features (NO center!)
    METADATA_FEATURES = [
        # Demographics (2 features)
        'age', 'gender',
        
        # Bone locations (9 features)
        'hand', 'ulna', 'radius', 'humerus', 'foot', 
        'tibia', 'fibula', 'femur', 'hip bone',
        
        # Joint involvement (6 features)
        'ankle-joint', 'knee-joint', 'hip-joint', 
        'wrist-joint', 'elbow-joint', 'shoulder-joint',
        
        # Body regions (3 features)
        'upper limb', 'lower limb', 'pelvis',
        
        # X-ray view (3 features)
        'frontal', 'lateral', 'oblique'
    ]
    
    # Label derivation logic:
    # tumor=0 → class_label=0 (Normal)
    # tumor=1, benign=1 → class_label=1 (Benign)
    # tumor=1, malignant=1 → class_label=2 (Malignant)
    CLASS_NAMES = ['Normal', 'Benign', 'Malignant']
    
    # ✅ FIXED: COCO categories (category_id MUST be 1, not 0)
    # Detectron2 internally remaps to 0-indexed, but COCO format requires 1-indexed
    COCO_CATEGORIES = [
        {"id": 1, "name": "tumor", "supercategory": "lesion"}
    ]


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def create_directory_structure():
    """Create output directory structure"""
    dirs = [
        Config.OUTPUT_DIR,
        f"{Config.OUTPUT_DIR}/coco_annotations",
        f"{Config.OUTPUT_DIR}/metadata_processed",
        f"{Config.OUTPUT_DIR}/splits",
        f"{Config.OUTPUT_DIR}/logs"
    ]
    for d in dirs:
        os.makedirs(d, exist_ok=True)
    print("✅ Directory structure created")


def polygon_to_bbox(points):
    """Convert polygon points to bounding box [x, y, width, height]"""
    points = np.array(points)
    x_min, y_min = points[:, 0].min(), points[:, 1].min()
    x_max, y_max = points[:, 0].max(), points[:, 1].max()
    width = x_max - x_min
    height = y_max - y_min
    return [float(x_min), float(y_min), float(width), float(height)]


def polygon_to_segmentation(points):
    """Convert polygon points to COCO segmentation format"""
    return [float(coord) for point in points for coord in point]


def compute_area(points):
    """Compute polygon area using shoelace formula"""
    points = np.array(points)
    x = points[:, 0]
    y = points[:, 1]
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


# ============================================================================
# ANNOTATION CONVERSION (LabelMe → COCO) - **INCLUDES NORMAL IMAGES**
# ============================================================================

def convert_labelme_to_coco(image_ids, split_name):
    """
    Convert LabelMe annotations to COCO format for given image_ids
    
    ✅ FIXED: category_id = 1 (COCO standard)
           Detectron2 internally remaps to 0-indexed for training
    ✅ NEW: Includes normal images (tumor=0) with zero annotations
    ✅ NOTE: image_id is split-local (valid since each split has separate COCO JSON)
    
    Args:
        image_ids: List of image filenames
        split_name: 'train', 'val', or 'test'
    
    Returns:
        dict: COCO format annotations
    """
    
    coco_output = {
        "info": {
            "description": "BTXRD Bone Tumor Dataset",
            "version": "1.0",
            "year": 2026,
            "contributor": "BTXRD Team",
            "date_created": "2026-01-19"
        },
        "licenses": [],
        "categories": Config.COCO_CATEGORIES,
        "images": [],
        "annotations": []
    }
    
    annotation_id = 1
    skipped_images = []
    skipped_reasons = {"no_image": 0}
    normal_images_count = 0  # Track normal images
    
    print(f"\n🔄 Converting {split_name} set: {len(image_ids)} images")
    
    for idx, image_id in enumerate(tqdm(image_ids, desc=f"Processing {split_name}")):
        json_path = Path(Config.RAW_ANNOTATIONS_DIR) / f"{Path(image_id).stem}.json"
        img_path = Path(Config.RAW_IMAGES_DIR) / image_id
        
        # Check image file exists (don't skip if no JSON!)
        if not img_path.exists():
            skipped_images.append(image_id)
            skipped_reasons["no_image"] += 1
            continue
        
        # Get image dimensions
        try:
            img = Image.open(img_path)
            width, height = img.size
            img.close()
        except Exception as e:
            print(f"⚠️  Error opening image {img_path}: {e}")
            skipped_images.append(image_id)
            skipped_reasons["no_image"] += 1
            continue
        
        # ✅ CLARIFICATION: image_id is split-local (idx + 1)
        # This is valid because each split (train/val/test) has its own COCO JSON
        # Global uniqueness is not required across splits
        image_info = {
            "id": idx + 1,  # Split-local ID (starts at 1 per COCO convention)
            "file_name": image_id,
            "width": width,
            "height": height
        }
        coco_output["images"].append(image_info)
        
        # Process annotations IF JSON exists
        if json_path.exists():
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    labelme_data = json.load(f)
            except Exception as e:
                print(f"⚠️  Error reading {json_path}: {e}")
                # Image still added above (normal case)
                normal_images_count += 1
                continue
            
            polygon_count = 0
            for shape in labelme_data.get("shapes", []):
                if shape["shape_type"] != "polygon":
                    continue
                
                points = shape["points"]
                if len(points) < 3:
                    continue
                
                try:
                    bbox = polygon_to_bbox(points)
                    segmentation = [polygon_to_segmentation(points)]
                    area = compute_area(points)
                except Exception as e:
                    print(f"⚠️  Error processing polygon in {image_id}: {e}")
                    continue
                
                if area < 100:
                    continue
                
                # ✅ FIXED: category_id = 1 (COCO standard)
                # Detectron2 will internally remap this to 0 during training
                annotation = {
                    "id": annotation_id,
                    "image_id": idx + 1,  # References split-local image_id
                    "category_id": 1,  # ✅ 1 for COCO standard (not 0!)
                    "bbox": bbox,
                    "segmentation": segmentation,
                    "area": float(area),
                    "iscrowd": 0
                }
                coco_output["annotations"].append(annotation)
                annotation_id += 1
                polygon_count += 1
            
            if polygon_count == 0:
                # Image with JSON but no valid polygons → likely normal
                normal_images_count += 1
        else:
            # No JSON → normal image (tumor=0)
            normal_images_count += 1
    
    output_path = Path(Config.OUTPUT_DIR) / "coco_annotations" / f"{split_name}.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, indent=2)
    
    # Improved statistics
    images_with_annotations = len(set(ann['image_id'] for ann in coco_output['annotations']))
    
    print(f"✅ {split_name}.json saved:")
    print(f"   Total images: {len(coco_output['images'])}")
    print(f"   Images with annotations: {images_with_annotations}")
    print(f"   Images without annotations (normal): {normal_images_count}")
    print(f"   Total annotations: {len(coco_output['annotations'])}")
    print(f"   ℹ️  Note: Normal images included for realistic class imbalance")
    
    if skipped_images:
        print(f"⚠️  Skipped {len(skipped_images)} images:")
        print(f"   - No image file: {skipped_reasons['no_image']}")
        
        skipped_path = Path(Config.OUTPUT_DIR) / "logs" / f"skipped_{split_name}.txt"
        with open(skipped_path, 'w') as f:
            f.write('\n'.join(skipped_images))
    
    return coco_output


# ============================================================================
# PATIENT GROUPING (LABEL-FREE)
# ============================================================================

def create_patient_groups(metadata_df):
    """
    Group images by patient using ONLY non-label features
    
    ⚠️ IMPORTANT LIMITATION:
    Because explicit patient identifiers are unavailable in this dataset,
    patient grouping is APPROXIMATED using demographic and anatomical metadata.
    This may result in limited patient ambiguity in rare edge cases.
    
    Patient fingerprint includes:
    ✅ center (institutional identifier)
    ✅ age (demographic proxy)
    ✅ gender (demographic proxy)
    ✅ anatomy_fingerprint (body location)
    ✅ joint_fingerprint (joint involvement)
    
    ❌ EXCLUDES (prevents label leakage):
    ❌ tumor, benign, malignant (diagnostic labels)
    
    📌 FOR PUBLICATION:
    Add this to Methods section:
    "Because explicit patient identifiers were unavailable, patient grouping was
    approximated using demographic and anatomical metadata, which may result in
    limited patient ambiguity in rare cases."
    
    Returns:
        DataFrame with patient_id column
    """
    
    df = metadata_df.copy()
    
    # Verify required columns
    required_cols = ['center', 'age', 'gender']
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        print(f"❌ ERROR: Missing required columns: {missing}")
        return None
    
    # Create anatomical fingerprint (bone location)
    anatomy_cols = ['hand', 'ulna', 'radius', 'humerus', 'foot', 
                    'tibia', 'fibula', 'femur', 'hip bone']
    available_anatomy = [col for col in anatomy_cols if col in df.columns]
    
    if not available_anatomy:
        print(f"⚠️  WARNING: No anatomy columns found!")
        df['anatomy_fingerprint'] = '0'
    else:
        df['anatomy_fingerprint'] = df[available_anatomy].astype(str).agg(''.join, axis=1)
    
    # Create joint fingerprint (joint involvement)
    joint_cols = ['ankle-joint', 'knee-joint', 'hip-joint', 
                  'wrist-joint', 'elbow-joint', 'shoulder-joint']
    available_joints = [col for col in joint_cols if col in df.columns]
    
    if not available_joints:
        print(f"⚠️  WARNING: No joint columns found!")
        df['joint_fingerprint'] = '0'
    else:
        df['joint_fingerprint'] = df[available_joints].astype(str).agg(''.join, axis=1)
    
    # ✅ Patient fingerprint WITHOUT labels (prevents leakage)
    df['patient_fingerprint'] = (
        df['center'].astype(str) + '_' +
        df['age'].astype(str) + '_' +
        df['gender'] + '_' +
        df['anatomy_fingerprint'] + '_' +
        df['joint_fingerprint']
    )
    
    # Assign patient IDs
    patient_id = 0
    patient_mapping = {}
    
    for fingerprint, group in df.groupby('patient_fingerprint'):
        patient_id += 1
        for img_id in group['image_id']:
            patient_mapping[img_id] = patient_id
    
    df['patient_id'] = df['image_id'].map(patient_mapping)
    
    # Statistics
    print(f"\n📊 Patient grouping statistics:")
    print(f"   Total images: {len(df)}")
    print(f"   Unique patients: {df['patient_id'].nunique()}")
    print(f"   Avg images per patient: {len(df) / df['patient_id'].nunique():.2f}")
    
    # Multi-view patients
    patient_counts = df.groupby('patient_id').size()
    multi_image_patients = (patient_counts > 1).sum()
    print(f"   Patients with multiple views: {multi_image_patients}")
    
    # Center distribution
    print(f"\n   Center distribution (by patient):")
    center_dist = df.groupby('center')['patient_id'].nunique()
    for center, count in center_dist.items():
        pct = count / df['patient_id'].nunique() * 100
        img_count = len(df[df['center'] == center])
        print(f"     Center {center}: {count} patients, {img_count} images ({pct:.1f}%)")
    
    # ✅ Collision analysis (explicitly report potential ambiguity)
    print(f"\n   Patient identity collision analysis:")
    df['weak_fingerprint'] = (
        df['center'].astype(str) + '_' +
        df['age'].astype(str) + '_' +
        df['gender']
    )
    weak_groups = df['weak_fingerprint'].nunique()
    full_groups = df['patient_fingerprint'].nunique()
    
    collision_prevention = full_groups - weak_groups
    if collision_prevention > 0:
        print(f"     Center+age+gender only: {weak_groups} groups")
        print(f"     Full fingerprint (with anatomy): {full_groups} groups")
        print(f"     ✅ Anatomy prevents {collision_prevention} potential collisions ({collision_prevention/weak_groups*100:.1f}%)")
    else:
        print(f"     ℹ️  Anatomy adds no separation (all patients unique by demographics)")
    
    # Explicitly state limitation
    print(f"\n   ⚠️  LIMITATION (report in paper):")
    print(f"      Patient IDs are APPROXIMATED (no explicit identifiers available)")
    print(f"      Rare edge cases may have patient ambiguity")
    print(f"      This is acceptable if stated in Methods section")
    
    # Verify no label leakage
    print(f"\n   ✅ Patient fingerprint is LABEL-FREE:")
    print(f"      Includes: center, age, gender, anatomy, joints")
    print(f"      Excludes: tumor, benign, malignant (no label leakage)")
    
    if multi_image_patients > 0:
        print(f"\n   ✅ Multi-view patients detected (will prevent leakage)")
        example_patient = patient_counts[patient_counts > 1].index[0]
        example_images = df[df['patient_id'] == example_patient][
            ['image_id', 'frontal', 'lateral', 'oblique', 'age', 'gender', 'center']
        ].head(5)
        print(f"\n   Example patient {example_patient} (multi-view):")
        print(example_images.to_string(index=False))
    
    # Clean up temporary columns
    df = df.drop(['anatomy_fingerprint', 'joint_fingerprint', 'patient_fingerprint', 'weak_fingerprint'], axis=1)
    
    return df


# ============================================================================
# STRATIFICATION HELPER
# ============================================================================

def stratify_by_class_and_center(patient_groups):
    """
    Stratified splitting by BOTH class AND center
    
    📌 FOR PUBLICATION:
    Add to Methods: "Splits were stratified by both diagnostic class and
    acquisition center to control for center bias."
    
    Args:
        patient_groups: DataFrame with [patient_id, class_label, center, image_id]
    
    Returns:
        train_patients, val_patients, test_patients
    """
    
    # Create composite stratification key
    patient_groups['strat_key'] = (
        patient_groups['class_label'].astype(str) + '_' + 
        patient_groups['center'].astype(str)
    )
    
    # Check stratification groups
    strat_counts = patient_groups['strat_key'].value_counts()
    print(f"\n   Stratification groups (class_center):")
    for key, count in strat_counts.items():
        cls, center = key.split('_')
        cls_name = Config.CLASS_NAMES[int(cls)]
        print(f"     {cls_name}, Center {center}: {count} patients")
    
    # Warn about small strata
    min_samples_needed = 3
    small_strata = strat_counts[strat_counts < min_samples_needed]
    if len(small_strata) > 0:
        print(f"\n   ⚠️  Warning: {len(small_strata)} strata have <{min_samples_needed} patients")
        print(f"      Will use relaxed stratification for these")
    
    # Try full stratification
    try:
        X = patient_groups['patient_id'].values
        y_strat = patient_groups['strat_key'].values
        
        # Split: (Train+Val) / Test
        X_temp, X_test, _, _ = train_test_split(
            X, X,
            test_size=Config.TEST_RATIO,
            stratify=y_strat,
            random_state=Config.RANDOM_SEED
        )
        
        # Get stratification keys for temp set
        y_strat_temp = patient_groups[patient_groups['patient_id'].isin(X_temp)]['strat_key'].values
        
        # Split: Train / Val
        val_ratio_adjusted = Config.VAL_RATIO / (Config.TRAIN_RATIO + Config.VAL_RATIO)
        X_train, X_val, _, _ = train_test_split(
            X_temp, X_temp,
            test_size=val_ratio_adjusted,
            stratify=y_strat_temp,
            random_state=Config.RANDOM_SEED
        )
        
        print(f"   ✅ Full center+class stratification successful")
        
    except ValueError as e:
        print(f"   ⚠️  Full stratification failed: {e}")
        print(f"   Falling back to class-only stratification")
        
        X = patient_groups['patient_id'].values
        y_class = patient_groups['class_label'].values
        
        X_temp, X_test, _, _ = train_test_split(
            X, X,
            test_size=Config.TEST_RATIO,
            stratify=y_class,
            random_state=Config.RANDOM_SEED
        )
        
        y_class_temp = patient_groups[patient_groups['patient_id'].isin(X_temp)]['class_label'].values
        val_ratio_adjusted = Config.VAL_RATIO / (Config.TRAIN_RATIO + Config.VAL_RATIO)
        X_train, X_val, _, _ = train_test_split(
            X_temp, X_temp,
            test_size=val_ratio_adjusted,
            stratify=y_class_temp,
            random_state=Config.RANDOM_SEED
        )
    
    return X_train, X_val, X_test


# ============================================================================
# DATASET SPLITTING (PATIENT-LEVEL, CENTER-AWARE)
# ============================================================================

def derive_class_label(row):
    """
    Derive class label from tumor/benign/malignant columns
    
    📌 FOR PUBLICATION:
    This implements PATIENT-LEVEL classification (not lesion-level).
    Each patient is assigned ONE dominant diagnosis.
    """
    if row['tumor'] == 0:
        return 0  # Normal
    elif row['tumor'] == 1 and row['benign'] == 1:
        return 1  # Benign
    elif row['tumor'] == 1 and row['malignant'] == 1:
        return 2  # Malignant
    else:
        return -1  # Invalid


def create_stratified_splits(metadata_df):
    """
    Create patient-level, center-aware stratified splits
    
    ✅ Patient-level (not image-level) - prevents data leakage
    ✅ Center-aware stratification - controls center bias
    ✅ Label-free patient fingerprint - no label information used for grouping
    ✅ Includes ALL images (even without annotation JSONs)
    
    📌 FOR PUBLICATION - Add these to Methods:
    1. "Splits were performed at the patient level (not image level) to prevent
       data leakage from multiple views of the same patient."
    2. "Center information was used only for stratified splitting and excluded
       from model inputs to avoid center-specific overfitting."
    3. "For patients with multiple images, patient-level labels were assigned
       via majority voting."
    
    Returns:
        dict: {'train': [image_ids], 'val': [image_ids], 'test': [image_ids]}
    """
    
    # Verify center column
    if 'center' not in metadata_df.columns:
        print("❌ ERROR: 'center' column not found in metadata!")
        print("   Available columns:", metadata_df.columns.tolist())
        return None
    
    # Filter by image file existence (NOT JSON existence!)
    valid_image_ids = []
    for img_id in metadata_df['image_id']:
        img_path = Path(Config.RAW_IMAGES_DIR) / img_id
        if img_path.exists():  # Only check image exists, not JSON
            valid_image_ids.append(img_id)
    
    df_valid = metadata_df[metadata_df['image_id'].isin(valid_image_ids)].copy()
    
    # Derive class labels
    df_valid['class_label'] = df_valid.apply(derive_class_label, axis=1)
    df_valid = df_valid[df_valid['class_label'] != -1]
    
    # Group by patient (label-free)
    df_valid = create_patient_groups(df_valid)
    
    if df_valid is None:
        return None
    
    # Dataset statistics
    print(f"\n📊 Dataset Statistics (BEFORE splitting):")
    print(f"   Total images: {len(df_valid)}")
    print(f"   Unique patients: {df_valid['patient_id'].nunique()}")
    print(f"   Centers: {sorted(df_valid['center'].unique())}")
    
    print(f"\n   Class distribution (by image):")
    class_dist = df_valid['class_label'].value_counts().sort_index()
    for cls_id, count in class_dist.items():
        cls_name = Config.CLASS_NAMES[cls_id]
        pct = count / len(df_valid) * 100
        print(f"     {cls_name}: {count} images ({pct:.1f}%)")
    
    # ✅ Aggregate patient-level info using MAJORITY VOTING
    # 📌 FOR PUBLICATION: "For patients with multiple images, patient-level
    #    labels were assigned via majority voting."
    patient_groups = df_valid.groupby('patient_id').agg({
        'class_label': lambda x: x.mode()[0],  # Majority vote
        'center': lambda x: x.mode()[0],        # Most common center
        'image_id': list
    }).reset_index()
    
    print(f"\n   Class distribution (by patient - majority voting):")
    patient_class_dist = patient_groups['class_label'].value_counts().sort_index()
    for cls_id, count in patient_class_dist.items():
        cls_name = Config.CLASS_NAMES[cls_id]
        pct = count / len(patient_groups) * 100
        print(f"     {cls_name}: {count} patients ({pct:.1f}%)")
    
    print(f"\n   Center distribution (by patient):")
    patient_center_dist = patient_groups['center'].value_counts().sort_index()
    for center, count in patient_center_dist.items():
        pct = count / len(patient_groups) * 100
        print(f"     Center {center}: {count} patients ({pct:.1f}%)")
    
    # Stratified split by CLASS + CENTER
    print(f"\n🔀 Performing patient-level, center-aware stratified split...")
    patients_train, patients_val, patients_test = stratify_by_class_and_center(patient_groups)
    
    # Map patients → images
    train_images = patient_groups[patient_groups['patient_id'].isin(patients_train)]['image_id'].explode().tolist()
    val_images = patient_groups[patient_groups['patient_id'].isin(patients_val)]['image_id'].explode().tolist()
    test_images = patient_groups[patient_groups['patient_id'].isin(patients_test)]['image_id'].explode().tolist()
    
    splits = {
        'train': train_images,
        'val': val_images,
        'test': test_images
    }
    
    # Save splits
    print(f"\n💾 Saving splits:")
    for split_name, image_ids in splits.items():
        num_patients = len(set(df_valid[df_valid['image_id'].isin(image_ids)]['patient_id']))
        output_path = Path(Config.OUTPUT_DIR) / "splits" / f"{split_name}.txt"
        with open(output_path, 'w') as f:
            f.write('\n'.join(image_ids))
        print(f"   {split_name}.txt: {num_patients} patients, {len(image_ids)} images")
    
    # Validate split integrity
    print(f"\n🔍 Validating split integrity...")
    
    # Check 1: No patient overlap
    train_patients_set = set(df_valid[df_valid['image_id'].isin(train_images)]['patient_id'])
    val_patients_set = set(df_valid[df_valid['image_id'].isin(val_images)]['patient_id'])
    test_patients_set = set(df_valid[df_valid['image_id'].isin(test_images)]['patient_id'])
    
    overlap_train_val = train_patients_set & val_patients_set
    overlap_train_test = train_patients_set & test_patients_set
    overlap_val_test = val_patients_set & test_patients_set
    
    if len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0:
        print(f"   ✅ PASS: No patient appears in multiple splits")
    else:
        print(f"   ❌ FAIL: Patient overlap detected!")
        print(f"      Train-Val: {len(overlap_train_val)}, Train-Test: {len(overlap_train_test)}, Val-Test: {len(overlap_val_test)}")
    
    # Check 2: Class distribution by split
    print(f"\n   Class distribution by split:")
    for split_name, image_ids in splits.items():
        split_df = df_valid[df_valid['image_id'].isin(image_ids)]
        print(f"\n   {split_name.upper()}:")
        for cls_id in range(len(Config.CLASS_NAMES)):
            cls_count = (split_df['class_label'] == cls_id).sum()
            patient_count = split_df[split_df['class_label'] == cls_id]['patient_id'].nunique()
            pct = cls_count / len(split_df) * 100
            print(f"     {Config.CLASS_NAMES[cls_id]}: {patient_count} patients, {cls_count} images ({pct:.1f}%)")
    
    # Check 3: Center distribution by split
    print(f"\n   Center distribution by split:")
    for split_name, image_ids in splits.items():
        split_df = df_valid[df_valid['image_id'].isin(image_ids)]
        print(f"\n   {split_name.upper()}:")
        for center in sorted(df_valid['center'].unique()):
            center_count = (split_df['center'] == center).sum()
            patient_count = split_df[split_df['center'] == center]['patient_id'].nunique()
            pct = center_count / len(split_df) * 100
            print(f"     Center {center}: {patient_count} patients, {center_count} images ({pct:.1f}%)")
    
    # Save patient mapping
    patient_map_path = Path(Config.OUTPUT_DIR) / "splits" / "patient_mapping.csv"
    df_valid[['image_id', 'patient_id', 'class_label', 'center', 'age', 'gender', 'frontal', 'lateral', 'oblique']].to_csv(
        patient_map_path, index=False
    )
    print(f"\n   Saved patient_mapping.csv for reference")
    
    print(f"\n✅ Patient-level, center-aware splitting complete!")
    print(f"   ✅ Data leakage prevented!")
    print(f"   ✅ Center bias controlled!")
    print(f"   ✅ Normal images (tumor=0) included!")
    
    return splits


# ============================================================================
# METADATA PREPROCESSING
# ============================================================================

def preprocess_metadata(metadata_df, image_ids, split_name, scaler=None):
    """
    Preprocess metadata for given image_ids
    23 features (NO center - used for splitting only)
    
    📌 FOR PUBLICATION:
    "Center information was used only for stratified splitting and excluded
    from model inputs to avoid center-specific overfitting."
    """
    
    df_split = metadata_df[metadata_df['image_id'].isin(image_ids)].copy()
    
    print(f"\n🧹 Preprocessing {split_name} metadata: {len(df_split)} samples")
    
    # Derive class labels
    df_split['class_label'] = df_split.apply(derive_class_label, axis=1)
    
    invalid_count = (df_split['class_label'] == -1).sum()
    if invalid_count > 0:
        print(f"⚠️  Removing {invalid_count} samples with invalid labels")
        df_split = df_split[df_split['class_label'] != -1]
    
    # Select features (23 features, NO center)
    features = Config.METADATA_FEATURES.copy()
    X = df_split[features].copy()
    
    # Encode gender
    X['gender'] = X['gender'].map({'M': 1, 'F': 0})
    if X['gender'].isna().any():
        print(f"⚠️  Warning: {X['gender'].isna().sum()} samples with invalid gender")
        X['gender'].fillna(0, inplace=True)
    
    # Normalize age
    if scaler is None:
        scaler = StandardScaler()
        X['age'] = scaler.fit_transform(X[['age']])
        print(f"   Age normalization: mean={scaler.mean_[0]:.2f}, std={scaler.scale_[0]:.2f}")
    else:
        X['age'] = scaler.transform(X[['age']])
    
    # Combine
    df_output = pd.concat([
        df_split[['image_id']].reset_index(drop=True),
        X.reset_index(drop=True),
        df_split[['class_label']].reset_index(drop=True)
    ], axis=1)
    
    # Class distribution
    class_dist = df_output['class_label'].value_counts().sort_index()
    print(f"   Class distribution:")
    for cls_id, count in class_dist.items():
        cls_name = Config.CLASS_NAMES[cls_id]
        pct = count / len(df_output) * 100
        print(f"     {cls_name}: {count} ({pct:.1f}%)")
    
    # Save
    output_path = Path(Config.OUTPUT_DIR) / "metadata_processed" / f"metadata_{split_name}.csv"
    df_output.to_csv(output_path, index=False)
    print(f"✅ metadata_{split_name}.csv saved: {len(df_output)} samples")
    
    return df_output, scaler


# ============================================================================
# STATISTICS
# ============================================================================

def generate_statistics(metadata_df, splits, coco_data):
    """
    Generate and save dataset statistics
    
    ✅ FIXED: Now filters metadata to only include images from splits
           before generating patient statistics (prevents phantom patients)
    """
    
    # ✅ FIX: Filter metadata to only include images from splits
    all_split_images = []
    for image_ids in splits.values():
        all_split_images.extend(image_ids)
    
    metadata_filtered = metadata_df[metadata_df['image_id'].isin(all_split_images)].copy()
    
    # ✅ FIX: Derive class labels on filtered data
    metadata_filtered['class_label'] = metadata_filtered.apply(derive_class_label, axis=1)
    metadata_filtered = metadata_filtered[metadata_filtered['class_label'] != -1]
    
    # Count patients per split
    df_with_patients = create_patient_groups(metadata_filtered)
    
    if df_with_patients is None:
        print("⚠️  Cannot generate statistics without patient grouping")
        return {}
    
    def count_patients(image_ids):
        return df_with_patients[df_with_patients['image_id'].isin(image_ids)]['patient_id'].nunique()
    
    def count_centers(image_ids):
        return df_with_patients[df_with_patients['image_id'].isin(image_ids)]['center'].nunique()
    
    stats = {
        "dataset_info": {
            "name": "BTXRD Bone Tumor Dataset",
            "date_processed": "2026-01-19",
            "total_samples_metadata": len(metadata_df),
            "valid_samples": sum(len(ids) for ids in splits.values()),
            "splitting_strategy": "patient-level with center-aware stratification",
            "multi_center": True,
            "num_centers": int(df_with_patients['center'].nunique()),
            "includes_normal_images": True
        },
        "splits": {
            "train_images": len(splits['train']),
            "train_patients": count_patients(splits['train']),
            "train_centers": count_centers(splits['train']),
            "val_images": len(splits['val']),
            "val_patients": count_patients(splits['val']),
            "val_centers": count_centers(splits['val']),
            "test_images": len(splits['test']),
            "test_patients": count_patients(splits['test']),
            "test_centers": count_centers(splits['test']),
            "train_ratio": Config.TRAIN_RATIO,
            "val_ratio": Config.VAL_RATIO,
            "test_ratio": Config.TEST_RATIO
        },
        "annotations": {
            "train_images": len(coco_data['train']['images']),
            "train_annotations": len(coco_data['train']['annotations']),
            "val_images": len(coco_data['val']['images']),
            "val_annotations": len(coco_data['val']['annotations']),
            "test_images": len(coco_data['test']['images']),
            "test_annotations": len(coco_data['test']['annotations'])
        },
        "metadata": {
            "num_features": len(Config.METADATA_FEATURES),
            "features": Config.METADATA_FEATURES,
            "excluded_feature": "center (used for stratification only)"
        },
        "classes": {
            "names": Config.CLASS_NAMES,
            "mapping": {"Normal": 0, "Benign": 1, "Malignant": 2}
        },
        "coco_format": {
            "category_id": 1,
            "note": "COCO standard requires category_id starting at 1. Detectron2 internally remaps to 0-indexed."
        },
        "data_leakage_prevention": {
            "splitting_level": "patient (not image or lesion)",
            "stratification": "class + center",
            "patient_fingerprint": "label-free (center, age, gender, anatomy, joints)",
            "patient_label_assignment": "majority voting for multi-image patients",
            "description": "All views of same patient kept in same split. Center distribution preserved. Normal images (tumor=0) included.",
            "validation": "No patient appears in multiple splits",
            "limitation": "Patient IDs approximated from demographics (no explicit identifiers available)"
        }
    }
    
    output_path = Path(Config.OUTPUT_DIR) / "statistics.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
    
    print(f"✅ statistics.json saved")
    
    return stats


# ============================================================================
# VALIDATION
# ============================================================================

def validate_alignment(splits):
    """Validate sample-level alignment"""
    print(f"\n🔍 Validating file alignment...")
    
    all_image_ids = []
    for split_name, image_ids in splits.items():
        all_image_ids.extend(image_ids)
    
    missing_files = {"images": 0, "masks": 0, "jsons": 0}
    
    for img_id in tqdm(all_image_ids, desc="Validating"):
        img_path = Path(Config.RAW_IMAGES_DIR) / img_id
        mask_path = Path(Config.RAW_MASKS_DIR) / f"{Path(img_id).stem}_mask.png"
        json_path = Path(Config.RAW_ANNOTATIONS_DIR) / f"{Path(img_id).stem}.json"
        
        if not img_path.exists():
            missing_files["images"] += 1
        if not mask_path.exists():
            missing_files["masks"] += 1
        if not json_path.exists():
            missing_files["jsons"] += 1
    
    print(f"✅ Alignment validation complete:")
    print(f"   Missing images: {missing_files['images']}")
    print(f"   Missing masks: {missing_files['masks']}")
    print(f"   Missing JSONs: {missing_files['jsons']} (expected for normal images)")


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Execute complete Stage 1 preprocessing pipeline"""
    
    print("=" * 80)
    print("STAGE 1: DATA PREPROCESSING & COCO CONVERSION (PRODUCTION-READY)")
    print("=" * 80)
    print("✅ category_id: 1 (COCO standard) - Detectron2 remaps internally")
    print("✅ Patient-level splitting (prevents data leakage)")
    print("✅ Center-aware stratification (controls center bias)")
    print("✅ Label-free patient fingerprint (no tumor/benign/malignant)")
    print("✅ Includes normal images (tumor=0) with zero annotations")
    print("✅ Explicit documentation of methodological choices")
    print("=" * 80)
    print("\n📌 FOR PUBLICATION - ADD THESE TO METHODS:")
    print("1. Patient IDs approximated (no explicit identifiers)")
    print("2. Majority voting for multi-image patients")
    print("3. Center excluded from model inputs")
    print("4. Normal images included for class imbalance")
    print("=" * 80)
    
    # Step 1: Create directories
    print("\n[1/6] Creating directory structure...")
    create_directory_structure()
    
    # Step 2: Load metadata
    print("\n[2/6] Loading metadata...")
    metadata_df = pd.read_excel(Config.METADATA_FILE)
    print(f"✅ Loaded {len(metadata_df)} samples from metadata")
    
    # Verify columns
    required_cols = ['image_id', 'tumor', 'benign', 'malignant', 'center'] + Config.METADATA_FEATURES
    missing_cols = set(required_cols) - set(metadata_df.columns)
    if missing_cols:
        print(f"❌ ERROR: Missing columns: {missing_cols}")
        return
    
    # Step 3: Create patient-level, center-aware splits
    print("\n[3/6] Creating patient-level, center-aware stratified splits...")
    splits = create_stratified_splits(metadata_df)
    
    if splits is None:
        print("❌ ERROR: Splitting failed!")
        return
    
    # Step 4: Convert to COCO format
    print("\n[4/6] Converting annotations to COCO format...")
    coco_data = {}
    for split_name, image_ids in splits.items():
        coco_data[split_name] = convert_labelme_to_coco(image_ids, split_name)
    
    # Step 5: Preprocess metadata
    print("\n[5/6] Preprocessing metadata (23 features, NO center)...")
    scaler = None
    for split_name in ['train', 'val', 'test']:
        image_ids = splits[split_name]
        _, scaler = preprocess_metadata(metadata_df, image_ids, split_name, scaler)
    
    # Step 6: Validate and generate statistics
    print("\n[6/6] Validation and statistics...")
    validate_alignment(splits)
    stats = generate_statistics(metadata_df, splits, coco_data)
    
    # Final summary
    print("\n" + "=" * 80)
    print("✅ STAGE 1 COMPLETE - PRODUCTION-READY!")
    print("=" * 80)
    print(f"\nOutputs saved to: {Config.OUTPUT_DIR}/")
    print(f"  ├── coco_annotations/")
    print(f"  │   ├── train.json ({stats['annotations']['train_images']} images, {stats['annotations']['train_annotations']} annotations)")
    print(f"  │   ├── val.json ({stats['annotations']['val_images']} images, {stats['annotations']['val_annotations']} annotations)")
    print(f"  │   └── test.json ({stats['annotations']['test_images']} images, {stats['annotations']['test_annotations']} annotations)")
    print(f"  ├── metadata_processed/")
    print(f"  │   ├── metadata_train.csv ({stats['splits']['train_patients']} patients, {stats['splits']['train_images']} images)")
    print(f"  │   ├── metadata_val.csv ({stats['splits']['val_patients']} patients, {stats['splits']['val_images']} images)")
    print(f"  │   └── metadata_test.csv ({stats['splits']['test_patients']} patients, {stats['splits']['test_images']} images)")
    print(f"  ├── splits/")
    print(f"  │   ├── train.txt, val.txt, test.txt")
    print(f"  │   └── patient_mapping.csv")
    print(f"  └── statistics.json")
    print(f"\n📊 Dataset Summary:")
    print(f"  Total patients: {stats['splits']['train_patients'] + stats['splits']['val_patients'] + stats['splits']['test_patients']}")
    print(f"  Train: {stats['splits']['train_patients']} patients, {stats['splits']['train_images']} images")
    print(f"  Val: {stats['splits']['val_patients']} patients, {stats['splits']['val_images']} images")
    print(f"  Test: {stats['splits']['test_patients']} patients, {stats['splits']['test_images']} images")
    print(f"  Centers: {stats['dataset_info']['num_centers']}")
    print(f"  Metadata features: {stats['metadata']['num_features']}")
    print(f"\n✅ All critical fixes applied:")
    print(f"  ✅ category_id = 1 (COCO standard, Detectron2 compatible)")
    print(f"  ✅ Split-local image IDs (documented)")
    print(f"  ✅ Patient collision risk acknowledged and mitigated")
    print(f"  ✅ Majority voting documented")
    print(f"  ✅ Patient-level splitting (no leakage)")
    print(f"  ✅ Center-aware stratification")
    print(f"  ✅ Normal images included in COCO JSON")
    print(f"\n🎯 Ready for Stage 2: Mask R-CNN Training")
    print(f"🔥 Q1/Q2 Publication-Ready Preprocessing Pipeline")
    print("=" * 80)


if __name__ == "__main__":
    main()


STAGE 1: DATA PREPROCESSING & COCO CONVERSION (PRODUCTION-READY)
✅ category_id: 1 (COCO standard) - Detectron2 remaps internally
✅ Patient-level splitting (prevents data leakage)
✅ Center-aware stratification (controls center bias)
✅ Label-free patient fingerprint (no tumor/benign/malignant)
✅ Includes normal images (tumor=0) with zero annotations
✅ Explicit documentation of methodological choices

📌 FOR PUBLICATION - ADD THESE TO METHODS:
1. Patient IDs approximated (no explicit identifiers)
2. Majority voting for multi-image patients
3. Center excluded from model inputs
4. Normal images included for class imbalance

[1/6] Creating directory structure...
✅ Directory structure created

[2/6] Loading metadata...
✅ Loaded 3746 samples from metadata

[3/6] Creating patient-level, center-aware stratified splits...

📊 Patient grouping statistics:
   Total images: 3746
   Unique patients: 1008
   Avg images per patient: 3.72
   Patients with multiple views: 688

   Center distribution (by pa

Processing train: 100%|██████████| 2602/2602 [00:17<00:00, 144.73it/s]


✅ train.json saved:
   Total images: 2602
   Images with annotations: 1285
   Images without annotations (normal): 1317
   Total annotations: 1617
   ℹ️  Note: Normal images included for realistic class imbalance

🔄 Converting val set: 580 images


Processing val: 100%|██████████| 580/580 [00:04<00:00, 133.36it/s]


✅ val.json saved:
   Total images: 580
   Images with annotations: 296
   Images without annotations (normal): 284
   Total annotations: 364
   ℹ️  Note: Normal images included for realistic class imbalance

🔄 Converting test set: 564 images


Processing test: 100%|██████████| 564/564 [00:04<00:00, 128.03it/s]


✅ test.json saved:
   Total images: 564
   Images with annotations: 286
   Images without annotations (normal): 278
   Total annotations: 337
   ℹ️  Note: Normal images included for realistic class imbalance

[5/6] Preprocessing metadata (23 features, NO center)...

🧹 Preprocessing train metadata: 2602 samples
   Age normalization: mean=34.34, std=20.81
   Class distribution:
     Normal: 1317 (50.6%)
     Benign: 1050 (40.4%)
     Malignant: 235 (9.0%)
✅ metadata_train.csv saved: 2602 samples

🧹 Preprocessing val metadata: 580 samples
   Class distribution:
     Normal: 284 (49.0%)
     Benign: 236 (40.7%)
     Malignant: 60 (10.3%)
✅ metadata_val.csv saved: 580 samples

🧹 Preprocessing test metadata: 564 samples
   Class distribution:
     Normal: 278 (49.3%)
     Benign: 239 (42.4%)
     Malignant: 47 (8.3%)
✅ metadata_test.csv saved: 564 samples

[6/6] Validation and statistics...

🔍 Validating file alignment...


Validating: 100%|██████████| 3746/3746 [00:02<00:00, 1280.93it/s]


✅ Alignment validation complete:
   Missing images: 0
   Missing masks: 3746
   Missing JSONs: 1879 (expected for normal images)

📊 Patient grouping statistics:
   Total images: 3746
   Unique patients: 1008
   Avg images per patient: 3.72
   Patients with multiple views: 688

   Center distribution (by patient):
     Center 1: 698 patients, 2938 images (69.2%)
     Center 2: 173 patients, 549 images (17.2%)
     Center 3: 137 patients, 259 images (13.6%)

   Patient identity collision analysis:
     Center+age+gender only: 295 groups
     Full fingerprint (with anatomy): 1008 groups
     ✅ Anatomy prevents 713 potential collisions (241.7%)

   ⚠️  LIMITATION (report in paper):
      Patient IDs are APPROXIMATED (no explicit identifiers available)
      Rare edge cases may have patient ambiguity
      This is acceptable if stated in Methods section

   ✅ Patient fingerprint is LABEL-FREE:
      Includes: center, age, gender, anatomy, joints
      Excludes: tumor, benign, malignant (no 

-----------------------------------------INTERMEDIATE FUSION-----------------------------------

In [2]:
"""
FUSION STEP 1: DATA PAIRING
============================
Pairs ROI crops with their corresponding clinical metadata.
Creates fusion-ready paired CSVs for train / val / test.

What this script does:
  1. Loads roi_metadata.csv (Stage 3B output)
  2. Loads metadata_train/val/test.csv (Stage 1 output, preprocessed)
  3. Joins on source_image == image_id
  4. Filters: tumor-only (class_label != 0 in metadata CSV)
  5. Resolves binary fusion labels: benign=0, malignant=1
  6. Validates patient-level integrity (no leakage)
  7. Reports class imbalance + flags fallback ROIs
  8. Saves fusion_paired_train/val/test.csv

Output columns per row:
  roi_filename, split, roi_class (str), fusion_label (0/1),
  source_image, roi_width, roi_height, match_type, confidence,
  patient_id, [22 clinical features]

Q1 Safety:
  ✅ No preprocessing fitted on val/test (scaler was fitted in Stage 1)
  ✅ Patient leakage check across all three splits
  ✅ Labels derived from roi_metadata only (not from metadata CSV labels)
  ✅ clinical features are FEATURES_SET_A (22 features, excludes 'foot')
"""

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION — adjust paths to match your Kaggle/local setup
# ============================================================================
ROI_METADATA_PATH   = "/kaggle/input/datasets/sadibhasan/roi-dataset/roi_metadata.csv"
METADATA_TRAIN_PATH = "preprocessed/metadata_processed/metadata_train.csv"
METADATA_VAL_PATH   = "preprocessed/metadata_processed/metadata_val.csv"
METADATA_TEST_PATH  = "preprocessed/metadata_processed/metadata_test.csv"
PATIENT_MAP_PATH    = "preprocessed/splits/patient_mapping.csv"   # from Stage 1
OUTPUT_DIR          = "fusion_outputs/step1_paired"

# These are the EXACT 22 features used by your clinical model (FEATURES_SET_A)
# 'foot' is in Stage 1 metadata (23 features) but was NOT in the clinical model
CLINICAL_FEATURES = [
    "age", "gender",                                    # demographics (2)
    "hand", "ulna", "radius", "humerus",               # upper limb bones (4)
    "femur", "tibia", "fibula", "hip bone",            # lower limb bones (4)
    "ankle-joint", "knee-joint", "hip-joint",          # lower joints (3)
    "wrist-joint", "elbow-joint", "shoulder-joint",    # upper joints (3)
    "upper limb", "lower limb", "pelvis",              # body regions (3)
    "frontal", "lateral", "oblique", "foot"                     # X-ray views (3)
]
# Total: 22 features ✓

# ROI label convention from Stage 3A: 1=benign, 2=malignant
# Fusion binary label: 0=benign, 1=malignant
ROI_LABEL_TO_BINARY = {1: 0, 2: 1}
ROI_CLASS_TO_BINARY = {"benign": 0, "malignant": 1}


# ============================================================================
# HELPERS
# ============================================================================
def normalize_img_key(img_id: str) -> str:
    """Strip path, keep filename only — OS-safe."""
    return Path(str(img_id)).name


def print_section(title: str):
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")


# ============================================================================
# STEP 1A: Load all source files
# ============================================================================
print_section("FUSION STEP 1: DATA PAIRING")

print("\n[1/7] Loading source files...")
roi_meta = pd.read_csv(ROI_METADATA_PATH)
meta_train = pd.read_csv(METADATA_TRAIN_PATH)
meta_val   = pd.read_csv(METADATA_VAL_PATH)
meta_test  = pd.read_csv(METADATA_TEST_PATH)

print(f"  roi_metadata     : {len(roi_meta):>5} rows | "
      f"splits: {dict(roi_meta['split'].value_counts())}")
print(f"  metadata_train   : {len(meta_train):>5} rows")
print(f"  metadata_val     : {len(meta_val):>5} rows")
print(f"  metadata_test    : {len(meta_test):>5} rows")

# Normalize image_id keys for safe joining
roi_meta['_join_key'] = roi_meta['source_image'].apply(normalize_img_key)
for df in [meta_train, meta_val, meta_test]:
    df['_join_key'] = df['image_id'].apply(normalize_img_key)

# Combine metadata splits for a single lookup table
meta_all = pd.concat([meta_train, meta_val, meta_test], ignore_index=True)
print(f"\n  Combined metadata: {len(meta_all):>5} rows")

# ============================================================================
# STEP 1B: Validate clinical feature availability
# ============================================================================
print("\n[2/7] Validating clinical features...")
missing_feats = [f for f in CLINICAL_FEATURES if f not in meta_all.columns]
if missing_feats:
    raise ValueError(f"❌ Missing clinical features in metadata CSV: {missing_feats}")
print(f"  ✅ All 22 clinical features present in metadata CSVs")

# Check 'foot' is excluded (it's in Stage 1 but not in clinical model)
if 'foot' in meta_all.columns:
    print(f"  ℹ️  'foot' column exists in metadata CSV but is intentionally "
          f"excluded (not in clinical model's FEATURES_SET_A)")

# ============================================================================
# STEP 1C: Filter metadata to tumor-only (class_label != 0)
# ============================================================================
print("\n[3/7] Filtering metadata to tumor-only rows...")
# Stage 1 class_label: 0=Normal, 1=Benign, 2=Malignant
# We want only Benign (1) and Malignant (2)
meta_tumor = meta_all[meta_all['class_label'] != 0].copy()
print(f"  Before filter: {len(meta_all)} rows")
print(f"  After filter : {len(meta_tumor)} rows "
      f"(removed {len(meta_all) - len(meta_tumor)} Normal rows)")

meta_label_dist = meta_tumor['class_label'].value_counts().sort_index()
print(f"  class_label distribution in metadata: "
      f"1=Benign: {meta_label_dist.get(1, 0)}, "
      f"2=Malignant: {meta_label_dist.get(2, 0)}")

# ============================================================================
# STEP 1D: Load patient mapping for patient-level validation
# ============================================================================
print("\n[4/7] Loading patient mapping...")
try:
    patient_map = pd.read_csv(PATIENT_MAP_PATH)
    patient_map['_join_key'] = patient_map['image_id'].apply(normalize_img_key)
    img_to_patient = dict(zip(patient_map['_join_key'],
                              patient_map['patient_id']))
    print(f"  Loaded {len(patient_map)} patient mappings")
    print(f"  Unique patients: {patient_map['patient_id'].nunique()}")
except FileNotFoundError:
    print(f"  ⚠️  Patient mapping not found at {PATIENT_MAP_PATH}")
    print(f"       Patient-level leakage check will be skipped.")
    img_to_patient = {}

# ============================================================================
# STEP 1E: Join ROI metadata with clinical metadata
# ============================================================================
print("\n[5/7] Joining ROI metadata with clinical metadata...")

# Build the clinical lookup: _join_key → clinical features + patient_id
clinical_cols = CLINICAL_FEATURES + ['_join_key']
if 'patient_id' in meta_tumor.columns:
    clinical_cols = ['patient_id'] + clinical_cols

clinical_lookup = meta_tumor[clinical_cols].drop_duplicates(
    subset=['_join_key']
).copy()

print(f"  Unique tumor images in metadata: {len(clinical_lookup)}")
print(f"  Unique ROI source images        : "
      f"{roi_meta['_join_key'].nunique()}")

# Join
merged = roi_meta.merge(clinical_lookup, on='_join_key', how='inner')
unmatched = len(roi_meta) - len(merged)
print(f"\n  ROIs before join : {len(roi_meta)}")
print(f"  ROIs after join  : {len(merged)}")
if unmatched > 0:
    print(f"  ⚠️  {unmatched} ROIs had no match in metadata "
          f"(likely Normal images that were filtered out)")

    # Show which source images are unmatched
    matched_keys = set(merged['_join_key'])
    unmatched_rows = roi_meta[~roi_meta['_join_key'].isin(matched_keys)]
    print(f"  Unmatched source images (sample):")
    print(f"    {unmatched_rows['source_image'].head(5).tolist()}")

# ============================================================================
# STEP 1F: Create binary fusion label + add patient_id
# ============================================================================
print("\n[6/7] Creating fusion labels and adding patient IDs...")

# Binary label from roi_metadata class column (ground truth from Stage 3A)
# Using roi_metadata labels (not metadata CSV labels) — they are consistent
merged['fusion_label'] = merged['class'].map(ROI_CLASS_TO_BINARY)
if merged['fusion_label'].isna().any():
    n_bad = merged['fusion_label'].isna().sum()
    raise ValueError(
        f"❌ {n_bad} rows have unmappable class strings: "
        f"{merged.loc[merged['fusion_label'].isna(), 'class'].unique()}"
    )
merged['fusion_label'] = merged['fusion_label'].astype(int)

# Add patient_id from mapping if not already present
if 'patient_id' not in merged.columns or merged['patient_id'].isna().all():
    merged['patient_id'] = merged['_join_key'].map(img_to_patient)
    print(f"  patient_id added from patient_mapping.csv")
else:
    print(f"  patient_id already present from metadata CSV join")

# Drop internal join key
merged = merged.drop(columns=['_join_key'])

# Verify label consistency with class_label column from ROI metadata
# ROI class_label: 1=benign→fusion 0, 2=malignant→fusion 1
merged['_expected_label'] = merged['class_label'].map(ROI_LABEL_TO_BINARY)
mismatch = (merged['fusion_label'] != merged['_expected_label']).sum()
if mismatch > 0:
    raise ValueError(
        f"❌ Label mismatch: {mismatch} rows have inconsistent "
        f"class string vs class_label. Check roi_metadata.csv."
    )
merged = merged.drop(columns=['_expected_label'])
print(f"  ✅ fusion_label consistent with class_label (0=benign, 1=malignant)")

# ============================================================================
# STEP 1G: Split back into train / val / test
# ============================================================================
print("\n[7/7] Splitting, validating, and saving...")

split_dfs = {}
for split_name in ['train', 'val', 'test']:
    split_df = merged[merged['split'] == split_name].copy().reset_index(drop=True)
    split_dfs[split_name] = split_df

# Patient-level leakage check
if img_to_patient:
    train_pts = set(split_dfs['train']['patient_id'].dropna())
    val_pts   = set(split_dfs['val']['patient_id'].dropna())
    test_pts  = set(split_dfs['test']['patient_id'].dropna())

    ov_tv = train_pts & val_pts
    ov_tt = train_pts & test_pts
    ov_vt = val_pts   & test_pts

    print(f"\n  Patient overlap check:")
    print(f"    Train patients: {len(train_pts)}")
    print(f"    Val patients  : {len(val_pts)}")
    print(f"    Test patients : {len(test_pts)}")

    if ov_tv or ov_tt or ov_vt:
        raise RuntimeError(
            f"❌ LEAKAGE DETECTED: Train∩Val={len(ov_tv)}, "
            f"Train∩Test={len(ov_tt)}, Val∩Test={len(ov_vt)}"
        )
    print(f"    ✅ No patient overlap across splits")

# Save
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
output_cols = (
    ['roi_filename', 'split', 'class', 'fusion_label', 'class_label',
     'source_image', 'patient_id', 'match_type', 'confidence',
     'roi_width', 'roi_height', 'subtype']
    + CLINICAL_FEATURES
)
# Only keep columns that exist
output_cols = [c for c in output_cols if c in merged.columns]

print(f"\n  Split statistics:")
print(f"  {'Split':<8} {'Total ROIs':>10} {'Benign':>8} "
      f"{'Malignant':>10} {'Patients':>9} {'Fallback ROIs':>14}")
print(f"  {'-'*60}")

for split_name, df in split_dfs.items():
    n_benign    = (df['fusion_label'] == 0).sum()
    n_malignant = (df['fusion_label'] == 1).sum()
    n_patients  = df['patient_id'].nunique() if 'patient_id' in df else 'N/A'
    n_fallback  = (df['match_type'] == 'fallback').sum() \
                  if 'match_type' in df.columns else 0
    ratio       = n_benign / max(n_malignant, 1)

    print(f"  {split_name:<8} {len(df):>10} {n_benign:>8} "
          f"{n_malignant:>10} {n_patients:>9} {n_fallback:>14}")
    print(f"  {'':8} Imbalance ratio: {ratio:.2f}:1 "
          f"(Benign:Malignant)")

    save_path = Path(OUTPUT_DIR) / f"fusion_paired_{split_name}.csv"
    df[output_cols].to_csv(save_path, index=False)
    print(f"  {'':8} ✅ Saved: {save_path}")

# ============================================================================
# SUMMARY
# ============================================================================
print_section("STEP 1 COMPLETE — SUMMARY")

total_rois = sum(len(df) for df in split_dfs.values())
total_benign    = sum((df['fusion_label'] == 0).sum()
                      for df in split_dfs.values())
total_malignant = sum((df['fusion_label'] == 1).sum()
                      for df in split_dfs.values())

print(f"""
  Total paired ROIs   : {total_rois}
  Total benign ROIs   : {total_benign}
  Total malignant ROIs: {total_malignant}
  Overall imbalance   : {total_benign / max(total_malignant, 1):.2f}:1

  Output files:
    {OUTPUT_DIR}/fusion_paired_train.csv
    {OUTPUT_DIR}/fusion_paired_val.csv
    {OUTPUT_DIR}/fusion_paired_test.csv

  Columns in output:
    ROI info  : roi_filename, split, class, fusion_label, class_label,
                source_image, patient_id, match_type, confidence,
                roi_width, roi_height, subtype
    Clinical  : {CLINICAL_FEATURES}
    (22 features, same as FEATURES_SET_A in your clinical model)

  ⚠️  IMPORTANT NOTES FOR STEP 2:
    - ROI crops are at:
        stage3_roi_dataset/{{split}}/{{class}}/{{roi_filename}}
    - fusion_label: 0=benign, 1=malignant
    - 'fallback' ROIs (low-confidence detection) are flagged in match_type
      Consider running sensitivity analysis with/without them
    - Age is already StandardScaler-normalized from Stage 1
      (same format the clinical model was trained on)

  ✅ Ready for Step 2: convnext feature extraction
""")



  FUSION STEP 1: DATA PAIRING

[1/7] Loading source files...
  roi_metadata     :  2098 rows | splits: {'train': np.int64(1520), 'val': np.int64(295), 'test': np.int64(283)}
  metadata_train   :  2602 rows
  metadata_val     :   580 rows
  metadata_test    :   564 rows

  Combined metadata:  3746 rows

[2/7] Validating clinical features...
  ✅ All 22 clinical features present in metadata CSVs
  ℹ️  'foot' column exists in metadata CSV but is intentionally excluded (not in clinical model's FEATURES_SET_A)

[3/7] Filtering metadata to tumor-only rows...
  Before filter: 3746 rows
  After filter : 1867 rows (removed 1879 Normal rows)
  class_label distribution in metadata: 1=Benign: 1525, 2=Malignant: 342

[4/7] Loading patient mapping...
  Loaded 3746 patient mappings
  Unique patients: 1008

[5/7] Joining ROI metadata with clinical metadata...
  Unique tumor images in metadata: 1867
  Unique ROI source images        : 1692

  ROIs before join : 2098
  ROIs after join  : 2098

[6/7] Cre

In [3]:
"""
FUSION STEP 2: CONVNEXT-TINY-SE FEATURE EXTRACTION
====================================================
Replaces ResNet18SE Step 2 entirely.

Changes from ResNet18SE version:
  ✅ Model: TumorClassifierConvNeXtTinySE (768-dim, not 512)
  ✅ Two checkpoint conditions: with_aug / no_aug
  ✅ Feature tap: after pool(se(features(x))) → 768-dim vector
  ✅ Output dirs: step2_features/with_aug/ and step2_features/no_aug/
  ✅ Inference always uses val_transform (no augmentation regardless of condition)
  ✅ Near-zero check updated for 768-dim

Run this cell TWICE — once with AUG_CONDITION = 'with_aug',
once with AUG_CONDITION = 'no_aug'. Or leave it as-is and it
loops both conditions automatically.

Output per condition per split:
  step2_features/{condition}/image_features_train.npy  shape: (N_train, 768)
  step2_features/{condition}/image_features_val.npy    shape: (N_val,   768)
  step2_features/{condition}/image_features_test.npy   shape: (N_test,  768)
  step2_features/{condition}/image_features_train_meta.csv
  step2_features/{condition}/image_features_val_meta.csv
  step2_features/{condition}/image_features_test_meta.csv
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image, UnidentifiedImageError

import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# ── Checkpoint paths — update these to your actual Kaggle input paths ──────
CHECKPOINTS = {
    'with_aug': "/kaggle/input/datasets/sadibhasan/convnext-best-classimodel/final_best_with_aug.pth",
    'no_aug':   "/kaggle/input/datasets/sadibhasan/convnext-best-classimodel/final_best_no_aug.pth",
}

# ROI dataset root (from Stage 3A)
ROI_DATASET_DIR = "/kaggle/input/datasets/sadibhasan/roi-dataset"

# Paired CSVs from Step 1
PAIRED_DIR  = "fusion_outputs/step1_paired"

# Output root — subfolders per condition created automatically
OUTPUT_ROOT = "fusion_outputs/step2_features"

IMAGE_SIZE  = 256   # Must match your ConvNeXt training
BATCH_SIZE  = 64
NUM_WORKERS = 4
RANDOM_SEED = 42

CLASS_TO_LABEL = {'benign': 0, 'malignant': 1}

_BILINEAR = (
    Image.Resampling.BILINEAR
    if hasattr(Image, 'Resampling')
    else Image.BILINEAR
)

# ============================================================================
# MODEL — exact copy from your classification notebook
# ============================================================================

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block — exact copy from classification notebook."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class TumorClassifierConvNeXtTinySE(nn.Module):
    """
    ConvNeXt-Tiny with SE block — exact architecture from classification notebook.
    Loaded in full so state_dict keys match perfectly, then stripped for extraction.
    """
    def __init__(self, n=2, r=16):
        super().__init__()
        bb = torchvision.models.convnext_tiny(weights=None)  # weights loaded from ckpt
        self.features = bb.features
        self.se       = SEBlock(768, r)
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.head     = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(768),
            nn.Dropout(0.6),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, n)
        )

    def forward(self, x):
        x = self.pool(self.se(self.features(x)))   # → (B, 768, 1, 1)
        return self.head(x)


class ConvNeXtTinySEFeatureExtractor(nn.Module):
    """
    Same as TumorClassifierConvNeXtTinySE but forward() returns the
    768-dim vector BEFORE the head (after pool + se + flatten).

    forward() output shape: (batch_size, 768)

    This is the correct intermediate fusion extraction point:
      - AFTER the SE block (task-aware channel recalibration kept)
      - AFTER global average pool (spatial dims collapsed)
      - BEFORE LayerNorm / Dropout / Linear (task-specific head removed)
    """
    def __init__(self, full_model: TumorClassifierConvNeXtTinySE):
        super().__init__()
        self.features = full_model.features
        self.se       = full_model.se
        self.pool     = full_model.pool
        # head intentionally NOT included

    def forward(self, x):
        x = self.pool(self.se(self.features(x)))  # → (B, 768, 1, 1)
        return x.flatten(1)                        # → (B, 768)


# ============================================================================
# IMAGE PREPROCESSING — identical to val_transform from classification notebook
# CRITICAL: Always use val_transform for feature extraction regardless of
#           whether the model was trained with or without augmentation.
#           Training augmentation affects learned weights only, not inference.
# ============================================================================

def resize_with_padding(img: Image.Image,
                        target_size=(256, 256)) -> Image.Image:
    """Aspect-ratio preserving resize with black padding — same as classification."""
    old_w, old_h = img.size
    ratio  = min(target_size[0] / old_w, target_size[1] / old_h)
    new_w  = int(old_w * ratio)
    new_h  = int(old_h * ratio)
    img    = img.resize((new_w, new_h), _BILINEAR)
    canvas = Image.new("RGB", target_size, (0, 0, 0))
    canvas.paste(img, ((target_size[0] - new_w) // 2,
                       (target_size[1] - new_h) // 2))
    return canvas


# No augmentation — val_transform only, always
val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# ============================================================================
# DATASET
# ============================================================================

class ROIFeatureDataset(Dataset):
    """
    Loads ROI crops for feature extraction.
    Returns: (image_tensor, fusion_label, roi_filename)

    Missing/corrupt files return a black image (sentinel prefix added to
    filename). extract_features() detects resulting near-zero vectors via
    L2-norm < 1e-6 and raises RuntimeError with the full list of failures.
    """
    def __init__(self, paired_df: pd.DataFrame, roi_root: str):
        self.df       = paired_df.reset_index(drop=True)
        self.roi_root = roi_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        roi_path = os.path.join(
            self.roi_root,
            row['split'],
            row['class'],
            row['roi_filename']
        )

        load_failed = False
        try:
            img = Image.open(roi_path).convert("RGB")
        except (FileNotFoundError, OSError, UnidentifiedImageError):
            img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
            load_failed = True

        img = resize_with_padding(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = val_transform(img)

        fname = f"__LOAD_FAILED__{row['roi_filename']}" if load_failed else row['roi_filename']
        return img, int(row['fusion_label']), fname


# ============================================================================
# LOAD MODEL
# ============================================================================

def load_feature_extractor(model_path: str,
                            device: torch.device,
                            condition: str):
    print(f"\n  Loading checkpoint [{condition}]: {model_path}")
    checkpoint = torch.load(model_path, map_location=device)

    if 'model_state_dict' in checkpoint:
        state_dict  = checkpoint['model_state_dict']
        print(f"  Checkpoint epoch : {checkpoint.get('epoch', 'N/A')}")
        best_metric = checkpoint.get('best_auc_pr',
                       checkpoint.get('best_val_auc', 'N/A'))
        print(f"  Best metric      : {best_metric}")
    else:
        state_dict = checkpoint
        print("  ⚠️  Raw state dict (no metadata wrapper)")

    # Build full model and load weights
    full_model = TumorClassifierConvNeXtTinySE(n=2, r=16)
    full_model.load_state_dict(state_dict, strict=True)

    # Wrap as feature extractor (strips head)
    extractor = ConvNeXtTinySEFeatureExtractor(full_model)
    extractor = extractor.to(device)
    extractor.eval()

    total_params = sum(p.numel() for p in extractor.parameters())
    print(f"  Extractor params : {total_params:,}  (head removed ✓)")
    print(f"  Output dim       : 768")

    # Verify forward pass
    with torch.no_grad():
        dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
        out   = extractor(dummy)
    assert out.shape == (1, 768), f"Expected (1, 768), got {out.shape}"
    print(f"  ✅ Forward pass verified: dummy → {out.shape}")

    return extractor


# ============================================================================
# EXTRACT FEATURES FOR ONE SPLIT
# ============================================================================

def extract_features(extractor,
                     paired_df: pd.DataFrame,
                     split_name: str,
                     device: torch.device):
    print(f"\n  Extracting: {split_name} ({len(paired_df)} ROIs)...")

    dataset = ROIFeatureDataset(paired_df, ROI_DATASET_DIR)
    loader  = DataLoader(
        dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = False,       # MUST be False — preserve row order
        num_workers = NUM_WORKERS,
        pin_memory  = True
    )

    all_features, all_labels, all_filenames = [], [], []

    extractor.eval()
    with torch.no_grad():
        for batch_imgs, batch_labels, batch_fnames in loader:
            feats = extractor(batch_imgs.to(device))   # (B, 768)
            all_features.append(feats.cpu().numpy())
            all_labels.extend(batch_labels.numpy().tolist())
            all_filenames.extend(batch_fnames)

    features = np.concatenate(all_features, axis=0)   # (N, 768)

    # ── Sanity checks ──────────────────────────────────────────────────────
    assert features.shape == (len(paired_df), 768), \
        f"Shape mismatch: {features.shape} vs expected ({len(paired_df)}, 768)"

    nan_count = np.isnan(features).sum()
    inf_count = np.isinf(features).sum()
    if nan_count > 0 or inf_count > 0:
        print(f"  ⚠️  NaN: {nan_count}, Inf: {inf_count}")
    else:
        print(f"  ✅ No NaN/Inf in extracted features")

    # Near-zero vector check (failed image loads)
    feat_norms = np.linalg.norm(features, axis=1)
    zero_mask  = feat_norms < 1e-6
    zero_vecs  = zero_mask.sum()
    if zero_vecs > 0:
        failed_files = [
            fname.replace("__LOAD_FAILED__", "")
            for fname in all_filenames
            if fname.startswith("__LOAD_FAILED__")
        ]
        all_filenames = [
            fname.replace("__LOAD_FAILED__", "") for fname in all_filenames
        ]
        error_msg = (
            f"\n  ❌ FATAL: {zero_vecs} near-zero feature vector(s) in '{split_name}'.\n"
            f"  These result from ROI files that could not be opened.\n"
            f"  Affected files ({len(failed_files)} identified):\n"
        )
        for i, f in enumerate(failed_files[:20]):
            error_msg += f"    [{i+1}] {f}\n"
        if len(failed_files) > 20:
            error_msg += f"    ... and {len(failed_files) - 20} more.\n"
        raise RuntimeError(error_msg)
    else:
        print(f"  ✅ No near-zero feature vectors")

    all_filenames = [f.replace("__LOAD_FAILED__", "") for f in all_filenames]

    print(f"  Feature stats: min={features.min():.4f}, max={features.max():.4f}, "
          f"mean={features.mean():.4f}, std={features.std():.4f}")
    print(f"  Feature shape : {features.shape}")

    labels_arr  = np.array(all_labels)
    n_benign    = (labels_arr == 0).sum()
    n_malignant = (labels_arr == 1).sum()
    print(f"  Label dist    : benign={n_benign}, malignant={n_malignant}")

    return features, labels_arr, all_filenames


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 70)
    print("  FUSION STEP 2: ConvNeXt-Tiny-SE FEATURE EXTRACTION")
    print("=" * 70)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device: {device}")
    if device.type == 'cuda':
        print(f"  GPU   : {torch.cuda.get_device_name(0)}")
        print(f"  VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Load paired CSVs (same for both conditions — only model weights differ)
    print("\n[1/3] Loading paired datasets from Step 1...")
    splits = {}
    for split_name in ['train', 'val', 'test']:
        path = Path(PAIRED_DIR) / f"fusion_paired_{split_name}.csv"
        df   = pd.read_csv(path)
        splits[split_name] = df
        print(f"  {split_name}: {len(df)} ROIs")

    # Run extraction for both aug conditions
    for condition, ckpt_path in CHECKPOINTS.items():
        print(f"\n{'=' * 70}")
        print(f"  CONDITION: {condition.upper()}")
        print(f"{'=' * 70}")

        output_dir = Path(OUTPUT_ROOT) / condition
        output_dir.mkdir(parents=True, exist_ok=True)

        extractor = load_feature_extractor(ckpt_path, device, condition)

        results = {}
        for split_name, paired_df in splits.items():
            features, labels, filenames = extract_features(
                extractor, paired_df, split_name, device
            )
            results[split_name] = {
                'features':  features,
                'labels':    labels,
                'filenames': filenames
            }

        # Save outputs
        print(f"\n[3/3] Saving feature arrays [{condition}]...")
        for split_name, data in results.items():
            feat_path = output_dir / f"image_features_{split_name}.npy"
            np.save(feat_path, data['features'])
            print(f"  ✅ {feat_path}  shape={data['features'].shape}")

            meta_df = splits[split_name][
                ['roi_filename', 'source_image', 'fusion_label',
                 'class', 'patient_id', 'match_type', 'confidence']
            ].copy().reset_index(drop=True)
            meta_df['feat_row_idx'] = np.arange(len(meta_df))
            meta_path = output_dir / f"image_features_{split_name}_meta.csv"
            meta_df.to_csv(meta_path, index=False)
            print(f"  ✅ {meta_path}")

        # Cross-split summary
        print(f"\n  {'Split':<8} {'N ROIs':>8} {'Benign':>8} {'Malignant':>10} "
              f"{'Feat Mean':>10} {'Feat Std':>10}")
        print(f"  {'-' * 56}")
        for split_name, data in results.items():
            feats = data['features']
            labs  = data['labels']
            print(f"  {split_name:<8} {len(feats):>8} "
                  f"{(labs==0).sum():>8} {(labs==1).sum():>10} "
                  f"{feats.mean():>10.4f} {feats.std():>10.4f}")

        train_mean = results['train']['features'].mean(axis=0)
        val_mean   = results['val']['features'].mean(axis=0)
        cosine_sim = float(
            np.dot(train_mean, val_mean) /
            (np.linalg.norm(train_mean) * np.linalg.norm(val_mean) + 1e-8)
        )
        print(f"\n  Train-Val cosine similarity: {cosine_sim:.4f} (expected > 0.9)")

        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print(f"""
  ✅ STEP 2 COMPLETE
  Features saved to:
    {OUTPUT_ROOT}/with_aug/image_features_{{train,val,test}}.npy  shape: (N, 768)
    {OUTPUT_ROOT}/no_aug/image_features_{{train,val,test}}.npy    shape: (N, 768)

  ⚠️  IMPORTANT FOR STEP 3:
    - Scaler fitted on TRAIN features ONLY (no leakage)
    - NO PCA — learned projection is inside FusionNet (Step 6)
    - Step 3 only computes mean/std for normalisation
""")


if __name__ == "__main__":
    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    main()

  FUSION STEP 2: ConvNeXt-Tiny-SE FEATURE EXTRACTION

  Device: cuda
  GPU   : Tesla T4
  VRAM  : 15.6 GB

[1/3] Loading paired datasets from Step 1...
  train: 1520 ROIs
  val: 295 ROIs
  test: 283 ROIs

  CONDITION: WITH_AUG

  Loading checkpoint [with_aug]: /kaggle/input/datasets/sadibhasan/convnext-best-classimodel/final_best_with_aug.pth
  Checkpoint epoch : 24
  Best metric      : N/A
  Extractor params : 27,892,320  (head removed ✓)
  Output dim       : 768
  ✅ Forward pass verified: dummy → torch.Size([1, 768])

  Extracting: train (1520 ROIs)...
  ✅ No NaN/Inf in extracted features
  ✅ No near-zero feature vectors
  Feature stats: min=-1.7525, max=1.5081, mean=0.0019, std=0.0828
  Feature shape : (1520, 768)
  Label dist    : benign=1297, malignant=223

  Extracting: val (295 ROIs)...
  ✅ No NaN/Inf in extracted features
  ✅ No near-zero feature vectors
  Feature stats: min=-1.5102, max=1.3287, mean=0.0018, std=0.0821
  Feature shape : (295, 768)
  Label dist    : benign=249, 

In [4]:
"""
FUSION STEP 3: FEATURE NORMALISATION (REPLACES PCA)
=====================================================
PCA has been removed. This step replaces it with StandardScaler
normalisation only. The dimensionality reduction (768 → 256 → 128)
is now a LEARNED projection inside FusionNet (Step 6), trained
jointly with the fusion head.

Why this is better than PCA:
  - PCA finds directions of maximum variance, not task-discriminative variance
  - Learned Linear(768→256) adapts to the fusion objective during training
  - No information is discarded before the model can evaluate its relevance
  - Directly addresses the main reviewer criticism of the classical pipeline

What this step does:
  1. Loads 768-dim ConvNeXt features from Step 2 (both conditions)
  2. Fits StandardScaler on TRAIN features only (no leakage)
  3. Transforms val/test using the fitted scaler
  4. Saves normalised features as norm_features_{split}.npy per condition
  5. Saves feature_stats.json (mean, std) per condition
  6. Plots feature distribution before/after normalisation

Output per condition per split:
  step3_norm/{condition}/norm_features_train.npy  shape: (N_train, 768)
  step3_norm/{condition}/norm_features_val.npy    shape: (N_val,   768)
  step3_norm/{condition}/norm_features_test.npy   shape: (N_test,  768)
  step3_norm/{condition}/feature_scaler.joblib    fitted StandardScaler
  step3_norm/{condition}/feature_stats.json       mean, std, percentiles

Q1 Safety:
  ✅ Scaler fitted on TRAIN only, applied to val/test
  ✅ Both conditions get independent scalers
  ✅ Scaler saved for inference reproducibility
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pathlib import Path
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

# ============================================================================
# CONFIGURATION
# ============================================================================
FEATURES_ROOT = "fusion_outputs/step2_features"   # Step 2 outputs
PAIRED_DIR    = "fusion_outputs/step1_paired"      # Step 1 outputs
OUTPUT_ROOT   = "fusion_outputs/step3_norm"
RANDOM_SEED   = 42

CONDITIONS = ['with_aug', 'no_aug']

FEAT_DIM = 768   # ConvNeXt-Tiny-SE output dimension


# ============================================================================
# HELPERS
# ============================================================================

def print_section(title: str):
    print(f"\n{'=' * 70}\n  {title}\n{'=' * 70}")


def check_features(name: str, feat: np.ndarray) -> None:
    nan_c = np.isnan(feat).sum()
    inf_c = np.isinf(feat).sum()
    if nan_c > 0 or inf_c > 0:
        raise ValueError(f"❌ NaN/Inf in {name}: NaN={nan_c}, Inf={inf_c}")


# ============================================================================
# PROCESS ONE CONDITION
# ============================================================================

def process_condition(condition: str) -> dict:
    print_section(f"CONDITION: {condition.upper()}")

    feat_dir   = Path(FEATURES_ROOT) / condition
    output_dir = Path(OUTPUT_ROOT) / condition
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Load raw 768-dim features ──────────────────────────────────────────
    print(f"\n[1/5] Loading raw 768-dim features ({condition})...")
    feat_train = np.load(feat_dir / "image_features_train.npy")
    feat_val   = np.load(feat_dir / "image_features_val.npy")
    feat_test  = np.load(feat_dir / "image_features_test.npy")

    meta_train = pd.read_csv(feat_dir / "image_features_train_meta.csv")
    meta_val   = pd.read_csv(feat_dir / "image_features_val_meta.csv")
    meta_test  = pd.read_csv(feat_dir / "image_features_test_meta.csv")

    print(f"  train: {feat_train.shape}")
    print(f"  val  : {feat_val.shape}")
    print(f"  test : {feat_test.shape}")

    # Verify expected dimensionality
    assert feat_train.shape[1] == FEAT_DIM, \
        f"Expected {FEAT_DIM}-dim, got {feat_train.shape[1]}"

    # Row alignment check
    assert len(feat_train) == len(meta_train), "Train feature/meta row mismatch"
    assert len(feat_val)   == len(meta_val),   "Val feature/meta row mismatch"
    assert len(feat_test)  == len(meta_test),  "Test feature/meta row mismatch"
    print(f"  ✅ Feature/metadata row alignment verified")

    # ── Pre-normalisation statistics ────────────────────────────────────────
    print(f"\n[2/5] Pre-normalisation statistics (train):")
    print(f"  min  = {feat_train.min():.4f}")
    print(f"  max  = {feat_train.max():.4f}")
    print(f"  mean = {feat_train.mean():.4f}")
    print(f"  std  = {feat_train.std():.4f}")

    # ConvNeXt uses GELU not ReLU so features CAN be negative
    neg_pct = (feat_train < 0).mean() * 100
    print(f"  % negative values: {neg_pct:.1f}%  "
          f"(expected >0% — ConvNeXt uses GELU, not ReLU)")

    # ── Fit StandardScaler on TRAIN only ────────────────────────────────────
    print(f"\n[3/5] Fitting StandardScaler on TRAIN only ({condition})...")
    scaler = StandardScaler()
    feat_train_norm = scaler.fit_transform(feat_train)
    feat_val_norm   = scaler.transform(feat_val)
    feat_test_norm  = scaler.transform(feat_test)

    print(f"  Post-normalisation train stats:")
    print(f"    mean = {feat_train_norm.mean():.6f}  (should be ~0)")
    print(f"    std  = {feat_train_norm.std():.6f}   (should be ~1)")
    print(f"    min  = {feat_train_norm.min():.4f}")
    print(f"    max  = {feat_train_norm.max():.4f}")

    # NaN/Inf check
    for name, feat in [('train', feat_train_norm),
                        ('val',   feat_val_norm),
                        ('test',  feat_test_norm)]:
        check_features(name, feat)
    print(f"  ✅ No NaN/Inf in any normalised features")

    # ── Class separability check ─────────────────────────────────────────────
    print(f"\n[4/5] Class separability check ({condition})...")
    train_labels = meta_train['fusion_label'].values
    b_feat = feat_train_norm[train_labels == 0]
    m_feat = feat_train_norm[train_labels == 1]

    # Mean feature-space distance
    dist = np.linalg.norm(b_feat.mean(0) - m_feat.mean(0))
    # Per-feature mean separation
    per_feat_sep = np.abs(b_feat.mean(0) - m_feat.mean(0))
    top_idx      = np.argsort(per_feat_sep)[::-1][:5]

    print(f"  Benign    n={len(b_feat)}  feature mean norm: {np.linalg.norm(b_feat.mean(0)):.4f}")
    print(f"  Malignant n={len(m_feat)}  feature mean norm: {np.linalg.norm(m_feat.mean(0)):.4f}")
    print(f"  Mean feature-space L2 distance: {dist:.4f}")
    print(f"  Top 5 most separating feature dims (by mean separation):")
    for idx in top_idx:
        print(f"    dim {idx:>4}: ben={b_feat[:,idx].mean():+.4f}  "
              f"mal={m_feat[:,idx].mean():+.4f}  "
              f"Δ={per_feat_sep[idx]:.4f}")

    # ── Save outputs ──────────────────────────────────────────────────────────
    print(f"\n[5/5] Saving normalised features and scaler ({condition})...")

    for split_name, feat in [('train', feat_train_norm),
                               ('val',   feat_val_norm),
                               ('test',  feat_test_norm)]:
        path = output_dir / f"norm_features_{split_name}.npy"
        np.save(path, feat)
        print(f"  ✅ {path}  shape={feat.shape}")

    scaler_path = output_dir / "feature_scaler.joblib"
    joblib.dump(scaler, scaler_path)
    print(f"  ✅ {scaler_path}")

    stats = {
        "condition":       condition,
        "feat_dim":        FEAT_DIM,
        "normalisation":   "StandardScaler (fit on train only)",
        "note":            "No PCA — learned projection Linear(768→256→128) is inside FusionNet",
        "train_pre_norm": {
            "n_samples": int(len(feat_train)),
            "min":  float(feat_train.min()),
            "max":  float(feat_train.max()),
            "mean": float(feat_train.mean()),
            "std":  float(feat_train.std()),
            "pct_negative": float(neg_pct),
        },
        "train_post_norm": {
            "mean": float(feat_train_norm.mean()),
            "std":  float(feat_train_norm.std()),
            "min":  float(feat_train_norm.min()),
            "max":  float(feat_train_norm.max()),
        },
        "val_post_norm": {
            "n_samples": int(len(feat_val)),
            "mean": float(feat_val_norm.mean()),
            "std":  float(feat_val_norm.std()),
        },
        "test_post_norm": {
            "n_samples": int(len(feat_test)),
            "mean": float(feat_test_norm.mean()),
            "std":  float(feat_test_norm.std()),
        },
        "class_separability": {
            "l2_distance_feature_means": float(dist),
            "benign_n":    int(len(b_feat)),
            "malignant_n": int(len(m_feat)),
        },
        "random_seed": RANDOM_SEED,
    }
    stats_path = output_dir / "feature_stats.json"
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"  ✅ {stats_path}")

    # ── Distribution figure ───────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(
        f"ConvNeXt-Tiny-SE Feature Distribution [{condition}] — train set",
        fontsize=12, fontweight='bold'
    )

    # Pre-norm: sample 1000 random feature values for histogram
    rng = np.random.default_rng(RANDOM_SEED)
    sample_pre  = rng.choice(feat_train.flatten(),  size=min(100000, feat_train.size),  replace=False)
    sample_post = rng.choice(feat_train_norm.flatten(), size=min(100000, feat_train_norm.size), replace=False)

    axes[0].hist(sample_pre, bins=80, color='steelblue', alpha=0.75, edgecolor='none')
    axes[0].set_title("Before StandardScaler")
    axes[0].set_xlabel("Feature value"); axes[0].set_ylabel("Count")
    axes[0].grid(alpha=0.3)

    axes[1].hist(sample_post, bins=80, color='darkorange', alpha=0.75, edgecolor='none')
    axes[1].set_title("After StandardScaler  (mean≈0, std≈1)")
    axes[1].set_xlabel("Feature value"); axes[1].set_ylabel("Count")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fig_path = output_dir / "feature_distribution.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ {fig_path}")

    return stats


# ============================================================================
# MAIN
# ============================================================================

def main():
    print_section("FUSION STEP 3: FEATURE NORMALISATION (NO PCA)")
    print("""
  ℹ️  PCA has been replaced by a learned projection inside FusionNet.
  This step ONLY applies StandardScaler normalisation.
  The projection Linear(768→256→128) lives in Step 6 and trains jointly
  with the fusion head, adapting to the task rather than to variance.
""")

    all_stats = {}
    for condition in CONDITIONS:
        stats = process_condition(condition)
        all_stats[condition] = stats

    print_section("STEP 3 COMPLETE — SUMMARY")
    for condition, stats in all_stats.items():
        print(f"\n  [{condition}]")
        print(f"    train n={stats['train_pre_norm']['n_samples']:<6} "
              f"post-norm mean={stats['train_post_norm']['mean']:.4f}  "
              f"std={stats['train_post_norm']['std']:.4f}")
        print(f"    val n  ={stats['val_post_norm']['n_samples']:<6} "
              f"post-norm mean={stats['val_post_norm']['mean']:.4f}  "
              f"std={stats['val_post_norm']['std']:.4f}")
        print(f"    test n ={stats['test_post_norm']['n_samples']:<6} "
              f"post-norm mean={stats['test_post_norm']['mean']:.4f}  "
              f"std={stats['test_post_norm']['std']:.4f}")
        print(f"    class L2 dist = {stats['class_separability']['l2_distance_feature_means']:.4f}")

    print(f"""
  Output structure:
    {OUTPUT_ROOT}/with_aug/norm_features_{{train,val,test}}.npy  shape: (N, 768)
    {OUTPUT_ROOT}/no_aug/norm_features_{{train,val,test}}.npy    shape: (N, 768)
    {{condition}}/feature_scaler.joblib   ← apply at inference
    {{condition}}/feature_stats.json      ← stats for methods section

  📝 Methods text:
    "ConvNeXt-Tiny-SE pooled features (768-dim) were standardised using
     StandardScaler fitted on the training set only. No PCA was applied;
     dimensionality reduction is instead performed by a learned linear
     projection (768→256→128) inside the fusion network, trained jointly
     with the fusion objective to preserve task-discriminative directions."

  ✅ Ready for Step 4 (clinical LR probs) and Step 5 (FusionDataset)
""")


if __name__ == "__main__":
    np.random.seed(RANDOM_SEED)
    main()


  FUSION STEP 3: FEATURE NORMALISATION (NO PCA)

  ℹ️  PCA has been replaced by a learned projection inside FusionNet.
  This step ONLY applies StandardScaler normalisation.
  The projection Linear(768→256→128) lives in Step 6 and trains jointly
  with the fusion head, adapting to the task rather than to variance.


  CONDITION: WITH_AUG

[1/5] Loading raw 768-dim features (with_aug)...
  train: (1520, 768)
  val  : (295, 768)
  test : (283, 768)
  ✅ Feature/metadata row alignment verified

[2/5] Pre-normalisation statistics (train):
  min  = -1.7525
  max  = 1.5081
  mean = 0.0019
  std  = 0.0828
  % negative values: 48.8%  (expected >0% — ConvNeXt uses GELU, not ReLU)

[3/5] Fitting StandardScaler on TRAIN only (with_aug)...
  Post-normalisation train stats:
    mean = -0.000000  (should be ~0)
    std  = 1.000000   (should be ~1)
    min  = -5.9010
    max  = 5.7774
  ✅ No NaN/Inf in any normalised features

[4/5] Class separability check (with_aug)...
  Benign    n=1297  feature m

In [5]:
"""
FUSION STEP 4: CLINICAL LR PROBABILITY EXTRACTION
==================================================
Loads the best SET-A GroupCalibratedEnsemble clinical model and extracts
calibrated p_malignant for every ROI across train/val/test splits.

Changes from original version:
  ✅ GroupCalibratedModel alias added (class was renamed between saving and loading)
  ✅ Bundle loaded as dict — model extracted via bundle["model_object"]
  ✅ Features passed as pd.DataFrame with named columns (ColumnTransformer requires this)
  ✅ Works with GradientBoostingClassifier as inner model (no code change needed)
  ✅ Output identical to original: lr_probs_{split}.npy shape (N, 1)
  ✅ Diagnostic block added to investigate compressed probability range

Output:
  step4_lr_probs/lr_probs_train.npy  shape: (N_train, 1)
  step4_lr_probs/lr_probs_val.npy    shape: (N_val,   1)
  step4_lr_probs/lr_probs_test.npy   shape: (N_test,  1)
  step4_lr_probs/lr_audit.json       metadata + sanity stats
"""

import os
import json
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression

# =============================================================================
# CONFIGURATION
# =============================================================================

CLINICAL_MODEL_PATH = "/kaggle/input/datasets/sadibhasan/clinical-bestmodel-xgboost/BEST_SET_A_model.joblib"
PAIRED_DIR          = "fusion_outputs/step1_paired"
OUTPUT_DIR          = "fusion_outputs/step4_lr_probs"
RANDOM_SEED         = 42

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


# =============================================================================
# CLASS DEFINITION — must appear before joblib.load()
# The joblib was saved when the class was named GroupCalibratedModel.
# We define it as GroupCalibratedEnsemble then alias the original name
# so pickle can find it during deserialisation.
# =============================================================================

class GroupCalibratedEnsemble(BaseEstimator, ClassifierMixin):
    """
    Exact copy from Stage 2 clinical notebook.
    Must be defined here so joblib can deserialise the saved instance.
    """
    def __init__(self, base_pipeline, cv_splitter, random_state=42):
        self.base_pipeline   = base_pipeline
        self.cv_splitter     = cv_splitter
        self.random_state    = random_state
        self.final_pipeline_ = None
        self.platt_scaler_   = None
        self.classes_        = np.array([0, 1], dtype=int)

    def fit(self, X, y, groups):
        y_arr = y.values if isinstance(y, pd.Series) else np.asarray(y)
        self.final_pipeline_ = clone(self.base_pipeline)
        self.final_pipeline_.fit(X, y_arr)
        oof_prob = np.full(len(y_arr), np.nan, dtype=float)
        for train_idx, cal_idx in self.cv_splitter.split(X, y_arr, groups):
            fold_model = clone(self.base_pipeline)
            X_train_fold = X.iloc[train_idx] if isinstance(X, pd.DataFrame) else X[train_idx]
            X_cal_fold   = X.iloc[cal_idx]   if isinstance(X, pd.DataFrame) else X[cal_idx]
            fold_model.fit(X_train_fold, y_arr[train_idx])
            oof_prob[cal_idx] = fold_model.predict_proba(X_cal_fold)[:, 1]
        if np.isnan(oof_prob).any():
            raise RuntimeError("OOF probabilities contain NaN.")
        self.platt_scaler_ = LogisticRegression(
            solver="lbfgs", max_iter=3000, random_state=self.random_state
        )
        self.platt_scaler_.fit(oof_prob.reshape(-1, 1), y_arr)
        return self

    def predict_proba(self, X):
        if self.final_pipeline_ is None or self.platt_scaler_ is None:
            raise RuntimeError("Model not fitted.")
        p_base = self.final_pipeline_.predict_proba(X)[:, 1]
        return self.platt_scaler_.predict_proba(p_base.reshape(-1, 1))  # (N, 2)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ── Alias: the name the class had when joblib.dump() was called ──────────────
GroupCalibratedModel = GroupCalibratedEnsemble


# =============================================================================
# LOAD BUNDLE
# =============================================================================

print("=" * 70)
print("  FUSION STEP 4: CLINICAL PROBABILITY EXTRACTION")
print("=" * 70)

print(f"\n[1/5] Loading clinical model bundle...")
bundle = joblib.load(CLINICAL_MODEL_PATH)

assert isinstance(bundle, dict), \
    f"Expected dict bundle, got {type(bundle)}. Check joblib file."

clinical_model    = bundle["model_object"]
threshold_image   = float(bundle["threshold_image_level"])
threshold_patient = bundle.get("threshold_patient_level")
feature_names     = bundle["feature_names_raw"]

print(f"  Best model name     : {bundle.get('best_model_name')}")
print(f"  Image threshold     : {threshold_image:.3f}")
print(f"  Patient threshold   : {threshold_patient}")
print(f"  Feature set         : {bundle.get('feature_set')}")
print(f"  N features          : {len(feature_names)}")
print(f"  Features            : {feature_names}")
print(f"  Pipeline steps      : {list(clinical_model.final_pipeline_.named_steps.keys())}")

# Inner classifier name — robust to any step name
inner_clf      = list(clinical_model.final_pipeline_.named_steps.values())[-1]
inner_clf_name = type(inner_clf).__name__
print(f"  Inner classifier    : {inner_clf_name}")
print(f"  Calibration         : OOF Platt scaling (LogisticRegression on OOF probs)")

# Stored VAL metrics
val_m = bundle.get("val_metrics_image_level", {})
if val_m:
    print(f"\n  Stored VAL metrics (image-level):")
    for k, v in val_m.items():
        print(f"    {k:<28}: {v:.4f}")


# =============================================================================
# DIAGNOSTIC BLOCK
# Runs immediately after load, before any paired CSV processing.
# Investigates whether the Platt scaler is compressing all probabilities
# into a narrow range — which would make the clinical signal useless for fusion.
#
# What to look for:
#   Raw GBM scores  — if these are already compressed (std < 0.01), the GBM
#                     itself is not discriminating. The clinical features may
#                     genuinely not separate benign/malignant in this dataset.
#   Platt coef      — if near zero, the Platt scaler has suppressed variation.
#   Output range    — if all calibrated probs are in a 0.03-wide band, the
#                     clinical branch in FusionNet will learn nothing from them.
# =============================================================================

print(f"\n{'─' * 70}")
print(f"  DIAGNOSTIC: probability compression check")
print(f"{'─' * 70}")

# Load a sample of training data to run the diagnostic
_diag_path = Path(PAIRED_DIR) / "fusion_paired_train.csv"
if _diag_path.exists():
    _diag_df     = pd.read_csv(_diag_path)
    _diag_sample = _diag_df.drop_duplicates(subset='source_image').head(100)
    _X_diag      = _diag_sample[feature_names].copy()
    _labels_diag = _diag_sample['fusion_label'].values

    # Step 1: Raw GBM score (before Platt scaling)
    _raw_scores = clinical_model.final_pipeline_.predict_proba(_X_diag)[:, 1]
    print(f"\n  Raw GBM scores (before Platt, n={len(_raw_scores)} unique images):")
    print(f"    min    : {_raw_scores.min():.4f}")
    print(f"    max    : {_raw_scores.max():.4f}")
    print(f"    mean   : {_raw_scores.mean():.4f}")
    print(f"    std    : {_raw_scores.std():.4f}")
    if _raw_scores.std() < 0.01:
        print(f"    ⚠️  std < 0.01 — GBM itself is near-constant.")
        print(f"       The clinical features have very low discriminative power")
        print(f"       in this dataset. This is a Stage 2 model quality issue.")
    else:
        print(f"    ✅ Raw GBM scores have meaningful spread")

    # Class-conditional raw scores
    _b_raw = _raw_scores[_labels_diag == 0]
    _m_raw = _raw_scores[_labels_diag == 1]
    if len(_b_raw) > 0 and len(_m_raw) > 0:
        print(f"    mean | benign    : {_b_raw.mean():.4f}")
        print(f"    mean | malignant : {_m_raw.mean():.4f}")
        print(f"    raw Δmean        : {_m_raw.mean() - _b_raw.mean():+.4f}")

    # Step 2: Platt scaler coefficients
    _platt     = clinical_model.platt_scaler_
    _platt_coef = float(_platt.coef_[0][0])
    _platt_int  = float(_platt.intercept_[0])
    print(f"\n  Platt scaler (LogisticRegression on OOF probs):")
    print(f"    coef      : {_platt_coef:.4f}")
    print(f"    intercept : {_platt_int:.4f}")
    if abs(_platt_coef) < 0.5:
        print(f"    ⚠️  coef near zero — Platt scaler has heavily suppressed variation.")
        print(f"       This happens when OOF raw scores had very low spread,")
        print(f"       causing LogisticRegression to learn a near-flat mapping.")
    else:
        print(f"    ✅ Platt coef has meaningful magnitude")

    # Step 3: Calibrated output range on this sample
    _cal_probs = clinical_model.predict_proba(_X_diag)[:, 1]
    print(f"\n  Calibrated output (after Platt, n={len(_cal_probs)}):")
    print(f"    min    : {_cal_probs.min():.4f}")
    print(f"    max    : {_cal_probs.max():.4f}")
    print(f"    range  : {_cal_probs.max() - _cal_probs.min():.4f}")
    print(f"    std    : {_cal_probs.std():.4f}")

    # Step 4: Verdict
    print(f"\n  VERDICT:")
    _range = _cal_probs.max() - _cal_probs.min()
    if _range < 0.05:
        print(f"  ⚠️  COMPRESSED: calibrated probability range is only {_range:.4f}.")
        print(f"     The clinical signal will contribute very little to FusionNet.")
        print(f"     Options:")
        print(f"       A) Proceed — FusionNet will down-weight the clinical branch")
        print(f"          automatically. The image branch will dominate, which is")
        print(f"          consistent with the paper's modality contribution analysis.")
        print(f"       B) Use raw GBM scores (uncalibrated) instead of Platt output.")
        print(f"          Add USE_RAW_GBM_SCORES = True below to enable this.")
        print(f"       C) Retrain Stage 2 clinical model with better hyperparameters.")
    elif _range < 0.15:
        print(f"  ⚠️  NARROW: range={_range:.4f}. Signal present but weak.")
        print(f"     FusionNet can learn from this but clinical branch will be minor.")
    else:
        print(f"  ✅ GOOD: calibrated probability range={_range:.4f}. Signal is usable.")

else:
    print(f"  ⚠️  Cannot run diagnostic: {_diag_path} not found yet.")
    print(f"     Diagnostic will run automatically once Step 1 has completed.")

print(f"{'─' * 70}\n")

# =============================================================================
# OPTION: USE RAW GBM SCORES INSTEAD OF PLATT OUTPUT
# Set to True if calibrated probs are too compressed (range < 0.05).
# Raw scores preserve more spread but are not calibrated probabilities.
# If True, outputs are still clipped to [0,1] and labelled accordingly in audit.
# =============================================================================
USE_RAW_GBM_SCORES = True   # ← set True if calibrated range < 0.05

if USE_RAW_GBM_SCORES:
    print(f"  ℹ️  USE_RAW_GBM_SCORES=True: will use raw GBM predict_proba output")
    print(f"     (before Platt scaling). These are not calibrated probabilities.")
    print(f"     Update your methods section accordingly.\n")


# =============================================================================
# LOAD PAIRED CSVS
# =============================================================================

print(f"\n[2/5] Loading paired datasets from Step 1...")
splits = {}
for split_name in ['train', 'val', 'test']:
    path = Path(PAIRED_DIR) / f"fusion_paired_{split_name}.csv"
    df   = pd.read_csv(path)
    splits[split_name] = df
    n_b = (df['fusion_label'] == 0).sum()
    n_m = (df['fusion_label'] == 1).sum()
    print(f"  {split_name}: {len(df)} ROIs  (benign={n_b}, malignant={n_m})")

# Check all required clinical features are present
missing_cols = [f for f in feature_names if f not in splits['train'].columns]
if missing_cols:
    raise ValueError(
        f"❌ These clinical features are missing from paired CSV: {missing_cols}\n"
        f"   Available columns: {list(splits['train'].columns)}"
    )
print(f"  ✅ All {len(feature_names)} clinical features present in paired CSVs")


# =============================================================================
# EXTRACT PROBABILITIES
# =============================================================================

print(f"\n[3/5] Extracting p_malignant per ROI...")

def extract_lr_probs(df: pd.DataFrame,
                     model,
                     feature_names: list,
                     split_name: str,
                     use_raw: bool = False) -> np.ndarray:
    """
    Extract calibrated p_malignant for every ROI in df.

    The clinical model was trained at IMAGE level (one row per image).
    Each ROI belongs to a source image and inherits that image's clinical
    features — so all ROIs from the same source image get the same p_malignant.
    This is correct: clinical metadata is patient/image-level, not ROI-level.

    Steps:
      1. Get unique source images and their clinical feature rows
      2. Run predict_proba once per unique source image
      3. Broadcast p_malignant back to all ROIs of that source image
      4. Return array of shape (N_rois, 1) in original row order

    Args:
      use_raw: if True, use raw GBM score before Platt scaling.
               Use when calibrated probs are too compressed to be useful.
    """
    # Build unique-image clinical feature matrix
    unique_imgs = df.drop_duplicates(subset='source_image')[
        ['source_image'] + feature_names
    ].copy().reset_index(drop=True)

    X_clinical = unique_imgs[feature_names].copy()

    # Extract probabilities
    if use_raw:
        # Raw GBM score — more spread, not calibrated
        probs_unique = model.final_pipeline_.predict_proba(X_clinical)[:, 1]
    else:
        # Calibrated via Platt scaler (default)
        probs_unique = model.predict_proba(X_clinical)[:, 1]

    # Build lookup: source_image → p_malignant
    img_to_prob = dict(zip(unique_imgs['source_image'], probs_unique))

    # Broadcast to all ROIs in original row order
    p_roi = df['source_image'].map(img_to_prob).values.astype(np.float32)

    # Clip to [0,1] — raw GBM scores are already in [0,1] but be safe
    p_roi = np.clip(p_roi, 0.0, 1.0)

    # Validate — no NaN allowed
    n_missing = np.isnan(p_roi).sum()
    if n_missing > 0:
        missing = df[np.isnan(p_roi)]['source_image'].unique()[:5]
        raise ValueError(
            f"❌ {n_missing} ROIs in '{split_name}' have no clinical probability.\n"
            f"   First missing source images: {missing}"
        )

    assert p_roi.min() >= 0.0 and p_roi.max() <= 1.0, \
        f"p_malignant out of [0,1] in {split_name}: " \
        f"min={p_roi.min():.4f} max={p_roi.max():.4f}"

    # Stats
    labels = df['fusion_label'].values
    print(f"\n  {split_name}:")
    print(f"    N ROIs              : {len(df)}")
    print(f"    N unique images     : {len(unique_imgs)}")
    print(f"    Source              : {'raw GBM (uncalibrated)' if use_raw else 'Platt-calibrated'}")
    print(f"    p_malignant range   : [{p_roi.min():.4f}, {p_roi.max():.4f}]")
    print(f"    p_malignant mean    : {p_roi.mean():.4f}")
    print(f"    p_malignant std     : {p_roi.std():.4f}")

    b_mean = p_roi[labels == 0].mean() if (labels == 0).any() else float('nan')
    m_mean = p_roi[labels == 1].mean() if (labels == 1).any() else float('nan')
    print(f"    Mean p | benign     : {b_mean:.4f}")
    print(f"    Mean p | malignant  : {m_mean:.4f}")
    print(f"    Effect size (Δmean) : {m_mean - b_mean:+.4f}  "
          f"({'✅ positive — malignant higher' if m_mean > b_mean else '⚠️  unexpected direction'})")

    pred_pos_rate = (p_roi >= threshold_image).mean()
    true_pos_rate = labels.mean()
    print(f"    Pred pos rate @{threshold_image:.2f}  : {pred_pos_rate:.4f}  "
          f"(true prevalence={true_pos_rate:.4f})")

    return p_roi.reshape(-1, 1)   # (N, 1)


all_probs = {}
for split_name, df in splits.items():
    all_probs[split_name] = extract_lr_probs(
        df, clinical_model, feature_names, split_name,
        use_raw=USE_RAW_GBM_SCORES
    )


# =============================================================================
# SAVE OUTPUTS
# =============================================================================

print(f"\n[4/5] Saving lr_probs arrays...")
for split_name, probs in all_probs.items():
    path = Path(OUTPUT_DIR) / f"lr_probs_{split_name}.npy"
    np.save(path, probs)
    print(f"  ✅ {path}  shape={probs.shape}  dtype={probs.dtype}")

# Save audit JSON
audit = {
    "clinical_model_path": CLINICAL_MODEL_PATH,
    "best_model_name":     bundle.get("best_model_name"),
    "inner_classifier":    inner_clf_name,
    "calibration":         "raw GBM score (uncalibrated)" if USE_RAW_GBM_SCORES
                           else "OOF Platt scaling (LogisticRegression)",
    "use_raw_gbm_scores":  USE_RAW_GBM_SCORES,
    "threshold_image":     threshold_image,
    "threshold_patient":   threshold_patient,
    "feature_names":       feature_names,
    "n_features":          len(feature_names),
    "splits": {
        split_name: {
            "n_rois":          int(len(splits[split_name])),
            "n_unique_images": int(splits[split_name]['source_image'].nunique()),
            "p_mean":          float(all_probs[split_name].mean()),
            "p_std":           float(all_probs[split_name].std()),
            "p_min":           float(all_probs[split_name].min()),
            "p_max":           float(all_probs[split_name].max()),
            "p_range":         float(all_probs[split_name].max() -
                                     all_probs[split_name].min()),
        }
        for split_name in ['train', 'val', 'test']
    },
    "note": (
        "p_malignant computed at image level and broadcast to all ROIs "
        "of that source image. Clinical metadata is image/patient-level "
        "not ROI-level — broadcasting is correct."
    )
}
audit_path = Path(OUTPUT_DIR) / "lr_audit.json"
with open(audit_path, 'w') as f:
    json.dump(audit, f, indent=2)
print(f"  ✅ {audit_path}")


# =============================================================================
# CROSS-SPLIT SANITY CHECK
# =============================================================================

print(f"\n[5/5] Cross-split sanity check...")
print(f"\n  {'Split':<8} {'N ROIs':>8} {'N Images':>10} "
      f"{'p mean':>8} {'p std':>8} {'p min':>8} {'p max':>8} {'p range':>8}")
print(f"  {'-' * 70}")
for split_name in ['train', 'val', 'test']:
    p  = all_probs[split_name]
    ni = splits[split_name]['source_image'].nunique()
    print(f"  {split_name:<8} {len(p):>8} {ni:>10} "
          f"{p.mean():>8.4f} {p.std():>8.4f} "
          f"{p.min():>8.4f} {p.max():>8.4f} "
          f"{(p.max()-p.min()):>8.4f}")

train_mean = all_probs['train'].mean()
val_mean   = all_probs['val'].mean()
test_mean  = all_probs['test'].mean()
drift_vt   = abs(val_mean  - train_mean)
drift_tt   = abs(test_mean - train_mean)
print(f"\n  Distribution drift (|mean_split - mean_train|):")
print(f"    val  drift : {drift_vt:.4f}  "
      f"{'✅ OK' if drift_vt < 0.10 else '⚠️  large drift'}")
print(f"    test drift : {drift_tt:.4f}  "
      f"{'✅ OK' if drift_tt < 0.10 else '⚠️  large drift'}")

# Final signal quality verdict
overall_range = float(all_probs['train'].max() - all_probs['train'].min())
print(f"\n  Clinical signal quality (train p_range = {overall_range:.4f}):")
if overall_range < 0.05:
    print(f"  ⚠️  WEAK — range < 0.05. Consider setting USE_RAW_GBM_SCORES = True")
    print(f"     and re-running. FusionNet will still run but clinical branch")
    print(f"     contribution will be minimal. Document this in your paper.")
elif overall_range < 0.15:
    print(f"  ⚠️  NARROW — range {overall_range:.4f}. Signal present but weak.")
    print(f"     FusionNet will learn from it but clinical branch will be minor.")
else:
    print(f"  ✅ GOOD — range {overall_range:.4f}. Clinical signal is usable.")

print(f"""
  ✅ STEP 4 COMPLETE
  Outputs:
    step4_lr_probs/lr_probs_train.npy  shape: {all_probs['train'].shape}
    step4_lr_probs/lr_probs_val.npy    shape: {all_probs['val'].shape}
    step4_lr_probs/lr_probs_test.npy   shape: {all_probs['test'].shape}
    step4_lr_probs/lr_audit.json

  USE_RAW_GBM_SCORES = {USE_RAW_GBM_SCORES}
  {'(using raw GBM score — more spread, not calibrated)' if USE_RAW_GBM_SCORES
   else '(using Platt-calibrated probabilities)'}

  These files feed directly into Step 5 FusionDataset.
  No other steps need changes — all downstream steps load from .npy files.
""")

  FUSION STEP 4: CLINICAL PROBABILITY EXTRACTION

[1/5] Loading clinical model bundle...
  Best model name     : Gradient Boosting
  Image threshold     : 0.160
  Patient threshold   : 0.14
  Feature set         : SET-A (clinical only — deployable)
  N features          : 23
  Features            : ['age', 'gender', 'hand', 'ulna', 'radius', 'humerus', 'foot', 'femur', 'tibia', 'fibula', 'hip bone', 'ankle-joint', 'knee-joint', 'hip-joint', 'wrist-joint', 'elbow-joint', 'shoulder-joint', 'upper limb', 'lower limb', 'pelvis', 'frontal', 'lateral', 'oblique']
  Pipeline steps      : ['pre', 'to_np', 'clf']
  Inner classifier    : GradientBoostingClassifier
  Calibration         : OOF Platt scaling (LogisticRegression on OOF probs)

  Stored VAL metrics (image-level):
    accuracy                    : 0.6486
    balanced_accuracy           : 0.7051
    mcc                         : 0.3303
    macro_f1                    : 0.6073
    roc_auc                     : 0.7450
    pr_auc         

In [6]:
"""
FUSION STEP 5: FUSIONDATASET + DATALOADER CONSTRUCTION
=======================================================
Updated for ConvNeXt-Tiny-SE features (768-dim, no PCA).

Changes from ResNet18SE version:
  ✅ IMG_FEAT_DIM: 128 → 768  (raw normalised ConvNeXt features)
  ✅ TOTAL_DIM:    151 → 791  (768 + 22 + 1)
  ✅ Feature source: step3_norm/ instead of step3_pca/
  ✅ File names: norm_features_{split}.npy instead of pca_features_{split}.npy
  ✅ Supports both aug conditions via AUG_CONDITION config variable
  ✅ Everything else (WeightedRandomSampler, collate_fn, smoke test) identical

Each sample:
  img_feat   : (768,)  normalised ConvNeXt-Tiny-SE features   [Step 3]
  clin_feat  : (22,)   clinical metadata                       [Step 1]
  lr_prob    : (1,)    LR p_malignant                          [Step 4]
  ──────────────────────────────────────────────────────────────────────
  fused total: (791,)  per sample
"""

import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# ── Set this to 'with_aug' or 'no_aug' before running ──────────────────────
AUG_CONDITION = 'no_aug'   # ← CHANGE THIS to 'no_aug' for second run

PAIRED_DIR   = "fusion_outputs/step1_paired"
NORM_DIR     = f"fusion_outputs/step3_norm/{AUG_CONDITION}"   # ← updated path
LR_PROBS_DIR = "fusion_outputs/step4_lr_probs"
OUTPUT_DIR   = f"fusion_outputs/step5_dataset/{AUG_CONDITION}"

# ── Feature dimensions ──────────────────────────────────────────────────────
IMG_FEAT_DIM  = 768   # ← ConvNeXt-Tiny-SE (was 128 for ResNet18SE+PCA)
CLIN_FEAT_DIM = 23
LR_PROB_DIM   = 1
TOTAL_DIM     = IMG_FEAT_DIM + CLIN_FEAT_DIM + LR_PROB_DIM   # 792

MALIGNANT_WEIGHT = 2.0
BATCH_SIZE   = 64
NUM_WORKERS  = 4
RANDOM_SEED  = 42

CLINICAL_FEATURES = [
    "age", "gender",
    "hand", "ulna", "radius", "humerus",
    "femur", "tibia", "fibula", "hip bone",
    "ankle-joint", "knee-joint", "hip-joint",
    "wrist-joint", "elbow-joint", "shoulder-joint",
    "upper limb", "lower limb", "pelvis",
    "frontal", "lateral", "oblique", "foot"
]


# ============================================================================
# REPRODUCIBILITY
# ============================================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(RANDOM_SEED)


# ============================================================================
# HELPERS
# ============================================================================

def print_section(title: str):
    print(f"\n{'=' * 70}\n  {title}\n{'=' * 70}")


# ============================================================================
# ALIGNMENT VALIDATOR
# ============================================================================

def validate_alignment(paired_df, img_feats, lr_probs, split_name):
    n_rows = len(paired_df)
    n_img  = len(img_feats)
    n_lr   = len(lr_probs)

    if not (n_rows == n_img == n_lr):
        raise ValueError(
            f"❌ Alignment failure in {split_name}: "
            f"paired_df={n_rows}, img_feats={n_img}, lr_probs={n_lr}"
        )

    missing = [f for f in CLINICAL_FEATURES if f not in paired_df.columns]
    if missing:
        raise ValueError(f"❌ Missing clinical features in {split_name}: {missing}")

    labels = paired_df['fusion_label'].values
    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError(f"❌ Non-binary labels in {split_name}: {np.unique(labels)}")

    assert img_feats.shape[1] == IMG_FEAT_DIM, \
        f"Expected {IMG_FEAT_DIM}-dim features, got {img_feats.shape[1]}"

    print(f"  ✅ {split_name}: {n_rows} rows aligned  "
          f"feat_dim={img_feats.shape[1]}  ✓")


# ============================================================================
# FUSION DATASET
# ============================================================================

class FusionDataset(Dataset):
    """
    Multimodal fusion dataset — updated for 768-dim ConvNeXt features.

    Each sample returns a dict:
      img_feat    : FloatTensor (768,)  normalised ConvNeXt features
      clin_feat   : FloatTensor (22,)   clinical metadata
      lr_prob     : FloatTensor (1,)    LR p_malignant
      label       : LongTensor scalar   0=benign, 1=malignant
      source_image: str
      roi_filename: str
      patient_id  : int
      match_type  : str
    """

    def __init__(self, paired_df, img_feats, lr_probs, split_name):
        self.split_name = split_name
        self.n_samples  = len(paired_df)

        self.img_feats  = torch.tensor(img_feats.astype(np.float32))   # (N, 768)
        self.clin_feats = torch.tensor(
            paired_df[CLINICAL_FEATURES].values.astype(np.float32)
        )                                                               # (N, 22)
        self.lr_probs   = torch.tensor(
            lr_probs.astype(np.float32)
        )                                                               # (N, 1)
        self.labels     = torch.tensor(
            paired_df['fusion_label'].values.astype(np.int64)
        )

        self.roi_filenames = paired_df['roi_filename'].tolist()
        self.source_images = paired_df['source_image'].tolist()
        self.match_types   = (
            paired_df['match_type'].tolist()
            if 'match_type' in paired_df.columns
            else ['unknown'] * self.n_samples
        )
        self.patient_ids = (
            paired_df['patient_id'].fillna(-1).astype(int).tolist()
            if 'patient_id' in paired_df.columns
            else [-1] * self.n_samples
        )

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        return {
            'img_feat':     self.img_feats[idx],    # (768,)
            'clin_feat':    self.clin_feats[idx],   # (22,)
            'lr_prob':      self.lr_probs[idx],     # (1,)
            'label':        self.labels[idx],
            'source_image': self.source_images[idx],
            'roi_filename': self.roi_filenames[idx],
            'patient_id':   self.patient_ids[idx],
            'match_type':   self.match_types[idx],
        }

    def get_labels(self):
        return self.labels.numpy()

    def get_fused_tensor(self):
        """Full concatenated feature matrix (N, 791). Useful for sanity checks."""
        return torch.cat([self.img_feats, self.clin_feats, self.lr_probs], dim=1)


# ============================================================================
# WEIGHTED SAMPLER
# ============================================================================

def build_weighted_sampler(dataset, malignant_weight=2.0, seed=42):
    labels       = dataset.get_labels()
    label_counts = Counter(labels.tolist())

    class_weights = {
        0: 1.0 / label_counts[0],
        1: malignant_weight / label_counts[1]
    }
    sample_weights = np.array(
        [class_weights[int(l)] for l in labels], dtype=np.float32
    )

    g = torch.Generator()
    g.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights),
        num_samples = len(sample_weights),
        replacement = True,
        generator   = g
    )

    print(f"  WeightedRandomSampler:")
    print(f"    Benign    {label_counts[0]:>5}  weight={class_weights[0]:.6f}")
    print(f"    Malignant {label_counts[1]:>5}  weight={class_weights[1]:.6f}  (×{malignant_weight})")
    ratio = class_weights[1] / class_weights[0]
    print(f"    Effective ratio (mal/ben): {ratio:.2f}x")
    return sampler


# ============================================================================
# COLLATE FUNCTION
# ============================================================================

def fusion_collate_fn(batch):
    return {
        'img_feat':     torch.stack([s['img_feat']  for s in batch]),
        'clin_feat':    torch.stack([s['clin_feat'] for s in batch]),
        'lr_prob':      torch.stack([s['lr_prob']   for s in batch]),
        'label':        torch.stack([s['label']     for s in batch]),
        'source_image': [s['source_image'] for s in batch],
        'roi_filename': [s['roi_filename'] for s in batch],
        'patient_id':   [s['patient_id']   for s in batch],
        'match_type':   [s['match_type']   for s in batch],
    }


# ============================================================================
# DATALOADERS
# ============================================================================

def build_dataloaders(datasets, batch_size=64, num_workers=4,
                       malignant_weight=2.0, seed=42):
    def seed_worker(worker_id):
        s = torch.initial_seed() % (2 ** 32)
        np.random.seed(s)
        random.seed(s)

    g = torch.Generator()
    g.manual_seed(seed)
    loaders = {}

    print(f"\n  Building TRAIN DataLoader...")
    train_sampler = build_weighted_sampler(datasets['train'], malignant_weight, seed)
    loaders['train'] = DataLoader(
        datasets['train'],
        batch_size     = batch_size,
        sampler        = train_sampler,
        num_workers    = num_workers,
        pin_memory     = True,
        drop_last      = False,
        worker_init_fn = seed_worker,
        generator      = g,
        collate_fn     = fusion_collate_fn
    )
    print(f"    Batches per epoch: {len(loaders['train'])}")

    for split_name in ['val', 'test']:
        print(f"\n  Building {split_name.upper()} DataLoader...")
        loaders[split_name] = DataLoader(
            datasets[split_name],
            batch_size     = batch_size,
            shuffle        = False,
            num_workers    = num_workers,
            pin_memory     = True,
            drop_last      = False,
            worker_init_fn = seed_worker,
            generator      = g,
            collate_fn     = fusion_collate_fn
        )
        print(f"    Batches: {len(loaders[split_name])}")

    return loaders


# ============================================================================
# SMOKE TEST
# ============================================================================

def smoke_test(loaders, split_name='train'):
    print(f"\n  Smoke test on '{split_name}'...")
    batch = next(iter(loaders[split_name]))
    B     = batch['label'].shape[0]

    print(f"    Batch size      : {B}")
    print(f"    img_feat shape  : {batch['img_feat'].shape}  (expected [{B}, {IMG_FEAT_DIM}])")
    print(f"    clin_feat shape : {batch['clin_feat'].shape}  (expected [{B}, {CLIN_FEAT_DIM}])")
    print(f"    lr_prob shape   : {batch['lr_prob'].shape}  (expected [{B}, {LR_PROB_DIM}])")
    print(f"    label shape     : {batch['label'].shape}")

    assert batch['img_feat'].dtype  == torch.float32
    assert batch['clin_feat'].dtype == torch.float32
    assert batch['lr_prob'].dtype   == torch.float32
    assert batch['label'].dtype     == torch.long
    print(f"    ✅ All dtypes correct")

    lr_min = batch['lr_prob'].min().item()
    lr_max = batch['lr_prob'].max().item()
    assert 0.0 <= lr_min and lr_max <= 1.0
    print(f"    lr_prob range   : [{lr_min:.4f}, {lr_max:.4f}]  ✅ [0,1]")

    fused = torch.cat([batch['img_feat'], batch['clin_feat'], batch['lr_prob']], dim=1)
    assert fused.shape == (B, TOTAL_DIM), \
        f"Fused shape mismatch: {fused.shape} vs ({B}, {TOTAL_DIM})"
    print(f"    fused vector    : {fused.shape}  ✅ ({B}, {TOTAL_DIM})")

    assert batch['img_feat'].shape == (B, IMG_FEAT_DIM)
    print(f"    ✅ All shape assertions passed")


# ============================================================================
# MAIN
# ============================================================================

print_section(f"FUSION STEP 5: FUSIONDATASET + DATALOADER  [{AUG_CONDITION}]")
print(f"\n  IMG_FEAT_DIM = {IMG_FEAT_DIM}  (ConvNeXt-Tiny-SE, no PCA)")
print(f"  TOTAL_DIM    = {TOTAL_DIM}  (768 + 22 + 1)")
print(f"  Feature source: {NORM_DIR}")

# [1] Load feature sources
print("\n[1/5] Loading feature arrays...")
datasets_raw = {}
for split_name in ['train', 'val', 'test']:
    paired_df = pd.read_csv(Path(PAIRED_DIR) / f"fusion_paired_{split_name}.csv")
    img_feats = np.load(Path(NORM_DIR)       / f"norm_features_{split_name}.npy")
    lr_probs  = np.load(Path(LR_PROBS_DIR)   / f"lr_probs_{split_name}.npy")
    datasets_raw[split_name] = (paired_df, img_feats, lr_probs)
    print(f"  {split_name}: paired={len(paired_df)}  "
          f"img={img_feats.shape}  lr={lr_probs.shape}")

# [2] Validate alignment
print("\n[2/5] Validating row alignment...")
for split_name, (paired_df, img_feats, lr_probs) in datasets_raw.items():
    validate_alignment(paired_df, img_feats, lr_probs, split_name)

# [3] Build datasets
print("\n[3/5] Building FusionDataset objects...")
datasets = {}
for split_name, (paired_df, img_feats, lr_probs) in datasets_raw.items():
    ds = FusionDataset(paired_df, img_feats, lr_probs, split_name)
    datasets[split_name] = ds
    labels   = ds.get_labels()
    n_b, n_m = (labels == 0).sum(), (labels == 1).sum()
    fused    = ds.get_fused_tensor()
    print(f"\n  {split_name}: {len(ds)} samples  "
          f"(benign={n_b}, malignant={n_m}  ratio={n_b/max(n_m,1):.2f}:1)")
    print(f"    Fused tensor: {fused.shape}  range=[{fused.min():.4f}, {fused.max():.4f}]")
    assert fused.shape == (len(ds), TOTAL_DIM), f"Fused shape mismatch: {fused.shape}"
print(f"\n  ✅ All FusionDatasets built")

# [4] Build DataLoaders
print("\n[4/5] Building DataLoaders...")
loaders = build_dataloaders(
    datasets,
    batch_size       = BATCH_SIZE,
    num_workers      = NUM_WORKERS,
    malignant_weight = MALIGNANT_WEIGHT,
    seed             = RANDOM_SEED
)

# [5] Smoke test
print("\n[5/5] Smoke tests...")
for split_name in ['train', 'val', 'test']:
    smoke_test(loaders, split_name)

# Save dataset stats
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
stats = {
    "aug_condition": AUG_CONDITION,
    "feature_source": NORM_DIR,
    "input_dims": {
        "img_feat":  IMG_FEAT_DIM,
        "clin_feat": CLIN_FEAT_DIM,
        "lr_prob":   LR_PROB_DIM,
        "total":     TOTAL_DIM
    },
    "splits": {
        s: {
            "n_samples":   len(datasets[s]),
            "n_benign":    int((datasets[s].get_labels() == 0).sum()),
            "n_malignant": int((datasets[s].get_labels() == 1).sum()),
        }
        for s in ['train', 'val', 'test']
    },
    "sampler": {"type": "WeightedRandomSampler", "malignant_weight": MALIGNANT_WEIGHT},
    "batch_size": BATCH_SIZE,
    "random_seed": RANDOM_SEED,
}
with open(Path(OUTPUT_DIR) / "dataset_stats.json", 'w') as f:
    json.dump(stats, f, indent=2)
print(f"\n  ✅ {OUTPUT_DIR}/dataset_stats.json")

print_section("STEP 5 COMPLETE")
print(f"""
  [{AUG_CONDITION}] Input to FusionNet:
    img_feat   : ({IMG_FEAT_DIM},)  normalised ConvNeXt features
    clin_feat  : ({CLIN_FEAT_DIM},)   clinical metadata
    lr_prob    : ({LR_PROB_DIM},)    LR p_malignant
    ──────────────────────────────
    total      : ({TOTAL_DIM},)

  DataLoaders ready:
    train : {len(loaders['train'])} batches (WeightedRandomSampler ×{MALIGNANT_WEIGHT})
    val   : {len(loaders['val'])} batches
    test  : {len(loaders['test'])} batches

  ✅ Ready for Step 6 FusionNet
""")


  FUSION STEP 5: FUSIONDATASET + DATALOADER  [no_aug]

  IMG_FEAT_DIM = 768  (ConvNeXt-Tiny-SE, no PCA)
  TOTAL_DIM    = 792  (768 + 22 + 1)
  Feature source: fusion_outputs/step3_norm/no_aug

[1/5] Loading feature arrays...
  train: paired=1520  img=(1520, 768)  lr=(1520, 1)
  val: paired=295  img=(295, 768)  lr=(295, 1)
  test: paired=283  img=(283, 768)  lr=(283, 1)

[2/5] Validating row alignment...
  ✅ train: 1520 rows aligned  feat_dim=768  ✓
  ✅ val: 295 rows aligned  feat_dim=768  ✓
  ✅ test: 283 rows aligned  feat_dim=768  ✓

[3/5] Building FusionDataset objects...

  train: 1520 samples  (benign=1297, malignant=223  ratio=5.82:1)
    Fused tensor: torch.Size([1520, 792])  range=[-6.5212, 6.2759]

  val: 295 samples  (benign=249, malignant=46  ratio=5.41:1)
    Fused tensor: torch.Size([295, 792])  range=[-5.2832, 6.0603]

  test: 283 samples  (benign=242, malignant=41  ratio=5.90:1)
    Fused tensor: torch.Size([283, 792])  range=[-5.0512, 5.9207]

  ✅ All FusionDatasets bui

In [7]:
"""
FUSION STEP 6: FUSIONNET ARCHITECTURE
======================================
Updated for ConvNeXt-Tiny-SE 768-dim features.

Changes from ResNet18SE version:
  ✅ IMG_FEAT_DIM: 128 → 768
  ✅ Image encoder: learned two-stage projection replaces PCA
       Linear(768→256) → LayerNorm → GELU → Dropout(0.4)   [projection]
       Linear(256→128) → BN → ReLU → Dropout(0.3)          [embedding]
     This matches ConvNeXt's own head style (LayerNorm + GELU) in the
     first stage, then transitions to the BN + ReLU style of the
     fusion head for the second stage.
  ✅ IMG_EMBED_DIM stays 128 (same fusion head — no downstream changes)
  ✅ TOTAL_DIM: 151 → 792
  ✅ FusionNetImageOnly updated with same 768→256→128 projection
  ✅ FusionNetClinicalOnly: unchanged
  ✅ All ablation models unchanged except image encoder input dim

Architecture summary:
  Image encoder:
    Linear(768→256) → LayerNorm(256) → GELU → Dropout(0.4)
    Linear(256→128) → BN(128) → ReLU → Dropout(0.3)
    → 128-dim image embedding

  Clinical encoder (unchanged):
    Linear(23→64) → BN(64) → ReLU → Dropout(0.2)
    Linear(64→32) → BN(32) → ReLU
    → 32-dim clinical embedding

  Fusion head (unchanged):
    cat([128, 32]) = 160-dim
    Linear(160→64) → BN → ReLU → Dropout(0.4)
    Linear(64→32)  → BN → ReLU → Dropout(0.3)
    Linear(32→2)
    → 2-class logits

This cell writes step6_fusionnet.py to disk AND defines classes in-memory.
"""

_step6_code = '''
import torch
import torch.nn as nn

# ── Dimensions ───────────────────────────────────────────────────────────────
IMG_FEAT_DIM   = 768   # ConvNeXt-Tiny-SE (was 128 for ResNet18SE+PCA)
CLIN_FEAT_DIM  = 23
LR_PROB_DIM    = 1
CLIN_INPUT_DIM = CLIN_FEAT_DIM + LR_PROB_DIM   # 24

IMG_PROJ_DIM   = 256   # Intermediate projection dimension (new)
IMG_EMBED_DIM  = 128   # Final image embedding (same as before — head unchanged)
CLIN_EMBED_DIM = 32    # Clinical embedding (unchanged)
FUSED_DIM      = IMG_EMBED_DIM + CLIN_EMBED_DIM  # 160  (unchanged)
NUM_CLASSES    = 2

TOTAL_DIM = IMG_FEAT_DIM + CLIN_FEAT_DIM + LR_PROB_DIM  # 792


# ── Building blocks ──────────────────────────────────────────────────────────

class DenseBlock(nn.Module):
    """Linear → BN → ReLU → Dropout  (standard fusion block)"""
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
        )
    def forward(self, x):
        return self.net(x)


class ConvNeXtProjectionBlock(nn.Module):
    """
    First-stage projection for ConvNeXt features.
    Matches ConvNeXt head style: Linear → LayerNorm → GELU → Dropout
    (ConvNeXt uses LayerNorm + GELU throughout, not BN + ReLU)
    """
    def __init__(self, in_dim=768, out_dim=256, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=True),   # bias=True with LayerNorm
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
        )
    def forward(self, x):
        return self.net(x)


# ── FusionNet (primary model) ─────────────────────────────────────────────────

class FusionNet(nn.Module):
    """
    Dual-encoder intermediate fusion network.

    Image encoder  (768 → 256 → 128):
      Stage 1: ConvNeXtProjectionBlock(768→256)   LayerNorm+GELU style
      Stage 2: DenseBlock(256→128)                BN+ReLU style

    Clinical encoder (23 → 64 → 32): unchanged from ResNet18SE version

    Fusion head (160 → 64 → 32 → 2): unchanged
    """
    def __init__(self,
                 dropout_proj=0.40,
                 dropout_img=0.30,
                 dropout_clin=0.20,
                 dropout_fusion=0.40):
        super().__init__()

        # Image encoder: two-stage learned projection
        self.img_projection = ConvNeXtProjectionBlock(
            IMG_FEAT_DIM, IMG_PROJ_DIM, dropout=dropout_proj
        )
        self.img_embedding = DenseBlock(
            IMG_PROJ_DIM, IMG_EMBED_DIM, dropout=dropout_img
        )

        # Clinical encoder: unchanged
        self.clinical_encoder = nn.Sequential(
            DenseBlock(CLIN_INPUT_DIM, 64, dropout=dropout_clin),
            DenseBlock(64, CLIN_EMBED_DIM, dropout=dropout_clin * 0.5),
        )

        # Fusion head: unchanged
        self.fusion_head = nn.Sequential(
            DenseBlock(FUSED_DIM, 64, dropout=dropout_fusion),
            DenseBlock(64, 32, dropout=dropout_fusion * 0.75),
            nn.Linear(32, NUM_CLASSES),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            # LayerNorm initialised to weight=1, bias=0 by default ✓

    def forward(self, img_feat, clin_feat, lr_prob):
        img_emb  = self.img_embedding(self.img_projection(img_feat))
        clin_emb = self.clinical_encoder(torch.cat([clin_feat, lr_prob], dim=1))
        return self.fusion_head(torch.cat([img_emb, clin_emb], dim=1))

    def get_embeddings(self, img_feat, clin_feat, lr_prob):
        """Returns (img_emb, clin_emb, fused) for interpretability/t-SNE."""
        img_emb  = self.img_embedding(self.img_projection(img_feat))
        clin_emb = self.clinical_encoder(torch.cat([clin_feat, lr_prob], dim=1))
        fused    = torch.cat([img_emb, clin_emb], dim=1)
        return img_emb, clin_emb, fused


# ── FusionNetImageOnly (ablation) ─────────────────────────────────────────────

class FusionNetImageOnly(nn.Module):
    """
    Ablation: image features only. Same 768→256→128 projection as FusionNet.
    """
    def __init__(self, dropout_proj=0.40, dropout_img=0.30):
        super().__init__()
        self.img_projection = ConvNeXtProjectionBlock(
            IMG_FEAT_DIM, IMG_PROJ_DIM, dropout=dropout_proj
        )
        self.img_embedding = DenseBlock(
            IMG_PROJ_DIM, IMG_EMBED_DIM, dropout=dropout_img
        )
        self.head = nn.Sequential(
            DenseBlock(IMG_EMBED_DIM, 32, dropout=0.30),
            nn.Linear(32, NUM_CLASSES)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, img_feat, clin_feat=None, lr_prob=None):
        x = self.img_embedding(self.img_projection(img_feat))
        return self.head(x)


# ── FusionNetClinicalOnly (ablation) ──────────────────────────────────────────

class FusionNetClinicalOnly(nn.Module):
    """
    Ablation: clinical features + LR prob only. Unchanged from ResNet18SE version
    (clinical branch does not depend on image feature dimension).
    """
    def __init__(self, dropout_clin=0.20):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            DenseBlock(CLIN_INPUT_DIM, 64, dropout=dropout_clin),
            DenseBlock(64, CLIN_EMBED_DIM, dropout=dropout_clin * 0.5),
        )
        self.head = nn.Sequential(
            DenseBlock(CLIN_EMBED_DIM, 16, dropout=0.20),
            nn.Linear(16, NUM_CLASSES)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, img_feat=None, clin_feat=None, lr_prob=None):
        return self.head(
            self.clinical_encoder(torch.cat([clin_feat, lr_prob], dim=1))
        )


# ── Registry + factory ────────────────────────────────────────────────────────

MODEL_REGISTRY = {
    "fusion":        FusionNet,
    "image_only":    FusionNetImageOnly,
    "clinical_only": FusionNetClinicalOnly,
}


def get_fusion_model(model_type="fusion", verbose=True):
    if model_type not in MODEL_REGISTRY:
        raise ValueError(
            f"Unknown model_type \'{model_type}\'. Options: {list(MODEL_REGISTRY)}"
        )
    model     = MODEL_REGISTRY[model_type]()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if verbose:
        print(f"  [{model_type}]  params={trainable:,}  "
              f"params/train_sample={trainable/1520:.2f}x")
    return model
'''

# ── Write to disk (so Step 7 import works) ───────────────────────────────────
import os
_save_path = os.path.join(os.getcwd(), "step6_fusionnet.py")
with open(_save_path, "w") as f:
    f.write(_step6_code)
print(f"✅ Wrote step6_fusionnet.py to: {_save_path}")

# ── Define in-memory (so this cell's namespace has the classes immediately) ──
exec(compile(_step6_code, "step6_fusionnet.py", "exec"), globals())
print("✅ Classes defined in-memory: FusionNet, FusionNetImageOnly, FusionNetClinicalOnly")

# ── Verification ─────────────────────────────────────────────────────────────
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n  Device: {device}")
print(f"\n  Architecture summary:")
print(f"    IMG_FEAT_DIM   = {IMG_FEAT_DIM}   (ConvNeXt-Tiny-SE, normalised)")
print(f"    IMG_PROJ_DIM   = {IMG_PROJ_DIM}    (learned projection, new)")
print(f"    IMG_EMBED_DIM  = {IMG_EMBED_DIM}    (unchanged — head compatible)")
print(f"    CLIN_EMBED_DIM = {CLIN_EMBED_DIM}     (unchanged)")
print(f"    FUSED_DIM      = {FUSED_DIM}    (unchanged)")
print(f"    TOTAL_DIM      = {TOTAL_DIM}    (792 = 768+23+1)")

print(f"\n  Model verification:")
for mt in ["fusion", "image_only", "clinical_only"]:
    m = get_fusion_model(mt, verbose=False).to(device)
    m.eval()
    with torch.no_grad():
        out = m(
            torch.randn(4, IMG_FEAT_DIM).to(device),
            torch.randn(4, CLIN_FEAT_DIM).to(device),
            torch.rand(4,  LR_PROB_DIM).to(device),
        )
    assert out.shape == (4, NUM_CLASSES), f"Shape error: {out.shape}"
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  ✅ {mt:<15}  output={tuple(out.shape)}  params={n_params:,}")

# Verify get_embeddings for interpretability (Step 9)
m_full = get_fusion_model("fusion", verbose=False).to(device)
m_full.eval()
with torch.no_grad():
    ie, ce, fe = m_full.get_embeddings(
        torch.randn(4, IMG_FEAT_DIM).to(device),
        torch.randn(4, CLIN_FEAT_DIM).to(device),
        torch.rand(4,  LR_PROB_DIM).to(device),
    )
assert ie.shape == (4, IMG_EMBED_DIM),  f"img_emb shape: {ie.shape}"
assert ce.shape == (4, CLIN_EMBED_DIM), f"clin_emb shape: {ce.shape}"
assert fe.shape == (4, FUSED_DIM),      f"fused shape: {fe.shape}"
print(f"\n  ✅ get_embeddings verified: img={ie.shape} clin={ce.shape} fused={fe.shape}")
print(f"\n  ✅ Step 6 complete — ready for Step 7")

✅ Wrote step6_fusionnet.py to: /kaggle/working/step6_fusionnet.py
✅ Classes defined in-memory: FusionNet, FusionNetImageOnly, FusionNetClinicalOnly

  Device: cuda

  Architecture summary:
    IMG_FEAT_DIM   = 768   (ConvNeXt-Tiny-SE, normalised)
    IMG_PROJ_DIM   = 256    (learned projection, new)
    IMG_EMBED_DIM  = 128    (unchanged — head compatible)
    CLIN_EMBED_DIM = 32     (unchanged)
    FUSED_DIM      = 160    (unchanged)
    TOTAL_DIM      = 792    (792 = 768+23+1)

  Model verification:
  ✅ fusion           output=(4, 2)  params=246,722
  ✅ image_only       output=(4, 2)  params=234,626
  ✅ clinical_only    output=(4, 2)  params=4,354

  ✅ get_embeddings verified: img=torch.Size([4, 128]) clin=torch.Size([4, 32]) fused=torch.Size([4, 160])

  ✅ Step 6 complete — ready for Step 7


In [8]:
"""
FUSION STEP 7: TRAINING LOOP
==============================
Self-contained — loads all data from saved files (no heavy re-imports).
Trains three model variants and saves checkpoints.

  Model variants:
    'fusion'        → FusionNet (image + clinical + LR prob)  ← primary
    'image_only'    → FusionNetImageOnly (ablation)
    'clinical_only' → FusionNetClinicalOnly (ablation)

  Training setup:
    Loss     : FocalLoss(α=0.65, γ=1.2) — same as Stage 3B
    Optimizer: AdamW(lr=1e-3, weight_decay=1e-3)
    Scheduler: LinearWarmup(5 ep) → ReduceLROnPlateau(patience=7)
    Imbalance: WeightedRandomSampler (already applied, Step 5) + FocalLoss
    Gradient clip: 1.0
    Early stop: patience=15 on val AUC-PR, min_epochs=20

  Threshold optimization:
    Optimized on VAL ONLY at every epoch (Q1 safe).
    Same composite score + clinical constraint (malignant recall ≥ 0.85)
    as your Stage 2 clinical model training — consistent pipeline.

  ROI→Image aggregation:
    Val predictions aggregated mean(ROI probs per image) before metrics.
    This matches Stage 3B eval logic.

  Saves per model under fusion_outputs/step7_training/{model_type}/:
    best_auc_pr.pth          ← best checkpoint by val AUC-PR
    training_history.json    ← full per-epoch log
    training_curves.png      ← 300 dpi publication figure
    threshold_table_val.csv  ← full threshold sweep at best epoch

Q1 Safety:
  ✅ All threshold selection on VAL only
  ✅ Test set never loaded in this script
  ✅ Best checkpoint selected on val AUC-PR (never test)

BUG FIXES (this revision):
  [Fix 4] roi_to_image max-pool: pa.max(0) produced an invalid probability
           vector (element-wise max across ROIs — doesn't sum to 1). Fixed to
           select the single ROI with the highest malignant probability
           (pa[argmax(pa[:,1])]), which always yields a valid softmax vector.

  [Fix 6] WarmupScheduler.in_warmup() used `<=` instead of `<`.
           With warmup_epochs=5 this caused the plateau scheduler to be
           skipped for one extra epoch (epoch 6) after warmup finished,
           because `_epoch == warmup_epochs` was still treated as in-warmup
           AFTER step() had already incremented _epoch past the boundary.
           Changed to `<` so plateau fires correctly from epoch 6 onward.

  [Fix 8] compute_metrics exception fallbacks for auc_roc and auc_pr
           returned 0.0 on failure. A degenerate epoch (all-same-class
           predictions) would log AUC=0.0, which looks like a valid (poor)
           score and can mislead early stopping. Changed to float('nan')
           to make failures obvious in the training log and history JSON.

  [Fix 14] DataLoader Generator not reset between model types. The
            WeightedRandomSampler uses a seeded torch.Generator that
            advances its state during training. After 'fusion' finishes,
            'image_only' and 'clinical_only' inherit the advanced state
            even though set_seed() resets global torch/numpy/random state.
            Fixed by rebuilding both loaders fresh inside the loop before
            each model type, ensuring identical batch orderings for all
            three ablation models.
"""

import os
import sys
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, matthews_corrcoef,
    confusion_matrix,
)

# Safe sys.path insert — works in both Jupyter and .py scripts
# __file__ is not defined in Jupyter cells, so we use a fallback
try:
    _this_dir = str(Path(__file__).parent)
except NameError:
    _this_dir = os.getcwd()   # Jupyter: use current working directory
sys.path.insert(0, _this_dir)

from step6_fusionnet import (
    FusionNet, FusionNetImageOnly, FusionNetClinicalOnly,
    get_fusion_model,
    IMG_FEAT_DIM, CLIN_FEAT_DIM, LR_PROB_DIM,
    MODEL_REGISTRY,
)

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG
# ============================================================================
PAIRED_DIR   = "fusion_outputs/step1_paired"
NORM_DIR      = "fusion_outputs/step3_norm/no_aug"
LR_PROBS_DIR = "fusion_outputs/step4_lr_probs"
OUTPUT_DIR   = "fusion_outputs/step7_training"

CLINICAL_FEATURES = [
    "age", "gender",
    "hand", "ulna", "radius", "humerus",
    "femur", "tibia", "fibula", "hip bone",
    "ankle-joint", "knee-joint", "hip-joint",
    "wrist-joint", "elbow-joint", "shoulder-joint",
    "upper limb", "lower limb", "pelvis",
    "frontal", "lateral", "oblique", "foot"
]

LEARNING_RATE    = 1e-3
WEIGHT_DECAY     = 1e-3
NUM_EPOCHS       = 100
WARMUP_EPOCHS    = 5
EARLY_STOP_PAT   = 15
MIN_EPOCHS       = 20
MIN_DELTA        = 1e-4
FOCAL_ALPHA      = 0.65
FOCAL_GAMMA      = 1.2
GRAD_CLIP        = 1.0
BATCH_SIZE       = 64
NUM_WORKERS      = 4
MALIGNANT_WEIGHT = 2.0
RANDOM_SEED      = 42

TARGET_MAL_RECALL = 0.85
HARD_MAX_POS_RATE = 0.90
ADAPTIVE_MARGIN   = 0.50
W_BAL, W_MCC, W_F1, W_POS = 1.00, 0.25, 0.10, 0.10

MODEL_TYPES = ['fusion', 'image_only', 'clinical_only']


# ============================================================================
# REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(RANDOM_SEED)


# ============================================================================
# DATASET (inline — avoids re-running heavy step5 code on import)
# ============================================================================
class FusionDataset(Dataset):
    def __init__(self, paired_df, img_feats, lr_probs):
        self.img_feats   = torch.tensor(img_feats.astype(np.float32))
        self.clin_feats  = torch.tensor(
            paired_df[CLINICAL_FEATURES].values.astype(np.float32)
        )
        self.lr_probs    = torch.tensor(lr_probs.astype(np.float32))
        self.labels      = torch.tensor(
            paired_df['fusion_label'].values.astype(np.int64)
        )
        self.source_imgs = paired_df['source_image'].tolist()
        self.roi_fnames  = paired_df['roi_filename'].tolist()
        self.patient_ids = (
            paired_df['patient_id'].fillna(-1).astype(int).tolist()
            if 'patient_id' in paired_df.columns
            else [-1] * len(paired_df)
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'img_feat':     self.img_feats[idx],
            'clin_feat':    self.clin_feats[idx],
            'lr_prob':      self.lr_probs[idx],
            'label':        self.labels[idx],
            'source_image': self.source_imgs[idx],
            'roi_filename': self.roi_fnames[idx],
            'patient_id':   self.patient_ids[idx],
        }


def collate_fn(batch):
    return {
        'img_feat':     torch.stack([s['img_feat']  for s in batch]),
        'clin_feat':    torch.stack([s['clin_feat'] for s in batch]),
        'lr_prob':      torch.stack([s['lr_prob']   for s in batch]),
        'label':        torch.stack([s['label']     for s in batch]),
        'source_image': [s['source_image'] for s in batch],
        'roi_filename': [s['roi_filename'] for s in batch],
        'patient_id':   [s['patient_id']   for s in batch],
    }


def build_loader(split_name: str, shuffle_val: bool = False):
    paired_df = pd.read_csv(
        Path(PAIRED_DIR) / f"fusion_paired_{split_name}.csv"
    )
    img_feats = np.load(
        Path(NORM_DIR) / f"norm_features_{split_name}.npy"
    )
    lr_probs  = np.load(
        Path(LR_PROBS_DIR) / f"lr_probs_{split_name}.npy"
    )
    ds = FusionDataset(paired_df, img_feats, lr_probs)

    if split_name == 'train':
        labels   = paired_df['fusion_label'].values
        counts   = np.bincount(labels)
        weights  = np.where(
            labels == 0,
            1.0 / counts[0],
            MALIGNANT_WEIGHT / counts[1]
        )
        # ✅ FIX (Issue 14): Generator is created fresh inside build_loader.
        #    Because build_loader is now called inside the per-model loop
        #    (after set_seed), the Generator always starts from the same
        #    seed for every model type, giving identical batch orderings
        #    across all three ablation runs.
        g = torch.Generator()
        g.manual_seed(RANDOM_SEED)
        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(weights.astype(np.float32)),
            num_samples=len(weights),
            replacement=True,
            generator=g,
        )
        return DataLoader(
            ds, batch_size=BATCH_SIZE, sampler=sampler,
            num_workers=NUM_WORKERS, pin_memory=True,
            drop_last=False, collate_fn=collate_fn,
        )
    else:
        return DataLoader(
            ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
            drop_last=False, collate_fn=collate_fn,
        )


# ============================================================================
# FOCAL LOSS (identical to Stage 3B)
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.65, gamma=1.2, label_smoothing=0.0):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, reduction='none',
                               label_smoothing=self.label_smoothing)
        pt   = torch.exp(-ce)
        at   = torch.where(targets == 1,
                            torch.full_like(ce, self.alpha),
                            torch.full_like(ce, 1.0 - self.alpha))
        return (at * (1 - pt) ** self.gamma * ce).mean()


# ============================================================================
# WARMUP SCHEDULER
# ============================================================================
class WarmupScheduler:
    def __init__(self, optimizer, warmup_epochs: int, base_lr: float):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.base_lr       = base_lr
        self._epoch        = 0

    def step(self):
        if self._epoch < self.warmup_epochs:
            lr = self.base_lr * (self._epoch + 1) / self.warmup_epochs
            for pg in self.optimizer.param_groups:
                pg['lr'] = lr
        self._epoch += 1

    def in_warmup(self):
        # ✅ FIX (Issue 6): was `self._epoch <= self.warmup_epochs`.
        #    With warmup_epochs=5 and the step()-then-check pattern in the
        #    training loop, the old code caused epoch 6 to still be treated
        #    as in-warmup (because after step() _epoch becomes 6, and
        #    6 <= 5 is False... wait, let me trace this carefully).
        #
        #    Training loop per epoch:
        #      if warmup.in_warmup(): warmup.step()   ← step increments _epoch
        #      ...
        #      if not warmup.in_warmup(): plateau.step()
        #
        #    With `<=`: at the START of epoch 6, _epoch=5.
        #      in_warmup() → 5 <= 5 = True  → warmup.step() fires (_epoch→6)
        #      in_warmup() → 6 <= 5 = False → plateau fires ✓ (looks correct)
        #
        #    BUT at the START of epoch 7, _epoch=6:
        #      in_warmup() → 6 <= 5 = False → warmup.step() skipped ✓
        #      in_warmup() → 6 <= 5 = False → plateau fires ✓
        #
        #    The real problem: at the START of epoch 6, _epoch=5 causes
        #    warmup.step() to fire one extra time (when it should already be
        #    done), calling the LR formula with _epoch=5:
        #      lr = base_lr * (5+1) / 5 = base_lr * 1.2  ← OVERSHOOTS base_lr!
        #    This gives a 20% LR spike above the intended maximum on epoch 6.
        #
        #    With `<`: at the START of epoch 6, _epoch=5:
        #      in_warmup() → 5 < 5 = False → warmup.step() skipped ✓
        #      in_warmup() → 5 < 5 = False → plateau fires ✓
        #    LR stays at base_lr from epoch 5 onward. No overshoot.
        return self._epoch < self.warmup_epochs


# ============================================================================
# ROI → IMAGE LEVEL AGGREGATION
# ============================================================================
def roi_to_image(labels, probs, source_images, method='mean'):
    """
    Aggregate ROI-level probabilities to image level.
    Returns (img_labels, img_probs) where img_probs has shape (N_img, 2).

    method='mean' : mean of all ROI softmax vectors for the image (default)
    method='max'  : select the single ROI with highest malignant probability
                    → returns that ROI's full softmax vector (sums to 1 ✓)

    BUG FIX (Issue 4):
      The original `pa.max(0)` computed the element-wise maximum ACROSS ROIs
      independently for each class dimension. For a 2-ROI image where
        ROI-A = [0.7, 0.3]  and  ROI-B = [0.2, 0.8],
      pa.max(0) returned [0.7, 0.8] which sums to 1.5 — not a valid
      probability vector. Any downstream threshold or AUC computation on
      this would be quietly wrong.

      Fix: instead of element-wise max, find the ROI with the highest
      malignant probability (pa[:, 1].argmax()) and return that row intact.
      This is the correct "most suspicious ROI" aggregation strategy.
    """
    grouped_p = defaultdict(list)
    grouped_l = defaultdict(list)
    for l, p, s in zip(labels, probs, source_images):
        grouped_p[s].append(p)
        grouped_l[s].append(int(l))

    img_labels, img_probs = [], []
    for s in sorted(grouped_p):
        img_labels.append(grouped_l[s][0])          # same label per image
        pa = np.stack(grouped_p[s])                  # shape: (N_rois, 2)
        if method == 'mean':
            img_probs.append(pa.mean(0))
        else:
            # ✅ FIX (Issue 4): Select ROI with highest malignant prob.
            #    pa[argmax] is a single softmax row → always sums to 1.
            best_roi_idx = int(np.argmax(pa[:, 1]))
            img_probs.append(pa[best_roi_idx])

    return np.array(img_labels), np.array(img_probs)   # probs: (N_img, 2)


# ============================================================================
# THRESHOLD OPTIMIZER (VAL ONLY — same logic as Stage 2 clinical model)
# ============================================================================
def optimize_threshold(p_mal, y_true, min_mal_recall=0.85):
    """Returns (threshold, metrics_dict, table_df, status_str)."""
    thresholds  = np.arange(0.05, 0.96, 0.01)
    true_pr     = float(y_true.mean())
    max_pos     = float(min(HARD_MAX_POS_RATE, true_pr + ADAPTIVE_MARGIN))

    rows = []
    for t in thresholds:
        yh  = (p_mal >= t).astype(int)
        bal = float(balanced_accuracy_score(y_true, yh))
        mf1 = float(f1_score(y_true, yh, average='macro', zero_division=0))
        mcc = float(matthews_corrcoef(y_true, yh)) \
              if len(np.unique(yh)) > 1 else -1.0
        _, rec, _, _ = precision_recall_fscore_support(
            y_true, yh, average=None, zero_division=0
        )
        mal_rec  = float(rec[1]) if len(rec) > 1 else 0.0
        pos_rate = float(yh.mean())
        score    = (W_BAL * bal + W_MCC * mcc + W_F1 * mf1
                    - W_POS * abs(pos_rate - true_pr))
        rows.append({
            'threshold': float(t), 'score': score,
            'balanced_accuracy': bal, 'macro_f1': mf1, 'mcc': mcc,
            'malignant_recall': mal_rec, 'pos_rate': pos_rate,
            'max_pos_rate': max_pos,
        })

    tbl   = pd.DataFrame(rows)
    valid = tbl[
        (tbl['malignant_recall'] >= min_mal_recall) &
        (tbl['pos_rate'] <= max_pos)
    ]
    if valid.empty:
        best = tbl.sort_values('score', ascending=False).iloc[0]
        return float(best['threshold']), best.to_dict(), tbl, 'FAILED_CONSTRAINT'

    best = valid.sort_values('score', ascending=False).iloc[0]
    return float(best['threshold']), best.to_dict(), tbl, 'OK'


# ============================================================================
# METRICS (image-level)
# ============================================================================
def compute_metrics(y_true, img_probs, threshold):
    p_mal = img_probs[:, 1]
    yhat  = (p_mal >= threshold).astype(int)

    # ✅ FIX (Issue 8): Changed fallback from 0.0 to float('nan').
    #    Returning 0.0 made degenerate epochs (e.g. all-same-class
    #    predictions where roc_auc_score raises ValueError) look like
    #    valid-but-poor scores. Early stopping then silently used 0.0
    #    as the AUC-PR reference value and could incorrectly save such
    #    a checkpoint. NaN makes failures visible in the training log,
    #    history JSON, and training curve plots.
    try:   auc_roc = float(roc_auc_score(y_true, p_mal))
    except: auc_roc = float('nan')   # ✅ FIX: was 0.0
    try:   auc_pr = float(average_precision_score(y_true, p_mal))
    except: auc_pr = float('nan')    # ✅ FIX: was 0.0

    bal   = float(balanced_accuracy_score(y_true, yhat))
    mf1   = float(f1_score(y_true, yhat, average='macro', zero_division=0))
    mcc   = float(matthews_corrcoef(y_true, yhat)) \
            if len(np.unique(yhat)) > 1 else -1.0
    prec, rec, _, _ = precision_recall_fscore_support(
        y_true, yhat, average=None, zero_division=0
    )
    cm          = confusion_matrix(y_true, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = float(tn / (tn + fp + 1e-9))

    return {
        'auc_roc':               auc_roc,
        'auc_pr':                auc_pr,
        'balanced_accuracy':     bal,
        'macro_f1':              mf1,
        'mcc':                   mcc,
        'malignant_recall':      float(rec[1]) if len(rec) > 1 else 0.0,
        'malignant_precision':   float(prec[1]) if len(prec) > 1 else 0.0,
        'specificity':           specificity,
        'benign_recall':         float(rec[0]),
        'benign_precision':      float(prec[0]),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'pos_rate_pred':         float(yhat.mean()),
        'pos_rate_true':         float(y_true.mean()),
        'threshold':             float(threshold),
    }


# ============================================================================
# TRAIN ONE EPOCH
# ============================================================================
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_logits, all_lbls = [], []

    for batch in loader:
        img  = batch['img_feat'].to(device)
        clin = batch['clin_feat'].to(device)
        lr   = batch['lr_prob'].to(device)
        lbls = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(img, clin, lr)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()
        all_logits.append(logits.detach().cpu())
        all_lbls.append(lbls.cpu())

    avg_loss = total_loss / len(loader)
    lgt  = torch.cat(all_logits)
    lbl  = torch.cat(all_lbls)
    acc  = (lgt.argmax(1) == lbl).float().mean().item()
    return avg_loss, acc


# ============================================================================
# VALIDATE
# ============================================================================
def run_val(model, loader, criterion, device, threshold=0.5):
    model.eval()
    total_loss, all_probs, all_lbls, all_srcs = 0.0, [], [], []

    with torch.no_grad():
        for batch in loader:
            img  = batch['img_feat'].to(device)
            clin = batch['clin_feat'].to(device)
            lr   = batch['lr_prob'].to(device)
            lbls = batch['label'].to(device)

            logits = model(img, clin, lr)
            loss   = criterion(logits, lbls)
            probs  = F.softmax(logits, dim=1)

            total_loss += loss.item()
            all_probs.append(probs.cpu().numpy())
            all_lbls.extend(lbls.cpu().numpy().tolist())
            all_srcs.extend(batch['source_image'])

    avg_loss   = total_loss / len(loader)
    roi_probs  = np.concatenate(all_probs)
    roi_labels = np.array(all_lbls)
    img_labels, img_probs = roi_to_image(roi_labels, roi_probs, all_srcs)
    return avg_loss, img_labels, img_probs


# ============================================================================
# TRAINING CURVES FIGURE (publication-ready)
# ============================================================================
def plot_training_curves(history, model_type, save_dir, best_epoch, best_thr):
    E   = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(
        f"FusionNet ({model_type}) — Training Curves",
        fontsize=13, fontweight='bold'
    )

    def vline(ax):
        ax.axvline(best_epoch, color='red', ls=':', lw=1.2, alpha=0.7,
                   label=f'Best ep={best_epoch}')

    # Loss
    ax = axes[0, 0]
    ax.plot(E, history['train_loss'], lw=2, label='Train')
    ax.plot(E, history['val_loss'],   lw=2, ls='--', label='Val')
    vline(ax); ax.set(title='FocalLoss', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # AUC
    auc_roc = [m['auc_roc'] for m in history['val_metrics']]
    auc_pr  = [m['auc_pr']  for m in history['val_metrics']]
    ax = axes[0, 1]
    ax.plot(E, auc_roc, lw=2,   label='AUC-ROC')
    ax.plot(E, auc_pr,  lw=2.5, label='AUC-PR ⭐ (early stop)')
    vline(ax); ax.set(title='AUC', xlabel='Epoch', ylabel='Score')
    ax.legend(); ax.grid(alpha=0.3)

    # BalAcc + MCC
    bal = [m['balanced_accuracy'] for m in history['val_metrics']]
    mcc = [m['mcc']               for m in history['val_metrics']]
    ax = axes[0, 2]
    ax.plot(E, bal, lw=2, label='BalAcc')
    ax.plot(E, mcc, lw=2, label='MCC')
    vline(ax); ax.set(title='BalAcc & MCC', xlabel='Epoch', ylabel='Score')
    ax.legend(); ax.grid(alpha=0.3)

    # Malignant recall + precision
    mal_rec  = [m['malignant_recall']    for m in history['val_metrics']]
    mal_prec = [m['malignant_precision'] for m in history['val_metrics']]
    ax = axes[1, 0]
    ax.plot(E, mal_rec,  lw=2, color='red',    label='Mal Recall')
    ax.plot(E, mal_prec, lw=2, color='orange', label='Mal Precision')
    ax.axhline(TARGET_MAL_RECALL, ls='--', color='gray', lw=1,
               label=f'Target recall={TARGET_MAL_RECALL}')
    vline(ax); ax.set(title='Malignant Recall & Precision',
                      xlabel='Epoch', ylabel='Score')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Sensitivity vs Specificity
    spec = [m['specificity'] for m in history['val_metrics']]
    ax = axes[1, 1]
    ax.plot(E, mal_rec, lw=2, color='red',   label='Sensitivity')
    ax.plot(E, spec,    lw=2, color='green', ls='--', label='Specificity')
    vline(ax); ax.set(title='Sensitivity vs Specificity',
                      xlabel='Epoch', ylabel='Rate')
    ax.legend(); ax.grid(alpha=0.3)

    # Threshold over epochs
    thrs = [m.get('threshold', 0.5) for m in history['val_metrics']]
    ax = axes[1, 2]
    ax.plot(E, thrs, lw=2, color='purple', label='Optimized threshold')
    ax.axhline(best_thr, ls='--', color='red', lw=1.2,
               label=f'Final t={best_thr:.3f}')
    vline(ax); ax.set(title='VAL Threshold Per Epoch',
                      xlabel='Epoch', ylabel='Threshold')
    ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    p = Path(save_dir) / "training_curves.png"
    plt.savefig(p, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ Saved: {p}")


# ============================================================================
# TRAIN ONE MODEL TYPE
# ============================================================================
class _NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.bool_,)):        return bool(o)
        if isinstance(o, np.integer):         return int(o)
        if isinstance(o, (np.floating, float)): return float(o)
        if isinstance(o, np.ndarray):         return o.tolist()
        return super().default(o)


def train_one(model_type: str, device: torch.device,
              train_loader, val_loader) -> dict:
    print(f"\n{'=' * 70}")
    print(f"  TRAINING: {model_type.upper()}")
    print(f"{'=' * 70}")

    save_dir = Path(OUTPUT_DIR) / model_type
    save_dir.mkdir(parents=True, exist_ok=True)

    model    = get_fusion_model(model_type).to(device)

    # Slight label smoothing on training loss only (same as Stage 3B)
    train_criterion = FocalLoss(FOCAL_ALPHA, FOCAL_GAMMA, label_smoothing=0.05)
    val_criterion   = FocalLoss(FOCAL_ALPHA, FOCAL_GAMMA, label_smoothing=0.0)

    optimizer = optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    warmup  = WarmupScheduler(optimizer, WARMUP_EPOCHS, LEARNING_RATE)
    plateau = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=7, min_lr=1e-6
    )

    best_auc_pr    = -float('inf')
    best_threshold = 0.5
    best_thr_status = 'UNKNOWN'
    best_epoch     = 0
    patience_ctr   = 0
    ckpt_path      = save_dir / "best_auc_pr.pth"

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_metrics': []
    }

    print(f"\n  Epochs={NUM_EPOCHS}  EarlyStop patience={EARLY_STOP_PAT} "
          f"min_epochs={MIN_EPOCHS}")
    print(f"  FocalLoss α={FOCAL_ALPHA} γ={FOCAL_GAMMA}  "
          f"Warmup={WARMUP_EPOCHS} ep  GradClip={GRAD_CLIP}")
    print(f"  Constraint: malignant recall ≥ {TARGET_MAL_RECALL}\n")

    for epoch in range(NUM_EPOCHS):
        # LR warmup — in_warmup() now uses < (not <=) so no LR overshoot
        if warmup.in_warmup():
            warmup.step()

        # Train
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer, train_criterion, device
        )

        # Validate (ROI→image aggregation inside run_val)
        val_loss, val_img_labels, val_img_probs = run_val(
            model, val_loader, val_criterion, device, best_threshold
        )

        # Optimize threshold on VAL (Q1 safe)
        p_mal    = val_img_probs[:, 1]
        opt_thr, best_row, thr_tbl, status = optimize_threshold(
            p_mal, val_img_labels, TARGET_MAL_RECALL
        )
        best_threshold = opt_thr
        best_thr_status = status

        metrics = compute_metrics(val_img_labels, val_img_probs, opt_thr)
        metrics['threshold'] = opt_thr
        cur_auc_pr = metrics['auc_pr']

        # Step plateau after warmup
        if not warmup.in_warmup():
            plateau.step(cur_auc_pr)

        cur_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(float(tr_loss))
        history['train_acc'].append(float(tr_acc))
        history['val_loss'].append(float(val_loss))
        history['val_metrics'].append(metrics)

        print(f"  Ep {epoch+1:>3}/{NUM_EPOCHS} "
              f"| TrLoss={tr_loss:.4f} TrAcc={tr_acc*100:.1f}%"
              f" | VLoss={val_loss:.4f}"
              f" | AUC-PR={cur_auc_pr:.4f}⭐ AUC-ROC={metrics['auc_roc']:.4f}"
              f" | BalAcc={metrics['balanced_accuracy']:.4f}"
              f" | MalRec={metrics['malignant_recall']:.4f}"
              f" | MalPrec={metrics['malignant_precision']:.4f}"
              f" | Thr={opt_thr:.3f}({status})"
              f" | LR={cur_lr:.1e}"
              f" | TP={metrics['tp']} FP={metrics['fp']}"
              f" TN={metrics['tn']} FN={metrics['fn']}")

        # Save best + early stopping
        # Note: if cur_auc_pr is NaN (degenerate epoch), comparison returns
        # False so we never save a NaN checkpoint. Training continues.
        improved     = (not np.isnan(cur_auc_pr)) and (cur_auc_pr > best_auc_pr + MIN_DELTA)
        before_min   = epoch + 1 < MIN_EPOCHS

        if improved:
            best_auc_pr     = cur_auc_pr
            best_epoch      = epoch + 1
            patience_ctr    = 0
            torch.save({
                'epoch':            epoch + 1,
                'model_type':       model_type,
                'model_state_dict': model.state_dict(),
                'optimizer_state':  optimizer.state_dict(),
                'best_auc_pr':      best_auc_pr,
                'threshold_val':    best_threshold,
                'threshold_status': status,
                'val_metrics':      metrics,
                'config': {
                    'model_type':        model_type,
                    'img_feat_dim':      IMG_FEAT_DIM,
                    'clin_feat_dim':     CLIN_FEAT_DIM,
                    'lr_prob_dim':       LR_PROB_DIM,
                    'focal_alpha':       FOCAL_ALPHA,
                    'focal_gamma':       FOCAL_GAMMA,
                    'target_mal_recall': TARGET_MAL_RECALL,
                },
            }, ckpt_path)
            print(f"  💾 SAVED  AUC-PR={best_auc_pr:.4f}  "
                  f"Thr={best_threshold:.3f}({status})  ep={best_epoch}")
        else:
            if not before_min:
                patience_ctr += 1
                print(f"  ⏳ Patience {patience_ctr}/{EARLY_STOP_PAT}  "
                      f"best={best_auc_pr:.4f} @ ep{best_epoch}")
            else:
                print(f"  🛡️  Min-epochs guard ({epoch+1}/{MIN_EPOCHS})")

        if patience_ctr >= EARLY_STOP_PAT and not before_min:
            print(f"\n  ⛔ Early stop — best AUC-PR={best_auc_pr:.4f} "
                  f"@ ep{best_epoch}")
            break

    # Save history + threshold table + figure
    with open(save_dir / "training_history.json", 'w') as f:
        json.dump(history, f, indent=2, cls=_NpEncoder)

    thr_tbl.to_csv(save_dir / "threshold_table_val.csv", index=False)
    plot_training_curves(
        history, model_type, str(save_dir), best_epoch, best_threshold
    )

    return {
        'model_type':       model_type,
        'best_auc_pr':      best_auc_pr,
        'best_epoch':       best_epoch,
        'best_threshold':   best_threshold,
        'threshold_status': best_thr_status,
        'val_metrics':      history['val_metrics'][best_epoch - 1]
                            if best_epoch > 0 else {},
        'checkpoint_path':  str(ckpt_path),
        'epochs_trained':   len(history['train_loss']),
    }


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("  FUSION STEP 7: TRAINING ALL FUSION MODEL VARIANTS")
    print("=" * 70)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device : {device}")
    if device.type == 'cuda':
        print(f"  GPU    : {torch.cuda.get_device_name(0)}")
        print(f"  VRAM   : "
              f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    all_results = {}

    for model_type in MODEL_TYPES:
        # ✅ FIX (Issue 14): Rebuild DataLoaders INSIDE the loop, AFTER set_seed().
        #    Previously loaders were built once outside the loop, so the
        #    WeightedRandomSampler's internal Generator state was carried over
        #    from the previous model's training run — meaning image_only and
        #    clinical_only saw different batch orderings than fusion.
        #    Now: set_seed() resets global state, then build_loader() creates a
        #    fresh Generator seeded with RANDOM_SEED, so all three models see
        #    identical batch sequences for a fair ablation comparison.
        set_seed(RANDOM_SEED)
        print(f"\n  Building DataLoaders for {model_type}...")
        train_loader = build_loader('train')
        val_loader   = build_loader('val')
        print(f"  Train batches: {len(train_loader)}  "
              f"Val batches: {len(val_loader)}")

        try:
            result = train_one(model_type, device, train_loader, val_loader)
            all_results[model_type] = result
        except Exception as e:
            import traceback
            print(f"\n❌ {model_type} failed: {e}")
            traceback.print_exc()
        finally:
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    # ── Summary table ────────────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print(f"  TRAINING SUMMARY (VAL — Q1 safe)")
    print(f"{'=' * 70}")
    rows = []
    for mt, res in all_results.items():
        m = res.get('val_metrics', {})
        row = {
            'Model':         mt,
            'Best_AUC_PR':   round(res['best_auc_pr'], 4),
            'AUC_ROC':       round(m.get('auc_roc', 0), 4),
            'BalAcc':        round(m.get('balanced_accuracy', 0), 4),
            'MCC':           round(m.get('mcc', 0), 4),
            'MalRecall':     round(m.get('malignant_recall', 0), 4),
            'MalPrec':       round(m.get('malignant_precision', 0), 4),
            'Threshold':     round(res['best_threshold'], 3),
            'Thr_Status':    res['threshold_status'],
            'Best_Epoch':    res['best_epoch'],
            'Epochs_Trained': res['epochs_trained'],
        }
        rows.append(row)
        print(f"\n  {mt.upper()}")
        print(f"    Best AUC-PR  : {res['best_auc_pr']:.4f} @ ep{res['best_epoch']}")
        print(f"    AUC-ROC      : {m.get('auc_roc', 0):.4f}")
        print(f"    Balanced Acc : {m.get('balanced_accuracy', 0):.4f}")
        print(f"    MCC          : {m.get('mcc', 0):.4f}")
        print(f"    MalRecall    : {m.get('malignant_recall', 0):.4f}")
        print(f"    MalPrec      : {m.get('malignant_precision', 0):.4f}")
        print(f"    Threshold    : {res['best_threshold']:.3f} "
              f"({res['threshold_status']})")
        print(f"    Checkpoint   : {res['checkpoint_path']}")

    summary_df = pd.DataFrame(rows)
    sp = Path(OUTPUT_DIR) / "training_summary.csv"
    summary_df.to_csv(sp, index=False)
    print(f"\n  ✅ Summary: {sp}")
    print(f"  ✅ Ready for Step 8: TEST evaluation + publication figures")

    return all_results


if __name__ == "__main__":
    main()

  FUSION STEP 7: TRAINING ALL FUSION MODEL VARIANTS

  Device : cuda
  GPU    : Tesla T4
  VRAM   : 15.6 GB

  Building DataLoaders for fusion...
  Train batches: 24  Val batches: 5

  TRAINING: FUSION
  [fusion]  params=246,722  params/train_sample=162.32x

  Epochs=100  EarlyStop patience=15 min_epochs=20
  FocalLoss α=0.65 γ=1.2  Warmup=5 ep  GradClip=1.0
  Constraint: malignant recall ≥ 0.85

  Ep   1/100 | TrLoss=0.1953 TrAcc=63.8% | VLoss=0.1303 | AUC-PR=0.5638⭐ AUC-ROC=0.8553 | BalAcc=0.8123 | MalRec=0.8696 | MalPrec=0.4545 | Thr=0.600(OK) | LR=2.0e-04 | TP=40 FP=48 TN=148 FN=6
  💾 SAVED  AUC-PR=0.5638  Thr=0.600(OK)  ep=1
  Ep   2/100 | TrLoss=0.1387 TrAcc=72.0% | VLoss=0.0850 | AUC-PR=0.8710⭐ AUC-ROC=0.9471 | BalAcc=0.9196 | MalRec=0.9565 | MalPrec=0.6567 | Thr=0.590(OK) | LR=4.0e-04 | TP=44 FP=23 TN=173 FN=2
  💾 SAVED  AUC-PR=0.8710  Thr=0.590(OK)  ep=2
  Ep   3/100 | TrLoss=0.0915 TrAcc=79.8% | VLoss=0.0504 | AUC-PR=0.9390⭐ AUC-ROC=0.9687 | BalAcc=0.9412 | MalRec=0.9130 | Ma

In [12]:
"""
FUSION STEP 8: TEST EVALUATION + PUBLICATION FIGURES
======================================================
FIRST and ONLY time the test set is touched.
Loads best checkpoints from Step 7, evaluates on TEST,
compares all models against the Stage 2 clinical LR baseline.

BUG FIXES (this revision):
  [Fix 4a]     roi_to_image: element-wise pa.max(0) replaced with
               pa[argmax(pa[:,1])] — selects the single most suspicious ROI.

  [Fix 4b]     roi_to_image_with_patients: same max-pool fix.

  [Fix LR-align] bootstrap_ci for clinical_lr now uses patient_ids derived
               directly from lr_audit_sorted (not the fusion model list).

  [Fix LR-shape] lr_probs_test.npy has shape (N,1) from Step 4 — only
               p_malignant is stored. Extract column 0, not 1.
               Handles (N,1), (N,2), and 1-D automatically.
"""

import os
import sys
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, matthews_corrcoef,
    confusion_matrix, roc_curve, precision_recall_curve,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

try:
    _this_dir = str(Path(__file__).parent)
except NameError:
    _this_dir = os.getcwd()
sys.path.insert(0, _this_dir)

from step6_fusionnet import (
    FusionNet, FusionNetImageOnly, FusionNetClinicalOnly,
    get_fusion_model,
    IMG_FEAT_DIM, CLIN_FEAT_DIM, LR_PROB_DIM,
    MODEL_REGISTRY,
)

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG
# ============================================================================
PAIRED_DIR   = "fusion_outputs/step1_paired"
NORM_DIR     = "fusion_outputs/step3_norm/no_aug"
LR_PROBS_DIR = "fusion_outputs/step4_lr_probs"
TRAINING_DIR = "fusion_outputs/step7_training"
OUTPUT_DIR   = "fusion_outputs/step8_evaluation"

CLINICAL_FEATURES = [
    "age", "gender",
    "hand", "ulna", "radius", "humerus",
    "femur", "tibia", "fibula", "hip bone",
    "ankle-joint", "knee-joint", "hip-joint",
    "wrist-joint", "elbow-joint", "shoulder-joint",
    "upper limb", "lower limb", "pelvis",
    "frontal", "lateral", "oblique", "foot"
]

BATCH_SIZE  = 64
RANDOM_SEED = 42
N_BOOT      = 2000
MODEL_TYPES = ['fusion', 'image_only', 'clinical_only']

PALETTE = {
    'fusion':        '#1f77b4',
    'image_only':    '#ff7f0e',
    'clinical_only': '#2ca02c',
    'clinical_lr':   '#9467bd',
}
MODEL_LABELS = {
    'fusion':        'FusionNet (proposed)',
    'image_only':    'Image-Only MLP',
    'clinical_only': 'Clinical-Only MLP',
    'clinical_lr':   'Clinical LR (Stage 2 baseline)',
}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


# ============================================================================
# HELPERS
# ============================================================================
def p(title):
    print(f"\n{'=' * 70}\n  {title}\n{'=' * 70}")


try:
    import joblib
    from sklearn.base import BaseEstimator, ClassifierMixin, clone
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.linear_model import LogisticRegression as _LR

    class GroupCalibratedEnsemble(BaseEstimator, ClassifierMixin):
        """Exact copy from Stage 2. Required for joblib deserialization."""
        def __init__(self, base_pipeline, cv_splitter, random_state=42):
            self.base_pipeline   = base_pipeline
            self.cv_splitter     = cv_splitter
            self.random_state    = random_state
            self.final_pipeline_ = None
            self.platt_scaler_   = None
            self.classes_        = np.array([0, 1], dtype=int)

        def fit(self, X, y, groups):
            y_arr = y.values if hasattr(y, 'values') else np.asarray(y)
            self.final_pipeline_ = clone(self.base_pipeline)
            self.final_pipeline_.fit(X, y_arr)
            oof = np.full(len(y_arr), np.nan)
            for ti, ci in self.cv_splitter.split(X, y_arr, groups):
                fm = clone(self.base_pipeline)
                Xtr = X.iloc[ti] if hasattr(X, 'iloc') else X[ti]
                Xca = X.iloc[ci] if hasattr(X, 'iloc') else X[ci]
                fm.fit(Xtr, y_arr[ti])
                oof[ci] = fm.predict_proba(Xca)[:, 1]
            self.platt_scaler_ = _LR(solver='lbfgs', max_iter=3000,
                                     random_state=self.random_state)
            self.platt_scaler_.fit(oof.reshape(-1, 1), y_arr)
            return self

        def predict_proba(self, X):
            p_raw = self.final_pipeline_.predict_proba(X)[:, 1]
            return self.platt_scaler_.predict_proba(p_raw.reshape(-1, 1))

        def predict(self, X):
            return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    _JOBLIB_OK = True
except ImportError:
    _JOBLIB_OK = False


# ============================================================================
# DATA LOADING
# ============================================================================
def load_test_data():
    paired   = pd.read_csv(Path(PAIRED_DIR) / "fusion_paired_test.csv")
    img_feat = np.load(Path(NORM_DIR)       / "norm_features_test.npy")
    lr_probs = np.load(Path(LR_PROBS_DIR)   / "lr_probs_test.npy")
    assert len(paired) == len(img_feat) == len(lr_probs), "Row mismatch"
    return paired, img_feat, lr_probs


# ============================================================================
# INFERENCE
# ============================================================================
def run_inference(model, paired_df, img_feat, lr_probs, device):
    model.eval()
    img_t  = torch.tensor(img_feat.astype(np.float32))
    clin_t = torch.tensor(paired_df[CLINICAL_FEATURES].values.astype(np.float32))
    lr_t   = torch.tensor(lr_probs.astype(np.float32))
    lbl_t  = torch.tensor(paired_df['fusion_label'].values.astype(np.int64))

    all_probs = []
    n = len(img_t)
    with torch.no_grad():
        for i in range(0, n, BATCH_SIZE):
            logits = model(img_t[i:i+BATCH_SIZE].to(device),
                           clin_t[i:i+BATCH_SIZE].to(device),
                           lr_t[i:i+BATCH_SIZE].to(device))
            all_probs.append(F.softmax(logits, dim=1).cpu().numpy())

    return lbl_t.numpy(), np.concatenate(all_probs)


# ============================================================================
# ROI → IMAGE AGGREGATION
# ============================================================================
def roi_to_image(roi_labels, roi_probs, source_images, method='mean'):
    """
    [Fix 4a] For method='max', select the ROI with highest p_malignant
    (valid softmax row) instead of element-wise max across ROIs.
    """
    grouped_p = defaultdict(list)
    grouped_l = defaultdict(list)
    for l, prob, s in zip(roi_labels, roi_probs, source_images):
        grouped_p[s].append(prob)
        grouped_l[s].append(int(l))

    img_labels, img_probs = [], []
    for s in sorted(grouped_p):
        img_labels.append(grouped_l[s][0])
        pa = np.stack(grouped_p[s])
        if method == 'mean':
            img_probs.append(pa.mean(0))
        else:
            img_probs.append(pa[int(np.argmax(pa[:, 1]))])  # Fix 4a

    return np.array(img_labels), np.array(img_probs)


def roi_to_image_with_patients(roi_labels, roi_probs, source_images,
                                patient_ids, method='mean'):
    """[Fix 4b] Same max-pool fix + returns patient_ids."""
    grouped_p   = defaultdict(list)
    grouped_l   = defaultdict(list)
    grouped_pid = defaultdict(list)
    for l, prob, s, pid in zip(roi_labels, roi_probs, source_images, patient_ids):
        grouped_p[s].append(prob)
        grouped_l[s].append(int(l))
        grouped_pid[s].append(int(pid))

    img_labels, img_probs, img_pids = [], [], []
    for s in sorted(grouped_p):
        img_labels.append(grouped_l[s][0])
        img_pids.append(grouped_pid[s][0])
        pa = np.stack(grouped_p[s])
        if method == 'mean':
            img_probs.append(pa.mean(0))
        else:
            img_probs.append(pa[int(np.argmax(pa[:, 1]))])  # Fix 4b

    return np.array(img_labels), np.array(img_probs), np.array(img_pids)


# ============================================================================
# COMPREHENSIVE METRICS
# ============================================================================
def compute_metrics(y_true, img_probs, threshold):
    p_mal = img_probs[:, 1]
    yhat  = (p_mal >= threshold).astype(int)

    try:   auc_roc = float(roc_auc_score(y_true, p_mal))
    except: auc_roc = float('nan')
    try:   auc_pr = float(average_precision_score(y_true, p_mal))
    except: auc_pr = float('nan')

    bal   = float(balanced_accuracy_score(y_true, yhat))
    mf1   = float(f1_score(y_true, yhat, average='macro', zero_division=0))
    mcc   = float(matthews_corrcoef(y_true, yhat)) if len(np.unique(yhat)) > 1 else float('nan')
    brier = float(brier_score_loss(y_true, p_mal))

    prec, rec, _, _ = precision_recall_fscore_support(y_true, yhat, average=None, zero_division=0)
    cm = confusion_matrix(y_true, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        'auc_roc':             auc_roc,
        'auc_pr':              auc_pr,
        'brier':               brier,
        'balanced_accuracy':   bal,
        'macro_f1':            mf1,
        'mcc':                 mcc,
        'malignant_recall':    float(rec[1]) if len(rec) > 1 else float('nan'),
        'malignant_precision': float(prec[1]) if len(prec) > 1 else float('nan'),
        'specificity':         float(tn / (tn + fp + 1e-9)),
        'benign_recall':       float(rec[0]),
        'benign_precision':    float(prec[0]),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'pos_rate_pred':       float(yhat.mean()),
        'pos_rate_true':       float(y_true.mean()),
        'threshold':           float(threshold),
        'n_images':            int(len(y_true)),
        'n_malignant':         int(y_true.sum()),
        'n_benign':            int((1 - y_true).sum()),
    }


# ============================================================================
# BOOTSTRAP 95% CI — PATIENT RESAMPLED
# ============================================================================
def bootstrap_ci(y_true, img_probs, patient_ids, threshold, n_boot=2000, seed=42):
    rng   = np.random.default_rng(seed)
    pids  = np.unique(patient_ids)
    p_mal = img_probs[:, 1]

    metric_keys = [
        'auc_roc', 'auc_pr', 'brier', 'balanced_accuracy',
        'macro_f1', 'mcc', 'malignant_recall', 'malignant_precision', 'specificity',
    ]
    boot_rows = []
    for _ in range(n_boot):
        sampled = rng.choice(pids, size=len(pids), replace=True)
        mask    = np.isin(patient_ids, sampled)
        if mask.sum() == 0: continue
        yt, pm = y_true[mask], p_mal[mask]
        if len(np.unique(yt)) < 2: continue
        try:
            row = {
                'auc_roc': float(roc_auc_score(yt, pm)),
                'auc_pr':  float(average_precision_score(yt, pm)),
                'brier':   float(brier_score_loss(yt, pm)),
            }
            yh = (pm >= threshold).astype(int)
            row['balanced_accuracy'] = float(balanced_accuracy_score(yt, yh))
            row['macro_f1']          = float(f1_score(yt, yh, average='macro', zero_division=0))
            row['mcc']               = float(matthews_corrcoef(yt, yh)) if len(np.unique(yh)) > 1 else float('nan')
            pr, re, _, _ = precision_recall_fscore_support(yt, yh, average=None, zero_division=0)
            row['malignant_recall']    = float(re[1]) if len(re) > 1 else float('nan')
            row['malignant_precision'] = float(pr[1]) if len(pr) > 1 else float('nan')
            tn_, fp_, _, _ = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
            row['specificity'] = float(tn_ / (tn_ + fp_ + 1e-9))
            boot_rows.append(row)
        except Exception:
            continue

    boot_df = pd.DataFrame(boot_rows)
    # AFTER (fixed):
    out = {}
    for m in metric_keys:
        arr = boot_df[m].dropna().values
        if len(arr) < 10:                       # ← guard: too few valid samples
            out[m] = {
                'mean':      float('nan'),
                'ci95_low':  float('nan'),
                'ci95_high': float('nan'),
            }
            print(f"\n  ⚠️  Bootstrap: metric '{m}' has only {len(arr)} valid samples — CI set to NaN")
        else:
            out[m] = {
                'mean':      float(np.mean(arr)),
                'ci95_low':  float(np.percentile(arr, 2.5)),
                'ci95_high': float(np.percentile(arr, 97.5)),
            }
    return out, boot_df


# ============================================================================
# PATIENT-LEVEL EVALUATION
# ============================================================================
def patient_level_metrics(y_true_img, img_probs, patient_ids, threshold):
    grouped_p = defaultdict(list)
    grouped_l = defaultdict(list)
    for l, prob, pid in zip(y_true_img, img_probs, patient_ids):
        grouped_p[pid].append(prob)
        grouped_l[pid].append(int(l))

    pat_labels, pat_probs = [], []
    for pid in sorted(grouped_p):
        pat_labels.append(grouped_l[pid][0])
        pat_probs.append(np.stack(grouped_p[pid]).mean(0))

    return compute_metrics(np.array(pat_labels), np.array(pat_probs), threshold)


# ============================================================================
# PUBLICATION FIGURES
# ============================================================================
def fig_roc(results, save_dir):
    fig, ax = plt.subplots(figsize=(7, 6))
    for key, res in results.items():
        fpr, tpr, _ = roc_curve(res['img_labels'], res['img_probs'][:, 1])
        ax.plot(fpr, tpr, lw=2, color=PALETTE[key],
                label=f"{MODEL_LABELS[key]}  (AUC={res['metrics']['auc_roc']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves — TEST')
    ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig1_roc_curves.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig1_roc_curves.png")


def fig_prc(results, save_dir, prevalence):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.axhline(prevalence, ls='--', color='gray', lw=1,
               label=f'No-skill (prevalence={prevalence:.3f})')
    for key, res in results.items():
        prec, rec, _ = precision_recall_curve(res['img_labels'], res['img_probs'][:, 1])
        ax.plot(rec, prec, lw=2, color=PALETTE[key],
                label=f"{MODEL_LABELS[key]}  (AP={res['metrics']['auc_pr']:.3f})")
    ax.set(xlabel='Recall', ylabel='Precision', title='Precision–Recall Curves — TEST')
    ax.legend(fontsize=9, loc='upper right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig2_pr_curves.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig2_pr_curves.png")


def fig_confusion(results, save_dir):
    import seaborn as sns
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1: axes = [axes]
    for ax, (key, res) in zip(axes, results.items()):
        m  = res['metrics']
        cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
        cmn = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
        sns.heatmap(cmn, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                    xticklabels=['Benign', 'Malignant'],
                    yticklabels=['Benign', 'Malignant'])
        ax.set(title=f"{MODEL_LABELS[key]}\nRecall={m['malignant_recall']:.3f}  Prec={m['malignant_precision']:.3f}",
               xlabel='Predicted', ylabel='True')
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig3_confusion_matrices.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig3_confusion_matrices.png")


def fig_calibration(results, save_dir):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    for key, res in results.items():
        try:
            pt, pp = calibration_curve(res['img_labels'], res['img_probs'][:, 1],
                                       n_bins=8, strategy='uniform')
            ax.plot(pp, pt, lw=2, marker='o', markersize=5, color=PALETTE[key],
                    label=f"{MODEL_LABELS[key]}  (Brier={res['metrics']['brier']:.3f})")
        except Exception:
            pass
    ax.set(xlabel='Mean Predicted Probability', ylabel='Fraction Positive (Malignant)',
           title='Calibration Curves — TEST')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig4_calibration_curves.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig4_calibration_curves.png")


def fig_dca(results, save_dir):
    pts = np.linspace(0.05, 0.50, 46)
    fig, ax = plt.subplots(figsize=(8, 6))
    ref = next(iter(results.values()))
    y, N = ref['img_labels'], len(ref['img_labels'])
    ax.plot(pts, [(y.sum()/N) - ((N - y.sum())/N) * (t/(1-t)) for t in pts],
            lw=2, ls='--', color='gray', label='Treat all')
    ax.plot(pts, [0.0] * len(pts), lw=2, ls=':', color='black', label='Treat none')
    for key, res in results.items():
        pm = res['img_probs'][:, 1]
        nb = []
        for t in pts:
            yh = (pm >= t).astype(int)
            tn_, fp_, fn_, tp_ = confusion_matrix(y, yh, labels=[0, 1]).ravel()
            nb.append((tp_/N) - (fp_/N) * t / (1 - t + 1e-9))
        ax.plot(pts, nb, lw=2, color=PALETTE[key], label=MODEL_LABELS[key])
    ax.set(xlabel='Threshold Probability', ylabel='Net Benefit',
           title='Decision Curve Analysis — TEST', xlim=(0.05, 0.50))
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig5_decision_curve.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig5_decision_curve.png")


def fig_bootstrap_ci(ci_dict, save_dir):
    metrics_to_plot = ['auc_roc', 'auc_pr', 'balanced_accuracy',
                       'malignant_recall', 'malignant_precision', 'mcc']
    labels_plot = {
        'auc_roc': 'AUC-ROC', 'auc_pr': 'AUC-PR',
        'balanced_accuracy': 'Balanced Acc',
        'malignant_recall': 'Malignant Recall',
        'malignant_precision': 'Malignant Precision', 'mcc': 'MCC',
    }
    n = len(metrics_to_plot)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 5))
    fig.suptitle('Bootstrap 95% CI (Patient-Resampled) — TEST', fontsize=12, fontweight='bold')
    for ax, metric in zip(axes, metrics_to_plot):
        ys, means, lows, highs, colors = [], [], [], [], []
        for i, (key, ci) in enumerate(ci_dict.items()):
            if metric not in ci: continue
            ys.append(i)
            means.append(ci[metric]['mean'])
            lows.append(ci[metric]['mean'] - ci[metric]['ci95_low'])
            highs.append(ci[metric]['ci95_high'] - ci[metric]['mean'])
            colors.append(PALETTE[key])
        ax.barh(ys, means, xerr=[lows, highs], color=colors, alpha=0.8,
                capsize=5, height=0.5, error_kw={'elinewidth': 1.5})
        ax.set_yticks(ys)
        ax.set_yticklabels([MODEL_LABELS.get(k, k) for k in ci_dict], fontsize=8)
        ax.set_xlabel(labels_plot[metric], fontsize=9)
        ax.set_title(labels_plot[metric], fontsize=9)
        ax.grid(axis='x', alpha=0.3); ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig6_bootstrap_ci.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig6_bootstrap_ci.png")


# ============================================================================
# FALLBACK SENSITIVITY ANALYSIS
# ============================================================================
def sensitivity_fallback(paired_df, img_labels, img_probs,
                          patient_ids, threshold, save_dir, model_type):
    fallback_imgs = set(paired_df.loc[paired_df['match_type'] == 'fallback', 'source_image'])
    if not fallback_imgs:
        print("  ℹ️  No fallback ROIs — sensitivity analysis skipped")
        return
    sources = sorted(set(paired_df['source_image']))
    mask_nofb = np.array([sources[i] not in fallback_imgs for i in range(len(img_labels))])
    if mask_nofb.sum() < 10:
        print("  ⚠️  Too few non-fallback images for sensitivity analysis")
        return
    rows = []
    m_all  = compute_metrics(img_labels, img_probs, threshold)
    m_nofb = compute_metrics(img_labels[mask_nofb], img_probs[mask_nofb], threshold)
    for metric in ['auc_roc', 'auc_pr', 'balanced_accuracy',
                   'malignant_recall', 'malignant_precision', 'mcc']:
        rows.append({'metric': metric,
                     'with_fallback':    round(m_all.get(metric, float('nan')), 4),
                     'without_fallback': round(m_nofb.get(metric, float('nan')), 4)})
    df = pd.DataFrame(rows)
    df.to_csv(Path(save_dir) / f"sensitivity_fallback_{model_type}.csv", index=False)
    print(f"\n  Fallback sensitivity ({model_type}) — {len(fallback_imgs)} fallback images")
    print(df.to_string(index=False))


# ============================================================================
# MAIN
# ============================================================================
def main():
    p("FUSION STEP 8: TEST EVALUATION + PUBLICATION FIGURES")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device: {device}")
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    # ── [1/7] Load test data ──────────────────────────────────────────────────
    print("\n[1/7] Loading TEST data (first & only time)...")
    paired_df, img_feat_test, lr_probs_test = load_test_data()
    source_imgs     = paired_df['source_image'].tolist()
    patient_ids_roi = paired_df['patient_id'].fillna(-1).astype(int).values

    print(f"  TEST ROIs        : {len(paired_df)}")
    print(f"  TEST images      : {paired_df['source_image'].nunique()}")
    print(f"  Malignant        : {(paired_df['fusion_label'] == 1).sum()}")
    print(f"  Benign           : {(paired_df['fusion_label'] == 0).sum()}")
    print(f"  lr_probs shape   : {lr_probs_test.shape}")   # expect (N,1)

    # image-level patient_ids (aligned to sorted source_images)
    grouped_pid  = defaultdict(list)
    for pid, src in zip(patient_ids_roi, source_imgs):
        grouped_pid[src].append(int(pid))
    sorted_srcs  = sorted(grouped_pid)
    img_pids_arr = np.array([grouped_pid[s][0] for s in sorted_srcs])

    # ── [2/7] Checkpoints + inference ────────────────────────────────────────
    print("\n[2/7] Loading checkpoints + running TEST inference...")
    results = {}

    for model_type in MODEL_TYPES:
        ckpt_path = Path(TRAINING_DIR) / model_type / "best_auc_pr.pth"
        if not ckpt_path.exists():
            print(f"  ⚠️  Checkpoint not found: {ckpt_path} — skipping")
            continue

        ckpt      = torch.load(ckpt_path, map_location=device)
        threshold = float(ckpt.get('threshold_val', 0.5))
        print(f"\n  {model_type}:")
        print(f"    Checkpoint epoch : {ckpt.get('epoch', '?')}")
        print(f"    Val AUC-PR       : {ckpt.get('best_auc_pr', '?'):.4f}")
        print(f"    VAL threshold    : {threshold:.3f}  ({ckpt.get('threshold_status', '?')})")

        model = get_fusion_model(model_type, verbose=False).to(device)
        model.load_state_dict(ckpt['model_state_dict'])

        roi_labels, roi_probs  = run_inference(model, paired_df, img_feat_test, lr_probs_test, device)
        img_labels, img_probs  = roi_to_image(roi_labels, roi_probs, source_imgs)
        metrics                = compute_metrics(img_labels, img_probs, threshold)

        results[model_type] = {
            'img_labels':  img_labels,
            'img_probs':   img_probs,
            'metrics':     metrics,
            'threshold':   threshold,
            'patient_ids': img_pids_arr,
        }
        print(f"    TEST AUC-PR      : {metrics['auc_pr']:.4f}")
        print(f"    TEST AUC-ROC     : {metrics['auc_roc']:.4f}")
        print(f"    TEST MalRecall   : {metrics['malignant_recall']:.4f}")
        print(f"    TEST BalAcc      : {metrics['balanced_accuracy']:.4f}")
        print(f"    TEST TP/FP/TN/FN : {metrics['tp']}/{metrics['fp']}/{metrics['tn']}/{metrics['fn']}")

    # ── [3/7] Clinical LR baseline ────────────────────────────────────────────
    print("\n[3/7] Loading Clinical LR baseline (Stage 2)...")

    audit_path = Path(LR_PROBS_DIR) / "lr_probs_test_image_audit.csv"

    if not audit_path.exists():
        print("  ⚠️  lr_probs_test_image_audit.csv not found — building from .npy + paired CSV")

        # ✅ FIX (LR-shape): Step 4 saves shape (N,1) with p_malignant in col 0.
        #    Handle all three possible storage formats gracefully.
        if lr_probs_test.ndim == 2 and lr_probs_test.shape[1] == 1:
            p_mal_roi = lr_probs_test[:, 0]      # (N,1) — Step 4 format  ← THE FIX
        elif lr_probs_test.ndim == 2 and lr_probs_test.shape[1] == 2:
            p_mal_roi = lr_probs_test[:, 1]      # (N,2) softmax format
        else:
            p_mal_roi = lr_probs_test.ravel()    # 1-D fallback

        img_p_mal, img_label, img_pid = {}, {}, {}
        for p_mal, src, lbl, pid in zip(
            p_mal_roi,
            paired_df['source_image'],
            paired_df['fusion_label'],
            paired_df['patient_id'].fillna(-1).astype(int),
        ):
            img_p_mal.setdefault(src, []).append(float(p_mal))
            img_label[src] = int(lbl)
            img_pid[src]   = int(pid)

        rows = []
        for src in sorted(img_p_mal):
            p_mean = float(np.mean(img_p_mal[src]))
            rows.append({
                'source_image': src,
                'p_malignant':  p_mean,
                'pred_label':   int(p_mean >= 0.5),  # placeholder; re-thresholded below
                'true_label':   img_label[src],
                'patient_id':   img_pid[src],
            })
        pd.DataFrame(rows).to_csv(audit_path, index=False)
        print(f"  ✅ Built audit CSV — {len(rows)} images → {audit_path}")

    lr_audit = pd.read_csv(audit_path)

    # Build (N,2) softmax array so compute_metrics works unchanged downstream
    lr_audit_sorted = lr_audit.sort_values('source_image').reset_index(drop=True)
    p_mal_img      = lr_audit_sorted['p_malignant'].values
    lr_img_probs   = np.stack([1 - p_mal_img, p_mal_img], axis=1)   # (N,2)
    lr_img_labels  = lr_audit_sorted['true_label'].values.astype(int)
    # ✅ FIX (LR-align): patient_ids from lr_audit_sorted, NOT from fusion list
    lr_img_pids    = lr_audit_sorted['patient_id'].fillna(-1).astype(int).values

    # Load VAL-optimised threshold if available
    lr_threshold   = 0.5
    lr_bundle_path = Path(LR_PROBS_DIR) / "lr_threshold_bundle.json"
    if lr_bundle_path.exists():
        with open(lr_bundle_path) as f:
            lr_threshold = float(json.load(f).get('threshold_val', 0.5))
        print(f"  LR VAL threshold loaded: {lr_threshold:.3f}")
    else:
        print(f"  ⚠️  lr_threshold_bundle.json not found — using threshold=0.5")

    lr_metrics = compute_metrics(lr_img_labels, lr_img_probs, lr_threshold)
    results['clinical_lr'] = {
        'img_labels':  lr_img_labels,
        'img_probs':   lr_img_probs,
        'metrics':     lr_metrics,
        'threshold':   lr_threshold,
        'patient_ids': lr_img_pids,
    }
    print(f"  LR TEST AUC-PR  : {lr_metrics['auc_pr']:.4f}")
    print(f"  LR TEST AUC-ROC : {lr_metrics['auc_roc']:.4f}")
    print(f"  LR TEST TP/FP/TN/FN: {lr_metrics['tp']}/{lr_metrics['fp']}/{lr_metrics['tn']}/{lr_metrics['fn']}")

    # ── [4/7] Metrics table ───────────────────────────────────────────────────
    print("\n[4/7] Building metrics tables...")
    metric_cols = [
        'auc_roc', 'auc_pr', 'brier', 'balanced_accuracy', 'macro_f1',
        'mcc', 'malignant_recall', 'malignant_precision', 'specificity',
        'benign_recall', 'benign_precision',
        'tp', 'fp', 'tn', 'fn',
        'pos_rate_pred', 'threshold', 'n_images', 'n_malignant', 'n_benign',
    ]
    table_rows = []
    for key, res in results.items():
        m = res['metrics']
        row = {'Model': MODEL_LABELS[key]}
        for c in metric_cols:
            v = m.get(c, float('nan'))
            row[c] = round(v, 4) if isinstance(v, float) else v
        table_rows.append(row)

    metrics_df = pd.DataFrame(table_rows)
    metrics_df.to_csv(Path(OUTPUT_DIR) / "metrics_table_test.csv", index=False)
    print("\n  COMPREHENSIVE TEST METRICS:")
    print(metrics_df[['Model', 'auc_roc', 'auc_pr', 'balanced_accuracy', 'mcc',
                       'malignant_recall', 'malignant_precision', 'specificity',
                       'tp', 'fp', 'tn', 'fn', 'threshold']].to_string(index=False))

    # ── [5/7] Bootstrap CI ────────────────────────────────────────────────────
    print(f"\n[5/7] Bootstrap 95% CI (N={N_BOOT} patient-resampled)...")
    ci_all = {}
    for key, res in results.items():
        print(f"  {key}...", end=' ', flush=True)
        ci, boot_df = bootstrap_ci(
            res['img_labels'], res['img_probs'],
            res['patient_ids'],           # ✅ always model-specific, row-aligned
            res['threshold'],
            n_boot=N_BOOT, seed=RANDOM_SEED,
        )
        ci_all[key] = ci
        boot_df.to_csv(Path(OUTPUT_DIR) / f"bootstrap_{key}.csv", index=False)
        print(f"AUC-PR {ci['auc_pr']['mean']:.4f} [{ci['auc_pr']['ci95_low']:.4f}–{ci['auc_pr']['ci95_high']:.4f}]")

    ci_rows = []
    for key, ci in ci_all.items():
        for metric, vals in ci.items():
            ci_rows.append({
                'Model': MODEL_LABELS[key], 'Metric': metric,
                'Mean': round(vals['mean'], 4),
                'CI95_Low': round(vals['ci95_low'], 4),
                'CI95_High': round(vals['ci95_high'], 4),
                'CI_Width': round(vals['ci95_high'] - vals['ci95_low'], 4),
            })
    pd.DataFrame(ci_rows).to_csv(Path(OUTPUT_DIR) / "bootstrap_ci_summary.csv", index=False)

    # ── [6/7] Ablation table ──────────────────────────────────────────────────
    print("\n[6/7] Ablation table (for paper)...")
    key_metrics = ['auc_roc', 'auc_pr', 'balanced_accuracy', 'mcc',
                   'malignant_recall', 'malignant_precision']
    ablation_rows = []
    for key in ['clinical_lr', 'image_only', 'clinical_only', 'fusion']:
        if key not in results: continue
        row = {'Model': MODEL_LABELS[key]}
        ci  = ci_all.get(key, {})
        m   = results[key]['metrics']
        for metric in key_metrics:
            val = m.get(metric, float('nan'))
            ci_v = ci.get(metric, {})
            lo, hi = ci_v.get('ci95_low', float('nan')), ci_v.get('ci95_high', float('nan'))
            row[metric] = (f"{val:.3f} [{lo:.3f}–{hi:.3f}]"
                           if not (np.isnan(lo) or np.isnan(hi)) else f"{val:.3f}")
        ablation_rows.append(row)
    ablation_df = pd.DataFrame(ablation_rows)
    ablation_df.to_csv(Path(OUTPUT_DIR) / "ablation_table_test.csv", index=False)
    print("\n  ABLATION TABLE (mean [95% CI]):")
    print(ablation_df.to_string(index=False))

    if 'fusion' in results:
        print("\n  PATIENT-LEVEL METRICS (fusion model):")
        pat_m = patient_level_metrics(
            results['fusion']['img_labels'],
            results['fusion']['img_probs'],
            results['fusion']['patient_ids'],
            results['fusion']['threshold'],
        )
        pat_df = pd.DataFrame([{
            'AUC-ROC': round(pat_m['auc_roc'], 4),
            'AUC-PR':  round(pat_m['auc_pr'], 4),
            'BalAcc':  round(pat_m['balanced_accuracy'], 4),
            'MCC':     round(pat_m['mcc'], 4),
            'MalRec':  round(pat_m['malignant_recall'], 4),
            'MalPrec': round(pat_m['malignant_precision'], 4),
            'n_patients': pat_m['n_images'],
        }])
        pat_df.to_csv(Path(OUTPUT_DIR) / "patient_level_metrics_fusion.csv", index=False)
        print(pat_df.to_string(index=False))

    # ── [7/7] Figures ─────────────────────────────────────────────────────────
    print("\n[7/7] Generating publication figures (300 dpi)...")
    prevalence = float(next(iter(results.values()))['img_labels'].mean())
    fig_roc(results, OUTPUT_DIR)
    fig_prc(results, OUTPUT_DIR, prevalence)
    fig_confusion(results, OUTPUT_DIR)
    fig_calibration(results, OUTPUT_DIR)
    fig_dca(results, OUTPUT_DIR)
    fig_bootstrap_ci(ci_all, OUTPUT_DIR)

    if 'fusion' in results:
        sensitivity_fallback(
            paired_df,
            results['fusion']['img_labels'],
            results['fusion']['img_probs'],
            results['fusion']['patient_ids'],
            results['fusion']['threshold'],
            OUTPUT_DIR, 'fusion',
        )

    p("STEP 8 COMPLETE — FINAL SUMMARY")
    print(f"\n  TEST SET RESULTS (image-level, threshold from VAL):\n  {'─'*60}")
    for key in ['clinical_lr', 'image_only', 'clinical_only', 'fusion']:
        if key not in results: continue
        m  = results[key]['metrics']
        ci = ci_all.get(key, {}).get('auc_pr', {})
        print(f"\n  {MODEL_LABELS[key]}:")
        print(f"    AUC-PR   : {m['auc_pr']:.4f}  [{ci.get('ci95_low', float('nan')):.4f}–{ci.get('ci95_high', float('nan')):.4f}]")
        print(f"    AUC-ROC  : {m['auc_roc']:.4f}")
        print(f"    BalAcc   : {m['balanced_accuracy']:.4f}")
        print(f"    MCC      : {m['mcc']:.4f}")
        print(f"    MalRecall: {m['malignant_recall']:.4f}  MalPrec: {m['malignant_precision']:.4f}")
        print(f"    TP={m['tp']} FP={m['fp']} TN={m['tn']} FN={m['fn']}  Thr={m['threshold']:.3f}")

    print(f"""
  Outputs: {OUTPUT_DIR}/
    metrics_table_test.csv, ablation_table_test.csv
    bootstrap_ci_summary.csv, patient_level_metrics_fusion.csv
    bootstrap_{{model}}.csv, sensitivity_fallback_fusion.csv
    fig1–fig6.png  ✅ DONE
""")


if __name__ == "__main__":
    main()


  FUSION STEP 8: TEST EVALUATION + PUBLICATION FIGURES

  Device: cuda

[1/7] Loading TEST data (first & only time)...
  TEST ROIs        : 283
  TEST images      : 241
  Malignant        : 41
  Benign           : 242
  lr_probs shape   : (283, 1)

[2/7] Loading checkpoints + running TEST inference...

  fusion:
    Checkpoint epoch : 20
    Val AUC-PR       : 0.9841
    VAL threshold    : 0.610  (OK)
    TEST AUC-PR      : 0.7571
    TEST AUC-ROC     : 0.9028
    TEST MalRecall   : 0.6341
    TEST BalAcc      : 0.7896
    TEST TP/FP/TN/FN : 26/11/189/15

  image_only:
    Checkpoint epoch : 17
    Val AUC-PR       : 0.9841
    VAL threshold    : 0.650  (OK)
    TEST AUC-PR      : 0.7377
    TEST AUC-ROC     : 0.9106
    TEST MalRecall   : 0.6098
    TEST BalAcc      : 0.7724
    TEST TP/FP/TN/FN : 25/13/187/16

  clinical_only:
    Checkpoint epoch : 33
    Val AUC-PR       : 0.5658
    VAL threshold    : 0.630  (OK)
    TEST AUC-PR      : 0.2574
    TEST AUC-ROC     : 0.5809
    TES

In [14]:
"""
FUSION STEP 9: INTERPRETABILITY & FEATURE IMPORTANCE
======================================================
Analyses driven by the Step 8 finding:
  FusionNet and Image-Only MLP produced IDENTICAL test decisions
  (same TP=34 FP=26 TN=174 FN=7) despite different AUC-PR scores.
  This step explains WHY and provides evidence for the paper.

Key questions answered:
  Q1. Group permutation importance: how much does each modality contribute?
  Q2. Clinical feature permutation: which of the 22 clinical features matter?
  Q3. SHAP values: clinical branch feature attribution on test samples
  Q4. t-SNE of learned embeddings: do representations separate classes?
  Q5. Error analysis: what's different about FP, FN, TP, TN cases?

Outputs (all 300 dpi): fusion_outputs/step9_interpretability/
  fig7_permutation_importance.png
  fig8_clinical_feature_importance.png
  fig9_shap_clinical.png
  fig10_tsne_embeddings.png
  fig11_error_analysis.png
  permutation_importance.csv
  clinical_feature_importance.csv
  shap_values_test.npy
  error_cases.csv

Q1 Safety:
  ✅ All analyses use test set for visualization only (no tuning)
  ✅ No new thresholds selected — uses VAL-optimized threshold from Step 8

BUG FIXES (this revision):
  [Fix 4c] roi_to_image: element-wise pa.max(0) replaced with
            pa[argmax(pa[:,1])] — same fix applied in Steps 7/8.

  [Fix 8b] auc_pr_score() fallback returned 0.0 on failure; changed to
            float('nan'). A degenerate permutation iteration (all-same
            predictions) would silently log 0.0 importance drop, masking
            the failure. NaN propagates visibly through the importance
            tables and is dropped by the CI calculation.

  [Fix THR] FUSION_THR was a hardcoded constant (0.210). If Step 7 is
            re-run with different data or hyperparameters, this constant
            becomes stale and error_analysis / fig_error_analysis use the
            wrong decision boundary. Fixed by loading FUSION_THR from the
            checkpoint at startup, falling back to 0.210 only if the
            checkpoint is unavailable.

  [Fix FEAT] CLINICAL_FEATURES was missing "foot" (22 features instead of
             23). The trained model's clinical_encoder expects input dim 24
             (23 clinical + 1 lr_prob = CLIN_INPUT_DIM). The missing feature
             caused a shape mismatch: (64x23) @ (24x64) → RuntimeError.
             Fixed by adding "foot" back to match Steps 6/7/8 exactly.
"""

import os, sys, json, warnings, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

import torch
import torch.nn.functional as F
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, f1_score,
)
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

try:
    _dir = str(Path(__file__).parent)
except NameError:
    _dir = os.getcwd()
sys.path.insert(0, _dir)

from step6_fusionnet import (
    FusionNet, get_fusion_model,
    IMG_FEAT_DIM, CLIN_FEAT_DIM, LR_PROB_DIM,
)

# ============================================================================
# CONFIG
# ============================================================================
PAIRED_DIR   = "fusion_outputs/step1_paired"
NORM_DIR     = "fusion_outputs/step3_norm/no_aug"
LR_PROBS_DIR = "fusion_outputs/step4_lr_probs"
CKPT_PATH    = "fusion_outputs/step7_training/fusion/best_auc_pr.pth"
OUTPUT_DIR   = "fusion_outputs/step9_interpretability"

# ✅ FIX (Fix FEAT): Added "foot" back — must be identical to Steps 6/7/8.
#    Steps 6/7/8 all include "foot" as the 9th bone location feature,
#    giving 23 clinical features total (CLIN_FEAT_DIM=23).
#    Step 9 was missing "foot", giving only 22 features → shape mismatch
#    when torch.cat([clin_feat, lr_prob], dim=1) produced (B,23) instead
#    of (B,24) expected by the trained clinical_encoder Linear(24→64).
CLINICAL_FEATURES = [
    "age", "gender",
    "hand", "ulna", "radius", "humerus",
    "femur", "tibia", "fibula", "hip bone",
    "ankle-joint", "knee-joint", "hip-joint",
    "wrist-joint", "elbow-joint", "shoulder-joint",
    "upper limb", "lower limb", "pelvis",
    "frontal", "lateral", "oblique", "foot"   # ← "foot" was missing here
]

# Sanity-check at import time so any future feature-list drift is caught
# immediately with a clear message rather than a cryptic shape error.
assert len(CLINICAL_FEATURES) == CLIN_FEAT_DIM, (
    f"CLINICAL_FEATURES length {len(CLINICAL_FEATURES)} != "
    f"CLIN_FEAT_DIM {CLIN_FEAT_DIM}. "
    "Step 9 feature list must exactly match Step 6."
)

BATCH_SIZE  = 64
N_PERM      = 50
RANDOM_SEED = 42

# ✅ FIX (Fix THR): FUSION_THR is loaded from the checkpoint below.
#    The fallback value is only used if the checkpoint is unavailable.
_FUSION_THR_FALLBACK = 0.210

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


# ============================================================================
# HELPERS
# ============================================================================
def p(title):
    print(f"\n{'='*68}\n  {title}\n{'='*68}")


class _NpEnc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)


# ============================================================================
# DATA LOADING
# ============================================================================
def load_test():
    paired   = pd.read_csv(Path(PAIRED_DIR) / "fusion_paired_test.csv")
    img_feat = np.load(Path(NORM_DIR)       / "norm_features_test.npy")
    lr_probs = np.load(Path(LR_PROBS_DIR)   / "lr_probs_test.npy")
    return paired, img_feat, lr_probs


def roi_to_image(roi_labels, roi_probs, source_images, method='mean'):
    """
    Aggregate ROI-level probabilities to image level.

    BUG FIX (Fix 4c): original pa.max(0) was element-wise across ROIs
      and produced invalid probability vectors (don't sum to 1).
      Fixed to select the ROI with the highest malignant probability.
    """
    g_p, g_l = defaultdict(list), defaultdict(list)
    for l, p_, s in zip(roi_labels, roi_probs, source_images):
        g_p[s].append(p_)
        g_l[s].append(int(l))
    img_l, img_p = [], []
    for s in sorted(g_p):
        img_l.append(g_l[s][0])
        pa = np.stack(g_p[s])
        if method == 'mean':
            img_p.append(pa.mean(0))
        else:
            # ✅ FIX (Fix 4c): valid softmax row
            img_p.append(pa[int(np.argmax(pa[:, 1]))])
    return np.array(img_l), np.array(img_p)


# ============================================================================
# MODEL INFERENCE HELPERS
# ============================================================================
def batch_inference(model, img_t, clin_t, lr_t, device):
    model.eval()
    all_probs = []
    n = len(img_t)
    with torch.no_grad():
        for i in range(0, n, BATCH_SIZE):
            out = model(
                img_t[i:i+BATCH_SIZE].to(device),
                clin_t[i:i+BATCH_SIZE].to(device),
                lr_t[i:i+BATCH_SIZE].to(device),
            )
            all_probs.append(F.softmax(out, dim=1).cpu().numpy())
    return np.concatenate(all_probs)


def get_image_level_probs(model, paired_df, img_feat, lr_probs, device):
    img_t  = torch.tensor(img_feat.astype(np.float32))
    clin_t = torch.tensor(
        paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    )
    lr_t   = torch.tensor(lr_probs.astype(np.float32))
    roi_labels = paired_df['fusion_label'].values.astype(int)
    sources    = paired_df['source_image'].tolist()
    roi_probs_out = batch_inference(model, img_t, clin_t, lr_t, device)
    return roi_to_image(roi_labels, roi_probs_out, sources)


def auc_pr_score(y_true, img_probs):
    # ✅ FIX (Fix 8b): return float('nan') on failure, not 0.0.
    #    Permutation iterations where all predictions are identical would
    #    cause average_precision_score to raise ValueError; returning 0.0
    #    silently made that permutation look like a maximal importance drop.
    try:
        return float(average_precision_score(y_true, img_probs[:, 1]))
    except Exception:
        return float('nan')


# ============================================================================
# 1. GROUP PERMUTATION IMPORTANCE
# ============================================================================
def group_permutation_importance(model, paired_df, img_feat, lr_probs,
                                  device, n_perm=N_PERM):
    p("1/5 — GROUP PERMUTATION IMPORTANCE")

    img_t  = torch.tensor(img_feat.astype(np.float32))
    clin_t = torch.tensor(
        paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    )
    lr_t    = torch.tensor(lr_probs.astype(np.float32))
    labels  = paired_df['fusion_label'].values.astype(int)
    sources = paired_df['source_image'].tolist()

    roi_p  = batch_inference(model, img_t, clin_t, lr_t, device)
    il, ip = roi_to_image(labels, roi_p, sources)
    baseline = auc_pr_score(il, ip)
    if np.isnan(baseline):
        raise RuntimeError("Baseline AUC-PR is NaN — check test data integrity.")
    print(f"  Baseline AUC-PR: {baseline:.4f}")

    rng    = np.random.default_rng(RANDOM_SEED)
    groups = {
        'Image (768→128-dim)':        'image',
        'Clinical features (23-dim)': 'clinical',
        'LR probability (1-dim)':     'lr_prob',
        'Clinical + LR (24-dim)':     'clinical_lr',
    }

    rows = []
    for label, gtype in groups.items():
        drops = []
        for _ in range(n_perm):
            perm_idx = rng.permutation(len(img_t))
            if gtype == 'image':
                p_img, p_clin, p_lr = img_t[perm_idx], clin_t, lr_t
            elif gtype == 'clinical':
                p_img, p_clin, p_lr = img_t, clin_t[perm_idx], lr_t
            elif gtype == 'lr_prob':
                p_img, p_clin, p_lr = img_t, clin_t, lr_t[perm_idx]
            else:
                p_img, p_clin, p_lr = img_t, clin_t[perm_idx], lr_t[perm_idx]

            roi_p_perm = batch_inference(model, p_img, p_clin, p_lr, device)
            il_p, ip_p = roi_to_image(labels, roi_p_perm, sources)
            score = auc_pr_score(il_p, ip_p)
            if not np.isnan(score):
                drops.append(baseline - score)

        if not drops:
            print(f"  ⚠️  {label}: all permutation iterations returned NaN — skipping")
            continue

        mean_drop = float(np.mean(drops))
        std_drop  = float(np.std(drops))
        pct       = mean_drop / baseline * 100
        rows.append({
            'Group':      label,
            'gtype':      gtype,
            'drop_mean':  mean_drop,
            'drop_std':   std_drop,
            'pct_drop':   pct,
            'importance': mean_drop,
            'n_valid':    len(drops),
        })
        print(f"  {label:35s}: ΔAUC-PR = {mean_drop:+.4f} ± {std_drop:.4f}  "
              f"({pct:+.1f}%)")

    df = pd.DataFrame(rows)
    df.to_csv(Path(OUTPUT_DIR) / "permutation_importance.csv", index=False)
    return df, baseline


# ============================================================================
# 2. CLINICAL FEATURE PERMUTATION IMPORTANCE
# ============================================================================
def clinical_feature_importance(model, paired_df, img_feat, lr_probs,
                                  device, n_perm=N_PERM):
    p("2/5 — CLINICAL FEATURE PERMUTATION IMPORTANCE")

    img_t     = torch.tensor(img_feat.astype(np.float32))
    clin_t_np = paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    clin_t    = torch.tensor(clin_t_np)
    lr_t      = torch.tensor(lr_probs.astype(np.float32))
    labels    = paired_df['fusion_label'].values.astype(int)
    sources   = paired_df['source_image'].tolist()

    roi_p  = batch_inference(model, img_t, clin_t, lr_t, device)
    il, ip = roi_to_image(labels, roi_p, sources)
    baseline = auc_pr_score(il, ip)
    if np.isnan(baseline):
        raise RuntimeError("Baseline AUC-PR is NaN")

    rng  = np.random.default_rng(RANDOM_SEED)
    rows = []

    for feat_name in CLINICAL_FEATURES + ['lr_prob']:
        drops = []
        for _ in range(n_perm):
            perm_idx = rng.permutation(len(img_t))
            if feat_name == 'lr_prob':
                p_lr   = lr_t[perm_idx]
                p_clin = clin_t
            else:
                feat_idx      = CLINICAL_FEATURES.index(feat_name)
                p_clin_np     = clin_t_np.copy()
                p_clin_np[:, feat_idx] = clin_t_np[perm_idx, feat_idx]
                p_clin = torch.tensor(p_clin_np)
                p_lr   = lr_t

            roi_p_perm = batch_inference(model, img_t, p_clin, p_lr, device)
            il_p, ip_p = roi_to_image(labels, roi_p_perm, sources)
            score = auc_pr_score(il_p, ip_p)
            if not np.isnan(score):
                drops.append(baseline - score)

        if not drops:
            continue
        mean_drop = float(np.mean(drops))
        std_drop  = float(np.std(drops))
        rows.append({
            'Feature':   feat_name,
            'drop_mean': mean_drop,
            'drop_std':  std_drop,
            'pct_drop':  mean_drop / baseline * 100,
            'n_valid':   len(drops),
        })

    df = pd.DataFrame(rows).sort_values('drop_mean', ascending=False)
    df.to_csv(Path(OUTPUT_DIR) / "clinical_feature_importance.csv", index=False)
    print(f"  Top 10 clinical features by AUC-PR importance:")
    for _, r in df.head(10).iterrows():
        print(f"    {r['Feature']:20s}: ΔAUC-PR = {r['drop_mean']:+.5f} ± {r['drop_std']:.5f}")
    return df, baseline


# ============================================================================
# 3. SHAP — CLINICAL BRANCH
# ============================================================================
def shap_clinical_analysis(model, paired_df, img_feat, lr_probs, device):
    p("3/5 — SHAP CLINICAL BRANCH ANALYSIS")

    try:
        import shap
    except ImportError:
        print("  ⚠️  shap not installed. Skipping. Run: pip install shap")
        return None, None

    img_t   = torch.tensor(img_feat.astype(np.float32))
    clin_np = paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    lr_np   = lr_probs.astype(np.float32)
    labels  = paired_df['fusion_label'].values.astype(int)

    img_mean    = img_t.mean(0, keepdim=True)
    # clin_np has 23 cols, lr_np has 1 col → concat = 24 cols
    clin_lr_np  = np.concatenate([clin_np, lr_np], axis=1)
    feat_names  = CLINICAL_FEATURES + ['lr_prob']

    n_clin = len(CLINICAL_FEATURES)  # 23

    def predict_clin_only(X_clin_lr):
        X_t       = torch.tensor(X_clin_lr.astype(np.float32)).to(device)
        img_batch = img_mean.expand(len(X_t), -1).to(device)
        with torch.no_grad():
            # X_t[:, :n_clin] = 23 clinical cols, X_t[:, n_clin:] = 1 lr col
            logits = model(img_batch, X_t[:, :n_clin], X_t[:, n_clin:])
            probs  = F.softmax(logits, dim=1)
        return probs[:, 1].cpu().numpy()

    rng       = np.random.default_rng(RANDOM_SEED)
    mal_idx   = np.where(labels == 1)[0]
    ben_idx   = np.where(labels == 0)[0]
    n_mal_bg  = min(30, len(mal_idx))
    n_ben_bg  = min(70, len(ben_idx))
    bg_idx    = np.concatenate([
        rng.choice(mal_idx, n_mal_bg, replace=False),
        rng.choice(ben_idx, n_ben_bg, replace=False),
    ])
    background  = clin_lr_np[bg_idx]
    test_sample = clin_lr_np

    print(f"  Background samples : {len(background)}")
    print(f"  Test samples       : {len(test_sample)}")
    print(f"  Computing SHAP (KernelExplainer)...")

    explainer = shap.KernelExplainer(predict_clin_only, background)
    shap_vals = explainer.shap_values(test_sample, nsamples=200, silent=True)
    np.save(Path(OUTPUT_DIR) / "shap_values_test.npy", shap_vals)
    print(f"  ✅ SHAP values shape: {shap_vals.shape}")
    return shap_vals, feat_names


# ============================================================================
# 4. t-SNE OF LEARNED EMBEDDINGS
# ============================================================================
def tsne_embeddings(model, paired_df, img_feat, lr_probs, device, fusion_thr):
    p("4/5 — t-SNE OF LEARNED EMBEDDINGS")

    img_t  = torch.tensor(img_feat.astype(np.float32))
    clin_t = torch.tensor(
        paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    )
    lr_t   = torch.tensor(lr_probs.astype(np.float32))
    labels = paired_df['fusion_label'].values.astype(int)

    model.eval()
    all_img_emb, all_clin_emb, all_fused, all_probs = [], [], [], []

    with torch.no_grad():
        for i in range(0, len(img_t), BATCH_SIZE):
            ib = img_t[i:i+BATCH_SIZE].to(device)
            cb = clin_t[i:i+BATCH_SIZE].to(device)
            lb = lr_t[i:i+BATCH_SIZE].to(device)
            ie, ce, fe = model.get_embeddings(ib, cb, lb)
            logits = model.fusion_head(fe)
            probs  = F.softmax(logits, dim=1)
            all_img_emb.append(ie.cpu().numpy())
            all_clin_emb.append(ce.cpu().numpy())
            all_fused.append(fe.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    img_emb  = np.concatenate(all_img_emb)
    clin_emb = np.concatenate(all_clin_emb)
    fused    = np.concatenate(all_fused)
    probs_np = np.concatenate(all_probs)
    preds    = (probs_np[:, 1] >= fusion_thr).astype(int)

    print(f"  ROIs: {len(labels)}  Malignant: {labels.sum()}  Benign: {(1-labels).sum()}")
    print(f"  Running t-SNE on 3 embedding spaces...")

    tsne = TSNE(n_components=2, perplexity=30, n_iter=1000,
                random_state=RANDOM_SEED, init='pca')

    results = {}
    for name, emb in [
        ('Image (128-dim)', img_emb),
        ('Clinical (32-dim)', clin_emb),
        ('Fused (160-dim)', fused),
    ]:
        z = tsne.fit_transform(emb)
        results[name] = z
        mal = z[labels == 1]; ben = z[labels == 0]
        between = np.linalg.norm(mal.mean(0) - ben.mean(0))
        within  = (np.std(mal) + np.std(ben)) / 2
        print(f"  {name:25s}: between/within = {between/within:.3f}")

    return results, labels, preds, probs_np


# ============================================================================
# 5. ERROR ANALYSIS
# ============================================================================
def error_analysis(model, paired_df, img_feat, lr_probs, device, fusion_thr):
    p("5/5 — ERROR ANALYSIS (FP / FN patterns)")

    img_t  = torch.tensor(img_feat.astype(np.float32))
    clin_t = torch.tensor(
        paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    )
    lr_t    = torch.tensor(lr_probs.astype(np.float32))
    labels  = paired_df['fusion_label'].values.astype(int)
    sources = paired_df['source_image'].tolist()

    roi_probs_out          = batch_inference(model, img_t, clin_t, lr_t, device)
    img_labels, img_probs  = roi_to_image(labels, roi_probs_out, sources)
    p_mal  = img_probs[:, 1]
    preds  = (p_mal >= fusion_thr).astype(int)

    clin_np  = paired_df[CLINICAL_FEATURES].values.astype(np.float32)
    lr_np    = lr_probs.astype(np.float32)
    src_list = sorted(set(sources))
    src_to_idx = {s: i for i, s in enumerate(src_list)}

    img_clin = np.zeros((len(src_list), len(CLINICAL_FEATURES)))
    img_lr   = np.zeros((len(src_list), 1))
    for i, src in enumerate(sources):
        idx = src_to_idx[src]
        img_clin[idx] = clin_np[i]
        img_lr[idx]   = lr_np[i]

    cats = []
    for yt, yh in zip(img_labels, preds):
        if yt == 1 and yh == 1:   cats.append('TP')
        elif yt == 0 and yh == 0: cats.append('TN')
        elif yt == 1 and yh == 0: cats.append('FN')
        else:                     cats.append('FP')
    cats = np.array(cats)

    print(f"\n  TEST IMAGE-LEVEL BREAKDOWN:")
    for c in ['TP', 'FP', 'TN', 'FN']:
        print(f"    {c}: {(cats == c).sum()}")

    rows = []
    for group in ['TP', 'FP', 'TN', 'FN']:
        mask = cats == group
        if mask.sum() == 0: continue
        row = {'Group': group, 'n': int(mask.sum())}
        row['p_malignant_mean'] = float(p_mal[mask].mean())
        row['p_malignant_std']  = float(p_mal[mask].std())
        row['age_mean']         = float(img_clin[mask, 0].mean())
        row['lr_prob_mean']     = float(img_lr[mask, 0].mean())
        row['upper_limb_rate']  = float(img_clin[mask, CLINICAL_FEATURES.index('upper limb')].mean())
        row['lower_limb_rate']  = float(img_clin[mask, CLINICAL_FEATURES.index('lower limb')].mean())
        row['frontal_rate']     = float(img_clin[mask, CLINICAL_FEATURES.index('frontal')].mean())
        rows.append(row)

    df = pd.DataFrame(rows)
    print("\n  Error group feature statistics:")
    print(df.to_string(index=False))
    df.to_csv(Path(OUTPUT_DIR) / "error_cases.csv", index=False)
    return df, cats, p_mal, img_labels, preds


# ============================================================================
# FIGURES
# ============================================================================
def fig_permutation(perm_df, baseline, save_dir):
    main = perm_df[perm_df['gtype'].isin(['image', 'clinical', 'lr_prob'])].copy()
    main = main.sort_values('drop_mean', ascending=True)
    colors = ['#ff7f0e', '#2ca02c', '#9467bd']
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(main['Group'], main['drop_mean'], xerr=main['drop_std'],
                   color=colors[:len(main)], alpha=0.85, capsize=5, height=0.5)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('AUC-PR Drop (↑ = more important)', fontsize=11)
    ax.set_title(f'Modality Contribution — FusionNet (baseline AUC-PR={baseline:.3f})',
                 fontsize=11, fontweight='bold')
    for bar, (_, row) in zip(bars, main.iterrows()):
        ax.text(max(row['drop_mean'] + row['drop_std'] + 0.001, 0.002),
                bar.get_y() + bar.get_height()/2,
                f"{row['pct_drop']:+.1f}%", va='center', ha='left', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig7_permutation_importance.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig7_permutation_importance.png")


def fig_clinical_importance(feat_df, save_dir):
    df = feat_df.copy().sort_values('drop_mean', ascending=True)

    def color(feat):
        if feat in ['age', 'gender']:                       return '#1f77b4'
        if feat in ['frontal', 'lateral', 'oblique']:       return '#d62728'
        if feat in ['upper limb', 'lower limb', 'pelvis']:  return '#9467bd'
        if 'joint' in feat:                                 return '#8c564b'
        if feat == 'lr_prob':                               return '#e377c2'
        return '#17becf'

    colors = [color(f) for f in df['Feature']]
    fig, ax = plt.subplots(figsize=(8, 9))
    ax.barh(df['Feature'], df['drop_mean'], xerr=df['drop_std'],
            color=colors, alpha=0.85, capsize=4, height=0.6)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('AUC-PR Drop after Permutation', fontsize=11)
    ax.set_title('Clinical Feature Importance — FusionNet\n'
                 '(permuting each feature independently)', fontsize=11, fontweight='bold')
    legend_els = [
        Line2D([0], [0], color='#1f77b4', lw=6, label='Demographics'),
        Line2D([0], [0], color='#17becf', lw=6, label='Individual bones'),
        Line2D([0], [0], color='#8c564b', lw=6, label='Joints'),
        Line2D([0], [0], color='#9467bd', lw=6, label='Body regions'),
        Line2D([0], [0], color='#d62728', lw=6, label='X-ray view'),
        Line2D([0], [0], color='#e377c2', lw=6, label='LR probability'),
    ]
    ax.legend(handles=legend_els, fontsize=8, loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig8_clinical_feature_importance.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig8_clinical_feature_importance.png")


def fig_shap(shap_vals, feat_names, test_labels, save_dir):
    try:
        import shap
    except ImportError:
        print("  ⚠️  shap not installed — skipping SHAP figure")
        return
    mean_abs = np.abs(shap_vals).mean(0)
    order    = np.argsort(mean_abs)[::-1][:15]
    fig, ax  = plt.subplots(figsize=(8, 7))
    shap.summary_plot(
        shap_vals[:, order],
        features=None,
        feature_names=[feat_names[i] for i in order],
        plot_type='bar', color='#1f77b4', show=False,
    )
    ax = plt.gca()
    ax.set_title('SHAP Feature Importance — Clinical Branch\n'
                 '(image features fixed to test-set mean)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig9_shap_clinical.png", dpi=300, bbox_inches='tight')
    plt.close(); print("  ✅ fig9_shap_clinical.png")


def fig_tsne(tsne_results, labels, preds, save_dir):
    embed_names = list(tsne_results.keys())
    n = len(embed_names)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 9))
    fig.suptitle('FusionNet t-SNE — Learned Embedding Spaces (TEST ROIs)',
                 fontsize=12, fontweight='bold')
    colors_true = {0: '#2196F3', 1: '#F44336'}
    colors_pred = {0: '#4CAF50', 1: '#FF9800'}
    label_names = {0: 'Benign', 1: 'Malignant'}
    for col, ename in enumerate(embed_names):
        z = tsne_results[ename]
        for lbl in [0, 1]:
            m = labels == lbl
            axes[0, col].scatter(z[m, 0], z[m, 1], c=colors_true[lbl],
                                 label=label_names[lbl], alpha=0.65, s=18, edgecolors='none')
        axes[0, col].set_title(f'{ename}\n(True label)', fontsize=9)
        axes[0, col].legend(fontsize=8, markerscale=1.5)
        axes[0, col].set_xticks([]); axes[0, col].set_yticks([])
        for pr in [0, 1]:
            m = preds == pr
            axes[1, col].scatter(z[m, 0], z[m, 1], c=colors_pred[pr],
                                 label=label_names[pr], alpha=0.65, s=18, edgecolors='none')
        axes[1, col].set_title(f'{ename}\n(Model prediction)', fontsize=9)
        axes[1, col].legend(fontsize=8, markerscale=1.5)
        axes[1, col].set_xticks([]); axes[1, col].set_yticks([])
    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig10_tsne_embeddings.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig10_tsne_embeddings.png")


def fig_error_analysis(error_df, cats, p_mal, true_labels, preds, fusion_thr, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('FusionNet Error Analysis — TEST Set (image-level)',
                 fontsize=12, fontweight='bold')
    cat_colors = {'TP': '#4CAF50', 'TN': '#2196F3', 'FP': '#FF9800', 'FN': '#F44336'}
    cat_order  = ['TP', 'TN', 'FP', 'FN']

    data_for_box, labels_box, colors_box = [], [], []
    for c in cat_order:
        mask = cats == c
        if mask.sum() > 0:
            data_for_box.append(p_mal[mask])
            labels_box.append(f"{c}\n(n={mask.sum()})")
            colors_box.append(cat_colors[c])

    bps = axes[0].boxplot(data_for_box, labels=labels_box, patch_artist=True)
    for patch, color in zip(bps['boxes'], colors_box):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    axes[0].axhline(fusion_thr, ls='--', color='gray', lw=1.2,
                    label=f'Threshold={fusion_thr:.3f}')
    axes[0].set_ylabel('p_malignant', fontsize=11)
    axes[0].set_title('p_malignant Distribution by Error Type', fontsize=10)
    axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)

    if len(error_df) > 0:
        cats_present = error_df['Group'].tolist()
        n_vals       = error_df['n'].tolist()
        lr_means     = error_df['lr_prob_mean'].tolist()
        x = np.arange(len(cats_present))
        ax2  = axes[1]
        bars = ax2.bar(x, n_vals, color=[cat_colors.get(c, '#999') for c in cats_present],
                       alpha=0.8, width=0.5, label='Count')
        ax2.set_xticks(x); ax2.set_xticklabels(cats_present, fontsize=11)
        ax2.set_ylabel('Count', fontsize=11)
        ax2.set_title('Count & LR Prob by Error Type', fontsize=10)
        for bar, n in zip(bars, n_vals):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     str(n), ha='center', va='bottom', fontsize=10)
        ax2b = ax2.twinx()
        ax2b.plot(x, lr_means, 'ko--', lw=1.5, markersize=7, label='Mean LR prob')
        ax2b.set_ylabel('Mean LR p_malignant', fontsize=10)
        ax2b.set_ylim(0, 0.5)
        lines,  lbls  = ax2.get_legend_handles_labels()
        lines2, lbls2 = ax2b.get_legend_handles_labels()
        ax2.legend(lines + lines2, lbls + lbls2, fontsize=8)
        ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(save_dir) / "fig11_error_analysis.png", dpi=300, bbox_inches='tight')
    plt.close(fig); print("  ✅ fig11_error_analysis.png")


# ============================================================================
# MAIN
# ============================================================================
def main():
    p("FUSION STEP 9: INTERPRETABILITY & FEATURE IMPORTANCE")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Device: {device}")

    print("\n  Loading test data...")
    paired_df, img_feat, lr_probs = load_test()
    print(f"  TEST ROIs  : {len(paired_df)}")
    print(f"  TEST images: {paired_df['source_image'].nunique()}")
    print(f"  Clinical features used: {len(CLINICAL_FEATURES)} "
          f"(CLIN_FEAT_DIM={CLIN_FEAT_DIM} ✅)")

    # ✅ FIX (Fix THR): Load FUSION_THR from the checkpoint, not from a
    #    hardcoded constant. This ensures the threshold is always consistent
    #    with the model that was actually trained in Step 7.
    print("\n  Loading FusionNet checkpoint...")
    ckpt = torch.load(CKPT_PATH, map_location=device)
    FUSION_THR = float(ckpt.get('threshold_val', _FUSION_THR_FALLBACK))
    if 'threshold_val' not in ckpt:
        print(f"  ⚠️  No 'threshold_val' in checkpoint — using fallback {_FUSION_THR_FALLBACK:.3f}")
    else:
        print(f"  Loaded threshold from checkpoint: {FUSION_THR:.3f}")

    model = get_fusion_model('fusion', verbose=True).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"  Loaded: epoch={ckpt['epoch']}  "
          f"val AUC-PR={ckpt['best_auc_pr']:.4f}  threshold={FUSION_THR:.3f}")

    # 1. Group permutation importance
    perm_df, baseline = group_permutation_importance(
        model, paired_df, img_feat, lr_probs, device
    )

    # 2. Clinical feature importance
    feat_df, _ = clinical_feature_importance(
        model, paired_df, img_feat, lr_probs, device
    )

    # 3. SHAP (clinical branch)
    shap_vals, feat_names = shap_clinical_analysis(
        model, paired_df, img_feat, lr_probs, device
    )

    # 4. t-SNE embeddings
    tsne_results, emb_labels, emb_preds, emb_probs = tsne_embeddings(
        model, paired_df, img_feat, lr_probs, device, FUSION_THR
    )

    # 5. Error analysis
    error_df, cats, p_mal, img_labels, img_preds = error_analysis(
        model, paired_df, img_feat, lr_probs, device, FUSION_THR
    )

    # ── Figures ──────────────────────────────────────────────────────────────
    p("GENERATING FIGURES")
    fig_permutation(perm_df, baseline, OUTPUT_DIR)
    fig_clinical_importance(feat_df, OUTPUT_DIR)
    if shap_vals is not None:
        fig_shap(shap_vals, feat_names, paired_df['fusion_label'].values, OUTPUT_DIR)
    fig_tsne(tsne_results, emb_labels, emb_preds, OUTPUT_DIR)
    fig_error_analysis(
        error_df, cats, p_mal, img_labels, img_preds, FUSION_THR, OUTPUT_DIR
    )

    p("STEP 9 COMPLETE — SUMMARY")
    print(f"""
  Key Findings:
  1. Modality importance  → permutation_importance.csv + fig7
  2. Clinical importance  → clinical_feature_importance.csv + fig8
  3. SHAP clinical branch → shap_values_test.npy + fig9
  4. t-SNE embeddings     → fig10
  5. Error analysis       → error_cases.csv + fig11

  Threshold used: {FUSION_THR:.3f} (loaded from checkpoint)

  Outputs: {OUTPUT_DIR}/
    permutation_importance.csv, clinical_feature_importance.csv
    shap_values_test.npy, error_cases.csv
    fig7–fig11.png

  ✅ Ready for Step 10: Statistical tests + paper compilation
""")


if __name__ == "__main__":
    main()


  FUSION STEP 9: INTERPRETABILITY & FEATURE IMPORTANCE

  Device: cuda

  Loading test data...
  TEST ROIs  : 283
  TEST images: 241
  Clinical features used: 23 (CLIN_FEAT_DIM=23 ✅)

  Loading FusionNet checkpoint...
  Loaded threshold from checkpoint: 0.610
  [fusion]  params=246,722  params/train_sample=162.32x
  Loaded: epoch=20  val AUC-PR=0.9841  threshold=0.610

  1/5 — GROUP PERMUTATION IMPORTANCE
  Baseline AUC-PR: 0.7571
  Image (768→128-dim)                : ΔAUC-PR = +0.5715 ± 0.0334  (+75.5%)
  Clinical features (23-dim)         : ΔAUC-PR = +0.0085 ± 0.0063  (+1.1%)
  LR probability (1-dim)             : ΔAUC-PR = -0.0007 ± 0.0005  (-0.1%)
  Clinical + LR (24-dim)             : ΔAUC-PR = +0.0075 ± 0.0058  (+1.0%)

  2/5 — CLINICAL FEATURE PERMUTATION IMPORTANCE
  Top 10 clinical features by AUC-PR importance:
    tibia               : ΔAUC-PR = +0.00273 ± 0.00372
    elbow-joint         : ΔAUC-PR = +0.00217 ± 0.00033
    humerus             : ΔAUC-PR = +0.00194 ± 0.00275


  STEP 10B: DEEP STATISTICAL ANALYSIS  [with_aug]

  Output directory: fusion_outputs/step10b_statistical/with_aug
  N_BOOTSTRAP: 2000  |  N_BINS_ECE: 10

[0] Loading Step 8 evaluation results...
  ⚠️  Not found: fusion_outputs/step8_evaluation/with_aug/roi_predictions_fusion.csv — skipping fusion
  ⚠️  Not found: fusion_outputs/step8_evaluation/with_aug/roi_predictions_image_only.csv — skipping image_only
  ⚠️  Not found: fusion_outputs/step8_evaluation/with_aug/roi_predictions_clinical_only.csv — skipping clinical_only


FileNotFoundError: No prediction files found in fusion_outputs/step8_evaluation/with_aug. Run Step 8 first and ensure roi_predictions.csv files are saved.

In [17]:
"""
FUSION STEP 10: STATISTICAL TESTS + PAPER-READY COMPILATION
=============================================================
Final step. Compiles all results into publication-ready outputs.

Statistical tests:
  1. DeLong test     — pairwise AUC-ROC comparisons (bootstrap-based)
  2. McNemar test    — pairwise prediction agreement (on test set)
  3. CI overlap check— whether bootstrap CIs overlap (from Step 8)

Paper outputs:
  1. LaTeX Table 2   — ablation table (image-level + patient-level)
  2. LaTeX Table 3   — statistical significance table
  3. Methods.md      — complete Methods section text
  4. Results.md      — complete Results section text
  5. report.md       — full study summary (all metrics, figures, findings)

All outputs: fusion_outputs/step10_paper/
  delong_tests.csv
  mcnemar_tests.csv
  table2_ablation.tex
  table3_statistics.tex
  Methods.md
  Results.md
  report.md

Q1 Safety:
  ✅ No model decisions made — reporting only
  ✅ All statistics computed from Step 8 test-set predictions
  ✅ All thresholds fixed from VAL (not re-tuned)

BUG FIXES (this revision):
  [Fix NaN-MCC]      ci_df may contain NaN entries for 'mcc' if Step 8 saw
                     degenerate bootstrap iterations. Previously these propagated
                     into the LaTeX tables as 'nan', which breaks LaTeX
                     compilation. Fixed by replacing NaN CI values with '--'
                     in fmt().

  [Fix boot-missing] run_delong_tests silently skipped pairs when a bootstrap
                     CSV was absent. Added an explicit check that at least 2
                     models have bootstrap data before attempting DeLong tests.

  [Fix mcnemar-guard] run_mcnemar_tests iterated over MODEL_ORDER but only
                     fetched rows where model label matched. Added explicit
                     assertion so label mismatches surface immediately.

  [Fix ConvNeXt]     All references to ResNet-18SE, ResNet18SE, PCA, 512-dim,
                     128-dim image features replaced with ConvNeXt-Tiny-SE,
                     768-dim, and the two-stage learned projection
                     (768→256→128) described in step6_fusionnet.py.
                     Removed all mentions of PCA (not used in this pipeline).

  [Fix metrics]      Hardcoded training summary table and test result table
                     in write_full_report() updated to match the actual Step 8
                     output (fusion: AUC-PR=0.7571 AUC-ROC=0.9028 Thr=0.610,
                     image_only: AUC-PR=0.7377 AUC-ROC=0.9106 Thr=0.650,
                     clinical_only: AUC-PR=0.2574 Thr=0.630,
                     clinical_lr: AUC-PR=0.1919 Thr=0.500,
                     TP/FP/TN/FN from Step 8 output).

  [Fix FUSED_DIM]    t-SNE and architecture descriptions updated:
                     IMG_EMBED_DIM=128, CLIN_EMBED_DIM=32, FUSED_DIM=160.
                     Previous code incorrectly stated 64+32=96.
"""

import os, sys, json, warnings
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.stats as stats

warnings.filterwarnings('ignore')

# ============================================================================
# PATHS
# ============================================================================
STEP8_DIR  = "fusion_outputs/step8_evaluation"
STEP7_DIR  = "fusion_outputs/step7_training"
OUTPUT_DIR = "fusion_outputs/step10_paper"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

MODEL_LABELS = {
    'fusion':        'FusionNet (proposed)',
    'image_only':    'Image-Only MLP',
    'clinical_only': 'Clinical-Only MLP',
    'clinical_lr':   'Clinical LR (Stage 2 baseline)',
}
MODEL_ORDER = ['clinical_lr', 'image_only', 'clinical_only', 'fusion']


def p(title):
    print(f"\n{'='*68}\n  {title}\n{'='*68}")


# ============================================================================
# LOAD STEP 8 RESULTS
# ============================================================================
def load_step8():
    metrics_df = pd.read_csv(Path(STEP8_DIR) / "metrics_table_test.csv")
    ci_df      = pd.read_csv(Path(STEP8_DIR) / "bootstrap_ci_summary.csv")

    boot = {}
    for mt in MODEL_ORDER:
        path = Path(STEP8_DIR) / f"bootstrap_{mt}.csv"
        if path.exists():
            boot[mt] = pd.read_csv(path)
        else:
            print(f"  ⚠️  Missing bootstrap file: {path.name}")

    lr_audit_path = Path(
        "fusion_outputs/step4_lr_probs/lr_probs_test_image_audit.csv"
    )
    lr_audit = (
        pd.read_csv(lr_audit_path) if lr_audit_path.exists() else pd.DataFrame()
    )

    # Build summary dict keyed by internal model key
    summary = {}
    for _, row in metrics_df.iterrows():
        for k, v in MODEL_LABELS.items():
            if v == row['Model']:
                summary[k] = row.to_dict()
                break

    return metrics_df, ci_df, boot, summary


# ============================================================================
# DELONG TEST (bootstrap-based)
# ============================================================================
def delong_bootstrap(boot_a, boot_b, metric='auc_roc', n_perm=10000, seed=42):
    """
    Bootstrap-based two-sided test for H0: metric(A) == metric(B).
    Returns: (diff, p_value, ci95_low, ci95_high)
    """
    rng = np.random.default_rng(seed)
    a   = boot_a[metric].dropna().values
    b   = boot_b[metric].dropna().values
    n   = min(len(a), len(b), n_perm)

    idx_a = rng.integers(0, len(a), n)
    idx_b = rng.integers(0, len(b), n)
    diffs = a[idx_a] - b[idx_b]

    diff_obs = float(np.mean(a) - np.mean(b))
    shifted  = diffs - np.mean(diffs)
    p_val    = float(np.mean(np.abs(shifted) >= np.abs(diff_obs)))
    p_val    = max(p_val, 1 / n)

    ci_low  = float(np.percentile(diffs, 2.5))
    ci_high = float(np.percentile(diffs, 97.5))
    return diff_obs, p_val, ci_low, ci_high


def run_delong_tests(boot, metrics=('auc_roc', 'auc_pr')):
    p("1/4 — DELONG / BOOTSTRAP SIGNIFICANCE TESTS")

    # ✅ FIX (Fix boot-missing): require at least 2 models with bootstrap data
    available = [k for k in MODEL_ORDER if k in boot]
    if len(available) < 2:
        print(
            f"  ⚠️  Only {len(available)} model(s) have bootstrap data; "
            f"need at least 2 for DeLong tests. Available: {available}"
        )
        return pd.DataFrame()

    rows = []
    pairs = [
        ('fusion',     'image_only',    "FusionNet vs Image-Only"),
        ('fusion',     'clinical_lr',   "FusionNet vs Clinical-LR"),
        ('fusion',     'clinical_only', "FusionNet vs Clinical-Only"),
        ('image_only', 'clinical_lr',   "Image-Only vs Clinical-LR"),
    ]
    for ma, mb, label in pairs:
        if ma not in boot or mb not in boot:
            print(f"  ⚠️  Missing bootstrap data for {label} — skipping")
            continue
        for metric in metrics:
            diff, p_val, ci_l, ci_h = delong_bootstrap(
                boot[ma], boot[mb], metric=metric
            )
            sig = (
                "***" if p_val < 0.001 else
                "**"  if p_val < 0.01  else
                "*"   if p_val < 0.05  else "ns"
            )
            rows.append({
                'Comparison':   label,
                'Metric':       metric,
                'Diff (A-B)':   round(diff, 4),
                'CI95_low':     round(ci_l, 4),
                'CI95_high':    round(ci_h, 4),
                'p_value':      round(p_val, 4),
                'Significance': sig,
            })
            print(
                f"  {label:40s} | {metric:8s} | "
                f"Δ={diff:+.4f} [{ci_l:+.4f},{ci_h:+.4f}] "
                f"p={p_val:.4f} {sig}"
            )

    df = pd.DataFrame(rows)
    df.to_csv(Path(OUTPUT_DIR) / "delong_tests.csv", index=False)
    return df


# ============================================================================
# McNEMAR TEST
# ============================================================================
def mcnemar_test(predictions_a, predictions_b, labels):
    a_correct = (predictions_a == labels)
    b_correct = (predictions_b == labels)
    b_count   = int(( a_correct & ~b_correct).sum())
    c_count   = int((~a_correct &  b_correct).sum())
    if b_count + c_count == 0:
        return 0.0, 1.0, b_count, c_count
    chi2  = (abs(b_count - c_count) - 1) ** 2 / (b_count + c_count)
    p_val = float(stats.chi2.sf(chi2, df=1))
    return float(chi2), p_val, b_count, c_count


def run_mcnemar_tests(boot, metrics_df):
    p("2/4 — McNEMAR TESTS (paired prediction agreement)")

    # ✅ FIX (Fix mcnemar-guard): build lookup with explicit label matching
    models_in_df = {}
    for _, row in metrics_df.iterrows():
        for k, v in MODEL_LABELS.items():
            if v.strip() == str(row['Model']).strip():
                models_in_df[k] = row
                break

    missing = [k for k in MODEL_ORDER if k not in models_in_df]
    if missing:
        print(
            f"  ⚠️  The following models are absent from "
            f"metrics_table_test.csv: {missing}\n"
            f"       Check that MODEL_LABELS values exactly match "
            f"the 'Model' column."
        )

    rows = []
    pairs = [
        ('fusion',   'image_only'),
        ('fusion',   'clinical_only'),
        ('fusion',   'clinical_lr'),
    ]
    for ma, mb in pairs:
        if ma not in models_in_df or mb not in models_in_df:
            print(f"  ⚠️  Skipping McNemar for {ma} vs {mb} — one or both missing")
            continue
        ra, rb = models_in_df[ma], models_in_df[mb]

        if (ra['tp'] == rb['tp'] and ra['fp'] == rb['fp'] and
                ra['tn'] == rb['tn'] and ra['fn'] == rb['fn']):
            note   = "IDENTICAL predictions — McNemar χ²=0, p=1.0 (trivially)"
            chi2, p_val, b_, c_ = 0.0, 1.0, 0, 0
        else:
            note   = "different predictions (exact McNemar requires raw predictions)"
            chi2, p_val, b_, c_ = float('nan'), float('nan'), -1, -1

        rows.append({
            'Comparison':    f"{MODEL_LABELS[ma]} vs {MODEL_LABELS[mb]}",
            'A_TP-FP-TN-FN': f"{ra['tp']}-{ra['fp']}-{ra['tn']}-{ra['fn']}",
            'B_TP-FP-TN-FN': f"{rb['tp']}-{rb['fp']}-{rb['tn']}-{rb['fn']}",
            'chi2':          round(chi2, 4),
            'p_value':       round(p_val, 4),
            'Note':          note,
        })
        print(f"\n  {MODEL_LABELS[ma]} vs {MODEL_LABELS[mb]}")
        print(f"    A: TP={ra['tp']} FP={ra['fp']} TN={ra['tn']} FN={ra['fn']}")
        print(f"    B: TP={rb['tp']} FP={rb['fp']} TN={rb['tn']} FN={rb['fn']}")
        print(f"    {note}")

    df = pd.DataFrame(rows)
    df.to_csv(Path(OUTPUT_DIR) / "mcnemar_tests.csv", index=False)
    return df


# ============================================================================
# LaTeX TABLE 2 — ABLATION TABLE
# ============================================================================
def latex_table2(ci_df, metrics_df, summary):
    p("3/4 — LaTeX TABLE 2 (Ablation Table)")

    ci_dict = {}
    for _, row in ci_df.iterrows():
        for k, v in MODEL_LABELS.items():
            if v == row['Model']:
                if k not in ci_dict:
                    ci_dict[k] = {}
                ci_dict[k][row['Metric']] = {
                    'mean': row['Mean'],
                    'lo':   row['CI95_Low'],
                    'hi':   row['CI95_High'],
                }
                break

    def fmt(k, m):
        d    = ci_dict.get(k, {}).get(m, {})
        if not d:
            return '--'
        mean = d.get('mean', float('nan'))
        lo   = d.get('lo',   float('nan'))
        hi   = d.get('hi',   float('nan'))
        # ✅ FIX (Fix NaN-MCC): NaN CI → '--' to prevent broken LaTeX output
        if np.isnan(mean):
            return '--'
        if np.isnan(lo) or np.isnan(hi):
            return f"{mean:.3f}"
        return f"{mean:.3f} [{lo:.3f}--{hi:.3f}]"

    lines = []
    lines.append(r"\begin{table*}[ht]")
    lines.append(r"\centering")
    lines.append(
        r"\caption{Ablation study results on the held-out test set (image-level,"
        r" $n=241$ images, 41 malignant, 200 benign). "
        r"Values are mean [95\% bootstrap CI] unless otherwise noted. "
        r"Thresholds were optimised on the validation set under the "
        r"clinical constraint (malignant recall $\geq 0.85$). "
        r"Image features are 768-dimensional ConvNeXt-Tiny-SE embeddings "
        r"projected to 128-dim via a learned two-stage MLP "
        r"(768$\to$256$\to$128). "
        r"FusionNet and Image-Only MLP produced identical binary decisions "
        r"(TP=26, FP=11, TN=189, FN=15); AUC-PR differs because the "
        r"clinical branch attenuates ranking separation without shifting "
        r"decisions at the chosen threshold.}"
    )
    lines.append(r"\label{tab:ablation}")
    lines.append(r"\resizebox{\textwidth}{!}{")
    lines.append(r"\begin{tabular}{lcccccccc}")
    lines.append(r"\hline")
    lines.append(
        r"\textbf{Model} & \textbf{AUC-ROC} & \textbf{AUC-PR} "
        r"& \textbf{Bal.~Acc.} & \textbf{MCC} "
        r"& \textbf{Sensitivity} & \textbf{Specificity} "
        r"& \textbf{TP/FP/TN/FN} & \textbf{Thr.} \\"
    )
    lines.append(r"\hline")

    name_map = {
        'clinical_lr':   r"Clinical LR \cite{stage2}",
        'image_only':    r"Image-Only MLP",
        'clinical_only': r"Clinical-Only MLP",
        'fusion':        r"\textbf{FusionNet (ours)}",
    }

    for k in MODEL_ORDER:
        if k not in summary:
            continue
        d        = summary[k]
        name     = name_map.get(k, k)
        tpfptnfn = (
            f"{int(d.get('tp',0))}/{int(d.get('fp',0))}/"
            f"{int(d.get('tn',0))}/{int(d.get('fn',0))}"
        )
        thr = f"{d.get('threshold', 0):.3f}"
        row = (
            f"  {name} & "
            f"{fmt(k,'auc_roc')} & "
            f"{fmt(k,'auc_pr')} & "
            f"{fmt(k,'balanced_accuracy')} & "
            f"{fmt(k,'mcc')} & "
            f"{fmt(k,'malignant_recall')} & "
            f"{fmt(k,'specificity')} & "
            f"{tpfptnfn} & "
            f"{thr} \\\\"
        )
        lines.append(row)

    lines.append(r"\hline")
    lines.append(
        r"\multicolumn{9}{l}{\textit{AUC: area under curve; "
        r"PR: precision--recall; Bal.~Acc.: balanced accuracy; "
        r"MCC: Matthews correlation coefficient.}} \\"
    )
    lines.append(
        r"\multicolumn{9}{l}{\textit{CI: 95\% bootstrap confidence "
        r"interval (patient-resampled, $N=2{,}000$).}} \\"
    )
    lines.append(r"\end{tabular}")
    lines.append(r"}")
    lines.append(r"\end{table*}")

    tex = "\n".join(lines)
    out = Path(OUTPUT_DIR) / "table2_ablation.tex"
    out.write_text(tex)
    print(f"  ✅ Saved: {out}")
    print("\n  Preview:")
    print(tex[:800])
    return tex


# ============================================================================
# LaTeX TABLE 3 — STATISTICAL SIGNIFICANCE
# ============================================================================
def latex_table3(delong_df):
    if len(delong_df) == 0:
        print("  ⚠️  DeLong results are empty — skipping Table 3")
        return ""

    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"\centering")
    lines.append(
        r"\caption{Pairwise AUC significance tests (bootstrap-based,"
        r" 10{,}000 permutations). $\Delta$ = model A $-$ model B. "
        r"ns: $p>0.05$; *: $p<0.05$; **: $p<0.01$; ***: $p<0.001$.}"
    )
    lines.append(r"\label{tab:stats}")
    lines.append(r"\begin{tabular}{llrrl}")
    lines.append(r"\hline")
    lines.append(
        r"\textbf{Comparison} & \textbf{Metric} & "
        r"$\Delta$ & \textbf{95\% CI} & \textbf{Sig.} \\"
    )
    lines.append(r"\hline")

    for _, row in delong_df.iterrows():
        comp   = row['Comparison'].replace('vs', r'\textit{vs}')
        metric = (
            row['Metric']
            .replace('_', r'\_')
            .upper()
            .replace(r'AUC\_', 'AUC-')
        )
        diff = f"{row['Diff (A-B)']:+.3f}"
        ci   = f"[{row['CI95_low']:+.3f}, {row['CI95_high']:+.3f}]"
        sig  = row['Significance']
        lines.append(f"  {comp} & {metric} & {diff} & {ci} & {sig} \\\\")

    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    tex = "\n".join(lines)
    out = Path(OUTPUT_DIR) / "table3_statistics.tex"
    out.write_text(tex)
    print(f"  ✅ Saved: {out}")
    return tex


# ============================================================================
# METHODS SECTION
# ============================================================================
def write_methods(summary):
    p("4/4 — GENERATING PAPER SECTIONS")

    methods = r"""# Methods

## 2.5 Multimodal Fusion Pipeline

### 2.5.1 Overview

We developed a multimodal fusion pipeline combining (i) appearance features
extracted by a ConvNeXt-Tiny with Squeeze-and-Excitation (SE) blocks from
detected bone tumour regions of interest (ROIs), (ii) structured clinical
metadata, and (iii) calibrated malignancy probabilities from a logistic
regression (LR) clinical model trained in Stage 2 of this study.

### 2.5.2 Feature Extraction

**Image features.** ConvNeXt-Tiny with Squeeze-and-Excitation (SE) blocks,
pre-trained on ImageNet, was applied to each detected ROI. The
768-dimensional feature vector from the ConvNeXt head (after global average
pooling and layer normalisation) was extracted and saved without dimensionality
reduction. Features were normalised per-dimension using StandardScaler
statistics fit on training ROIs only and applied unchanged to validation and
test sets (Step 3, no-augmentation variant).

**Clinical features.** A total of 23 structured clinical features were used,
covering patient demographics (age, gender), affected bone locations
(hand, ulna, radius, humerus, femur, tibia, fibula, hip bone, foot),
joint involvement (ankle-joint, knee-joint, hip-joint, wrist-joint,
elbow-joint, shoulder-joint), body region (upper limb, lower limb, pelvis),
and radiographic view (frontal, lateral, oblique).

**Clinical LR probability.** A calibrated malignancy probability scalar
$p_\text{malignant} \in [0, 1]$ was extracted from the Stage 2
GroupCalibratedEnsemble (logistic regression + Platt scaling). At the image
level, this produced a single probability per patient–image pair, which was
broadcast to all associated ROIs. The resulting scalar captures the clinical
prior for malignancy based on patient metadata alone.

### 2.5.3 FusionNet Architecture

We designed a dual-encoder intermediate fusion network (FusionNet). The
architecture consists of:

- **Image encoder** (768 → 256 → 128):
  - Stage 1: ConvNeXt-style projection block —
    Linear(768→256) → LayerNorm(256) → GELU → Dropout(0.40).
    LayerNorm and GELU match ConvNeXt's own head style.
  - Stage 2: standard dense block —
    Linear(256→128) → BatchNorm(128) → ReLU → Dropout(0.30).
  - Output: 128-dimensional image embedding.

- **Clinical encoder** (24 → 64 → 32):
  The 23 clinical features and 1 LR probability scalar are concatenated
  (24-dimensional input) and passed through two dense blocks
  (Linear–BatchNorm–ReLU–Dropout): 24→64→32 (dropout = 0.20).
  The LR probability is fed into this branch rather than treated as a
  third independent encoder, allowing the network to learn its relative
  contribution.
  - Output: 32-dimensional clinical embedding.

- **Fusion head** (160 → 64 → 32 → 2):
  The concatenated 160-dimensional embedding (128 image + 32 clinical)
  is passed through two dense blocks (160→64→32, dropout 0.40/0.30)
  followed by a linear classifier (32→2).
  - Output: 2-class logits.

All linear layers use Kaiming uniform initialisation. Bias terms are omitted
from layers followed by batch normalisation. LayerNorm layers use default
initialisation (weight=1, bias=0).

Two ablation variants sharing the same backbone were trained:
**Image-Only MLP** (same 768→256→128 image encoder + smaller head 128→32→2,
no clinical input) and **Clinical-Only MLP** (clinical encoder 24→64→32 +
head 32→16→2, no image input).

### 2.5.4 Training

All models were trained with the following configuration:

| Hyperparameter          | Value                                         |
|-------------------------|-----------------------------------------------|
| Loss function           | Focal Loss (α=0.65, γ=1.2)                   |
| Optimiser               | AdamW (lr=1×10⁻³, wd=1×10⁻³)                |
| LR scheduler            | Linear warmup (5 ep) + ReduceLROnPlateau      |
|                         | (patience=7, factor=0.5, min_lr=1×10⁻⁶)      |
| Gradient clipping       | max-norm = 1.0                                |
| Batch size              | 64                                            |
| Class imbalance         | WeightedRandomSampler (×2.0 malignant weight) |
|                         | + Focal Loss α=0.65                           |
| Early stopping          | patience=15, monitor=val AUC-PR, min_epochs=20|
| Label smoothing (train) | 0.05                                          |
| Random seed             | 42                                            |

### 2.5.5 Threshold Optimisation

Classification thresholds were optimised on the **validation set only**
using a composite score subject to the clinical constraint
malignant recall ≥ 0.85:

$$\text{score} = 1.00 \cdot \text{BalAcc} + 0.25 \cdot \text{MCC}
               + 0.10 \cdot \text{MacroF1}
               - 0.10 \cdot |\hat{r} - r|$$

where $r$ is the true malignant prevalence and $\hat{r}$ is the predicted
positive rate. Optimised thresholds were applied **unchanged** to the test set.

### 2.5.6 Evaluation

**Aggregation.** For images with multiple detected ROIs, softmax probabilities
were averaged across ROIs before applying the threshold (mean-pooling). The
final image-level probability is the mean of all ROI softmax vectors.
Patient-level predictions were computed by averaging image-level probabilities
across all images per patient.

**Metrics.** Primary metrics: AUC-PR and AUC-ROC. Secondary metrics: balanced
accuracy (BalAcc), Matthews correlation coefficient (MCC), malignant
sensitivity (recall), malignant precision, and specificity. All metrics were
computed at image level unless stated otherwise.

**Bootstrap confidence intervals.** 95% CIs were computed by patient-level
resampling ($N_\text{boot}=2{,}000$) to account for within-patient ROI
correlation. Statistical comparisons used bootstrap-based permutation tests
(10,000 permutations).

**Fallback sensitivity analysis.** A total of 19 test images (7.9%) had ROIs
detected only by the fallback low-confidence detector. A sensitivity analysis
compared metrics with and without these images.
"""

    # ✅ FIX (Fix metrics): all numbers come from the actual Step 8 output
    results = """# Results

## 3.3 Fusion Pipeline Evaluation

### 3.3.1 Ablation Study (Test Set, Image-Level)

Table 2 presents ablation results on the held-out test set
(n = 241 images, 41 malignant, 200 benign; 17.0% malignant prevalence).
All reported metrics use thresholds optimised on the validation set.

**FusionNet** (image + clinical + LR probability) achieved AUC-PR =
0.757 [95% CI: 0.618–0.853] and AUC-ROC = 0.903 [0.851–0.947],
with malignant sensitivity = 0.634, specificity = 0.945,
and MCC = 0.604 (TP=26, FP=11, TN=189, FN=15; threshold = 0.610).

The **Image-Only MLP** achieved AUC-PR = 0.738 [0.584–0.859] and
AUC-ROC = 0.911 [0.862–0.955], with TP=25, FP=13, TN=187, FN=16
(threshold = 0.650). Both image-based models substantially outperformed
the clinical-only baselines (Clinical LR: AUC-PR = 0.192, AUC-ROC = 0.551;
Clinical-Only MLP: AUC-PR = 0.257, AUC-ROC = 0.581).

FusionNet achieved marginally higher AUC-PR (+0.019) and lower FP count
(11 vs 13) than Image-Only MLP. The clinical branch contributes a small but
positive shift in precision at the chosen threshold, despite the LR
probability's compressed dynamic range
($p_\\text{malignant} \\in [0.07, 0.40]$, class effect size ≈ 0.88σ).

Bootstrap significance testing confirmed that differences between
image-based models (FusionNet, Image-Only) and clinical-only baselines
were statistically significant for both AUC-ROC and AUC-PR
(all p < 0.05; see Table 3). The difference between FusionNet and
Image-Only MLP was not statistically significant (p > 0.05), consistent
with their similar AUC profiles.

### 3.3.2 Patient-Level Results

Patient-level aggregation (mean-pooling across images per patient,
n = 99 patients, 18 malignant, 81 benign) improved FusionNet performance:
AUC-PR = 0.686, AUC-ROC = 0.895, balanced accuracy = 0.631,
MCC = 0.366, malignant sensitivity = 0.300, malignant precision = 0.667.
The lower sensitivity at patient level relative to image level reflects
the strict mean-pooling across multiple views per patient; individual
images where the model is uncertain dilute the aggregated probability.

### 3.3.3 Fallback ROI Sensitivity Analysis

Removing 19 test images (7.9%) whose ROIs were detected only by the
fallback detector modestly improved FusionNet performance:
AUC-PR: 0.757 → 0.780; malignant sensitivity: 0.634 → 0.667;
MCC: 0.604 → 0.632. This indicates that detection confidence measurably
affects downstream classification, motivating tighter confidence
filtering in clinical deployment.

### 3.3.4 Training Dynamics

FusionNet reached best validation AUC-PR at epoch 20 (early stopping
after 35 total epochs). Image-Only MLP peaked at epoch 17,
Clinical-Only MLP at epoch 33. The extended training required by
Clinical-Only MLP reflects the weaker and noisier signal in clinical
metadata alone. Both image-based models show substantial validation
AUC-PR drop from training to test (FusionNet: 0.984 → 0.757;
Image-Only: 0.984 → 0.738), consistent with the small training set
size (n = 1,520 ROIs) and deep ConvNeXt-Tiny-SE feature extraction.
"""

    (Path(OUTPUT_DIR) / "Methods.md").write_text(methods)
    (Path(OUTPUT_DIR) / "Results.md").write_text(results)
    print(f"  ✅ Methods.md")
    print(f"  ✅ Results.md")
    return methods, results


# ============================================================================
# FULL REPORT
# ============================================================================
def write_full_report(summary, ci_df, delong_df):
    ci_lut = {}
    for _, row in ci_df.iterrows():
        for k, v in MODEL_LABELS.items():
            if v == row['Model']:
                if k not in ci_lut:
                    ci_lut[k] = {}
                ci_lut[k][row['Metric']] = (
                    row['Mean'], row['CI95_Low'], row['CI95_High']
                )

    def ci(k, m):
        t = ci_lut.get(k, {}).get(m, None)
        if t is None:
            return 'N/A'
        if any(np.isnan(x) for x in t):
            return f"{t[0]:.4f} [N/A]" if not np.isnan(t[0]) else 'N/A'
        return f"{t[0]:.4f} [{t[1]:.4f}–{t[2]:.4f}]"

    delong_md = (
        delong_df.to_markdown(index=False) if len(delong_df) > 0 else 'N/A'
    )

    # ✅ FIX (Fix metrics): training summary uses actual Step 8 output values.
    #    Val AUC-PR, best epoch, thresholds and TP/FP/TN/FN from Step 8 log.
    report = f"""# Fusion Pipeline — Complete Results Report

**Architecture**: ConvNeXt-Tiny-SE image encoder (768-dim → 256 → 128)
**Test set**: 241 images | 41 malignant (17.0%) | 200 benign | 99 patients

---

## 1. Architecture Summary

| Component             | Details                                        |
|-----------------------|------------------------------------------------|
| Image backbone        | ConvNeXt-Tiny-SE (pretrained ImageNet)         |
| Image feature dim     | 768-dim (ConvNeXt head output)                 |
| Image encoder         | Linear(768→256) [LN+GELU] → Linear(256→128) [BN+ReLU] |
| Clinical input dim    | 23 features + 1 LR prob = 24-dim              |
| Clinical encoder      | Linear(24→64→32) [BN+ReLU]                    |
| Fusion dim            | 128 + 32 = 160-dim                             |
| Fusion head           | Linear(160→64→32→2) [BN+ReLU]                 |
| Total parameters      | ~17,600 (FusionNet)                            |

---

## 2. Training Summary (VAL — Q1 Safe)

| Model          | Best Val AUC-PR | Best Epoch | Val AUC-ROC | Threshold |
|----------------|-----------------|------------|-------------|-----------|
| FusionNet      | 0.9841          | 20         | —           | 0.610     |
| Image-Only MLP | 0.9841          | 17         | —           | 0.650     |
| Clinical-Only  | 0.5658          | 33         | —           | 0.630     |

---

## 3. Test Results — Image Level (primary)

| Model | AUC-ROC [95% CI] | AUC-PR [95% CI] | BalAcc [95% CI] | MCC [95% CI] | Sensitivity | Specificity | TP/FP/TN/FN | Thr |
|-------|------------------|-----------------|-----------------|--------------|-------------|-------------|-------------|-----|
| Clinical LR    | {ci('clinical_lr','auc_roc')} | {ci('clinical_lr','auc_pr')} | {ci('clinical_lr','balanced_accuracy')} | {ci('clinical_lr','mcc')} | {ci('clinical_lr','malignant_recall')} | {ci('clinical_lr','specificity')} | 0/0/200/41   | 0.500 |
| Image-Only MLP | {ci('image_only','auc_roc')} | {ci('image_only','auc_pr')} | {ci('image_only','balanced_accuracy')} | {ci('image_only','mcc')} | {ci('image_only','malignant_recall')} | {ci('image_only','specificity')} | 25/13/187/16 | 0.650 |
| Clinical-Only  | {ci('clinical_only','auc_roc')} | {ci('clinical_only','auc_pr')} | {ci('clinical_only','balanced_accuracy')} | {ci('clinical_only','mcc')} | {ci('clinical_only','malignant_recall')} | {ci('clinical_only','specificity')} | 20/91/109/21 | 0.630 |
| **FusionNet**  | {ci('fusion','auc_roc')} | {ci('fusion','auc_pr')} | {ci('fusion','balanced_accuracy')} | {ci('fusion','mcc')} | {ci('fusion','malignant_recall')} | {ci('fusion','specificity')} | **26/11/189/15** | 0.610 |

---

## 4. Test Results — Patient Level (FusionNet only)

| AUC-ROC | AUC-PR | BalAcc | MCC   | MalRec | MalPrec | n_patients |
|---------|--------|--------|-------|--------|---------|------------|
| 0.8949  | 0.6856 | 0.6310 | 0.366 | 0.300  | 0.6667  | 99         |

---

## 5. Statistical Tests

### DeLong Bootstrap Tests (AUC)

{delong_md}

### McNemar Test (Decision Agreement)
- FusionNet vs Image-Only: slightly different (26/11/189/15 vs 25/13/187/16)
  → exact McNemar requires raw per-image prediction vectors (see bootstrap CSVs)
- FusionNet vs Clinical-LR: substantially different (26 FP vs 0; 15 FN vs 41)

---

## 6. Key Findings

### Finding 1: Image features dominate
Both FusionNet and Image-Only MLP achieve AUC-ROC > 0.90, far above the
clinical-only baselines (≤ 0.58). The ConvNeXt-Tiny-SE 768-dim embeddings
carry the vast majority of the discriminative signal.

### Finding 2: Clinical branch has small positive effect on precision
FusionNet vs Image-Only: FP reduced from 13 to 11 (−15%), AUC-PR improved
by +0.019. The clinical branch shifts a small number of borderline benign
cases below the threshold, improving precision at no cost to sensitivity.

### Finding 3: Patient-level aggregation does not improve MalRec here
Image-level MalRec = 0.634 → Patient-level MalRec = 0.300. Mean-pooling
across uncertain multi-view images dilutes high-confidence single-view
signals. A max-pool patient aggregation strategy may be more appropriate.

### Finding 4: Fallback ROIs degrade performance measurably
19 fallback test images: AUC-PR 0.757→0.780, MalRec 0.634→0.667 without them.

### Finding 5: Val→Test generalisation gap
FusionNet: val AUC-PR=0.984 → test AUC-PR=0.757 (−23.1% relative).
Image-Only: val AUC-PR=0.984 → test AUC-PR=0.738 (−25.0% relative).
This large gap reflects the small, skewed test set (41 malignant cases)
and likely overfitting of the val threshold to the val distribution.

---

## 7. Fallback Sensitivity Analysis (FusionNet)

| Metric    | With fallback (n=241) | Without fallback (n=222) |
|-----------|-----------------------|--------------------------|
| AUC-ROC   | 0.9028                | 0.9064                   |
| AUC-PR    | 0.7571                | 0.7795                   |
| BalAcc    | 0.7896                | 0.8060                   |
| MalRec    | 0.6341                | 0.6667                   |
| MalPrec   | 0.7027                | 0.7222                   |
| MCC       | 0.6036                | 0.6319                   |

---

## 8. Limitations

1. **Dataset size**: 1,520 training ROIs from 1,209 images limits generalisation.
2. **Clinical signal compression**: Platt-calibrated LR probs in [0.07–0.40].
3. **Single-site data**: Generalisation to other centres is untested.
4. **Detection dependency**: 7.9% fallback test images measurably degrade performance.
5. **Val→Test gap**: Large (~23%) relative AUC-PR drop suggests val-set overfitting.
6. **Patient-level aggregation**: Mean-pool degrades sensitivity; max-pool warranted.

---

## 9. Recommendations

1. End-to-end fine-tuning of ConvNeXt-Tiny-SE jointly with FusionNet head.
2. Max-pool (or learned attention) patient-level aggregation.
3. Stronger clinical features or free-text NLP from radiology reports.
4. External validation cohort.
5. Confidence-weighted fusion using ROI detection confidence scores.

---

## 10. File Inventory

| Step    | Key output                      | Description                   |
|---------|---------------------------------|-------------------------------|
| Step 6  | step6_fusionnet.py              | FusionNet architecture        |
| Step 7  | best_auc_pr.pth per model       | Trained checkpoints           |
| Step 8  | metrics_table_test.csv          | Full test metrics              |
| Step 8  | ablation_table_test.csv         | Paper Table 2 draft           |
| Step 8  | bootstrap_ci_summary.csv        | 95% CI table                  |
| Step 8  | fig1–fig6.png                   | Publication figures            |
| Step 9  | permutation_importance.csv      | Modality importance            |
| Step 9  | fig7–fig11.png                  | Interpretability figures       |
| Step 10 | table2_ablation.tex             | LaTeX Table 2                  |
| Step 10 | table3_statistics.tex           | LaTeX Table 3                  |
| Step 10 | Methods.md                      | Methods section text           |
| Step 10 | Results.md                      | Results section text           |
| Step 10 | report.md                       | This document                  |
"""

    (Path(OUTPUT_DIR) / "report.md").write_text(report)
    print(f"  ✅ report.md")
    return report


# ============================================================================
# MAIN
# ============================================================================
def main():
    p("FUSION STEP 10: STATISTICAL TESTS + PAPER COMPILATION")

    print("\n  Loading Step 8 results...")
    metrics_df, ci_df, boot, summary = load_step8()
    print(f"  Models found   : {list(summary.keys())}")
    print(f"  CI rows        : {len(ci_df)}")
    print(f"  Bootstrap files: {list(boot.keys())}")

    # 1. DeLong tests
    delong_df = run_delong_tests(boot)

    # 2. McNemar tests
    mcnemar_df = run_mcnemar_tests(boot, metrics_df)

    # 3. LaTeX tables
    p("3/4 — LaTeX TABLES")
    latex_table2(ci_df, metrics_df, summary)
    if len(delong_df) > 0:
        latex_table3(delong_df)

    # 4. Methods + Results text + full report
    methods, results = write_methods(summary)
    write_full_report(summary, ci_df, delong_df)

    p("STEP 10 COMPLETE — FINAL PIPELINE SUMMARY")
    print(f"""
  ALL PIPELINE STEPS COMPLETE.

  ╔══════════════════════════════════════════════════════════════════╗
  ║  FINAL TEST RESULTS (image-level, threshold from VAL)            ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  Architecture : ConvNeXt-Tiny-SE  768-dim → 256 → 128            ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  FusionNet    : AUC-PR=0.7571  AUC-ROC=0.9028  MCC=0.6036       ║
  ║  Image-Only   : AUC-PR=0.7377  AUC-ROC=0.9106  MCC=0.5617       ║
  ║  Clinical-Only: AUC-PR=0.2574  AUC-ROC=0.5809  MCC=0.0247       ║
  ║  Clinical-LR  : AUC-PR=0.1919  AUC-ROC=0.5507  MCC=nan          ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  Patient-level (FusionNet): AUC-PR=0.6856  AUC-ROC=0.8949       ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  FusionNet best on: AUC-PR (+0.019 vs Image-Only), FP count      ║
  ║  Image-Only best on: AUC-ROC (+0.008 vs FusionNet)               ║
  ╚══════════════════════════════════════════════════════════════════╝

  Paper outputs: {OUTPUT_DIR}/
    table2_ablation.tex    ← copy into paper
    table3_statistics.tex  ← copy into paper
    Methods.md             ← Section 2.5
    Results.md             ← Section 3.3
    report.md              ← complete summary

  All figures: fusion_outputs/step8_evaluation/ (fig1–fig6)
               fusion_outputs/step9_interpretability/ (fig7–fig11)
""")


if __name__ == "__main__":
    main()


  FUSION STEP 10: STATISTICAL TESTS + PAPER COMPILATION

  Loading Step 8 results...
  Models found   : ['fusion', 'image_only', 'clinical_only', 'clinical_lr']
  CI rows        : 36
  Bootstrap files: ['clinical_lr', 'image_only', 'clinical_only', 'fusion']

  1/4 — DELONG / BOOTSTRAP SIGNIFICANCE TESTS
  FusionNet vs Image-Only                  | auc_roc  | Δ=-0.0083 [-0.0809,+0.0611] p=0.8120 ns
  FusionNet vs Image-Only                  | auc_pr   | Δ=+0.0175 [-0.1672,+0.2096] p=0.8405 ns
  FusionNet vs Clinical-LR                 | auc_roc  | Δ=+0.3496 [+0.2181,+0.4690] p=0.0005 ***
  FusionNet vs Clinical-LR                 | auc_pr   | Δ=+0.5549 [+0.3936,+0.6755] p=0.0005 ***
  FusionNet vs Clinical-Only               | auc_roc  | Δ=+0.3191 [+0.1822,+0.4465] p=0.0005 ***
  FusionNet vs Clinical-Only               | auc_pr   | Δ=+0.4869 [+0.3130,+0.6396] p=0.0005 ***
  Image-Only vs Clinical-LR                | auc_roc  | Δ=+0.3579 [+0.2287,+0.4777] p=0.0005 ***
  Image-Only vs 

In [19]:
"""
FUSION STEP 10B: DEEP STATISTICAL ANALYSIS
===========================================
Standalone script — run after Step 8 evaluation is complete.

Implements three analysis blocks that reviewers increasingly require
for medical AI papers:

  A. DeLong Test — pairwise AUC-ROC significance with exact confidence intervals
  B. Calibration Analysis — ECE, Brier score, reliability diagrams, MCE
  C. Error Analysis — cases corrected/worsened by fusion vs image-only baseline

All outputs saved to:
  fusion_outputs/step10b_statistical/

Dependencies: numpy, scipy, pandas, matplotlib, scikit-learn, torch
No extra installs required — all available on Kaggle.

References:
  DeLong ER et al. (1988) "Comparing the areas under two or more correlated
  receiver operating characteristic curves: a nonparametric approach."
  Biometrics 44(3):837-845.

  Naeini MP et al. (2015) "Obtaining Well Calibrated Probabilities Using
  Bayesian Binning into Quantiles." AAAI.

  Guo C et al. (2017) "On Calibration of Modern Neural Networks." ICML.

BUG FIXES (this revision):
  [Fix load_results] Step 8 never writes roi_predictions.csv — it only
    saves aggregated metrics/bootstrap CSVs. load_results() previously
    raised FileNotFoundError immediately. Fixed by running live inference
    directly from Step 7 checkpoints + Step 3 (no_aug) normalised features
    + Step 4 LR probs, identical to what Step 8 does internally. This
    makes Step 10B self-contained and independent of Step 8's file layout.

  [Fix ConvNeXt] All references to ResNet-18SE, PCA, 512-dim replaced
    with ConvNeXt-Tiny-SE, 768-dim, and learned 768→256→128 projection.

  [Fix AUG_CONDITION] Removed aug/no_aug branching — Step 7 trains on
    no_aug features only (NORM_DIR = step3_norm/no_aug hardcoded in Step 7).
    Step 10B always uses no_aug. AUG_CONDITION constant removed; paths
    simplified accordingly.

  [Fix threshold loading] Step 10B no longer looks for per-model
    threshold.json files (never created). Thresholds are loaded directly
    from Step 7 checkpoint 'threshold_val' key, same as Step 8.

  [Fix CLINICAL_FEATURES] Added missing "foot" feature to match Step 6/7/8
    (23 features). Previous list had only 22, causing shape mismatch when
    inference runs through the FusionNet clinical encoder.

  [Fix FUSED_DIM labels] Architecture descriptions updated:
    IMG_EMBED_DIM=128, CLIN_EMBED_DIM=32, FUSED_DIM=160 (not 96).
"""

import os
import sys
import json
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

from pathlib import Path
from collections import defaultdict
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, roc_curve,
    confusion_matrix, precision_recall_curve,
    average_precision_score,
)
from sklearn.calibration import calibration_curve

import torch
import torch.nn.functional as F

warnings.filterwarnings('ignore')

# ── sys.path so step6_fusionnet imports work ─────────────────────────────────
try:
    _this_dir = str(Path(__file__).parent)
except NameError:
    _this_dir = os.getcwd()
sys.path.insert(0, _this_dir)

from step6_fusionnet import (
    get_fusion_model,
    IMG_FEAT_DIM, CLIN_FEAT_DIM, LR_PROB_DIM,
)

# =============================================================================
# CONFIGURATION
# =============================================================================
PAIRED_DIR   = "fusion_outputs/step1_paired"
NORM_DIR     = "fusion_outputs/step3_norm/no_aug"   # always no_aug (Step 7 trains on no_aug)
LR_PROBS_DIR = "fusion_outputs/step4_lr_probs"
CKPT_DIR     = "fusion_outputs/step7_training"
OUTPUT_DIR   = "fusion_outputs/step10b_statistical"

# ✅ FIX (Fix CLINICAL_FEATURES): 23 features — must match Step 6/7/8 exactly.
#    Previous version had 22 (missing "foot"), causing (B,22) vs (B,24)
#    shape mismatch in clinical encoder Linear(24→64).
CLINICAL_FEATURES = [
    "age", "gender",
    "hand", "ulna", "radius", "humerus",
    "femur", "tibia", "fibula", "hip bone",
    "ankle-joint", "knee-joint", "hip-joint",
    "wrist-joint", "elbow-joint", "shoulder-joint",
    "upper limb", "lower limb", "pelvis",
    "frontal", "lateral", "oblique", "foot",   # ← "foot" must be present
]
assert len(CLINICAL_FEATURES) == CLIN_FEAT_DIM, (
    f"CLINICAL_FEATURES length {len(CLINICAL_FEATURES)} != CLIN_FEAT_DIM {CLIN_FEAT_DIM}"
)

BATCH_SIZE   = 64
N_BOOTSTRAP  = 2000
RANDOM_SEED  = 42
N_BINS_ECE   = 10
N_BINS_PLOT  = 10
ALPHA        = 0.05

MODEL_TYPES = ['fusion', 'image_only', 'clinical_only']
DISPLAY_NAMES = {
    'fusion':        'FusionNet',
    'image_only':    'Image-Only MLP',
    'clinical_only': 'Clinical-Only MLP',
    'lr_baseline':   'Clinical LR Baseline',
}
COLORS = {
    'fusion':        '#2166ac',
    'image_only':    '#d6604d',
    'clinical_only': '#4dac26',
    'lr_baseline':   '#969696',
}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)


# =============================================================================
# SECTION 0: LOAD / INFER RESULTS
# =============================================================================

def _run_inference_for_model(model_type: str, device: torch.device,
                              paired_df, img_feat, lr_probs) -> pd.DataFrame:
    """
    Load Step 7 checkpoint for model_type, run inference on the test set,
    return a DataFrame with columns:
      [source_image, patient_id, y_true, p_malignant, threshold]
    at ROI level (before image aggregation).
    """
    ckpt_path = Path(CKPT_DIR) / model_type / "best_auc_pr.pth"
    if not ckpt_path.exists():
        print(f"  ⚠️  Checkpoint not found: {ckpt_path} — skipping {model_type}")
        return pd.DataFrame()

    ckpt = torch.load(ckpt_path, map_location=device)
    threshold = float(ckpt.get('threshold_val', 0.5))

    # Integrity check — catch wrong checkpoint silently loaded
    model = get_fusion_model(model_type, verbose=False).to(device)
    ckpt_keys  = set(ckpt['model_state_dict'].keys())
    model_keys = set(model.state_dict().keys())
    if ckpt_keys != model_keys:
        print(f"  ❌ Checkpoint key mismatch for {model_type} — skipping")
        return pd.DataFrame()
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    img_t  = torch.tensor(img_feat.astype(np.float32))
    clin_t = torch.tensor(paired_df[CLINICAL_FEATURES].values.astype(np.float32))
    lr_t   = torch.tensor(lr_probs.astype(np.float32))

    all_probs = []
    n = len(img_t)
    with torch.no_grad():
        for i in range(0, n, BATCH_SIZE):
            out = model(
                img_t[i:i+BATCH_SIZE].to(device),
                clin_t[i:i+BATCH_SIZE].to(device),
                lr_t[i:i+BATCH_SIZE].to(device),
            )
            all_probs.append(F.softmax(out, dim=1).cpu().numpy())

    probs = np.concatenate(all_probs)   # (N, 2)

    df = pd.DataFrame({
        'source_image': paired_df['source_image'].values,
        'patient_id':   paired_df['patient_id'].fillna(-1).astype(int).values,
        'y_true':       paired_df['fusion_label'].values.astype(int),
        'p_malignant':  probs[:, 1],
        'threshold':    threshold,
    })
    print(f"  ✅ {model_type:<15}  ROIs={len(df)}"
          f"  malignant={df['y_true'].sum()}"
          f"  threshold={threshold:.3f}"
          f"  val AUC-PR={ckpt.get('best_auc_pr', float('nan')):.4f}")
    return df


def load_results() -> dict:
    """
    ✅ FIX (Fix load_results): Step 8 does NOT write roi_predictions.csv.
    This function re-runs inference from Step 7 checkpoints directly,
    producing equivalent per-ROI prediction DataFrames for all model types.
    Also loads Clinical LR baseline from Step 4 audit CSV.
    """
    print("\n[0] Running inference from Step 7 checkpoints (no_aug features)...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device: {device}")

    paired_df = pd.read_csv(Path(PAIRED_DIR) / "fusion_paired_test.csv")
    img_feat  = np.load(Path(NORM_DIR)       / "norm_features_test.npy")
    lr_probs  = np.load(Path(LR_PROBS_DIR)   / "lr_probs_test.npy")

    print(f"  TEST ROIs   : {len(paired_df)}")
    print(f"  TEST images : {paired_df['source_image'].nunique()}")
    print(f"  lr_probs shape: {lr_probs.shape}")

    results = {}
    for mt in MODEL_TYPES:
        df = _run_inference_for_model(mt, device, paired_df, img_feat, lr_probs)
        if len(df) > 0:
            results[mt] = df

    # Clinical LR baseline — load from Step 4 audit CSV (image-level)
    lr_audit_path = Path(LR_PROBS_DIR) / "lr_probs_test_image_audit.csv"
    if lr_audit_path.exists():
        lr_audit = pd.read_csv(lr_audit_path)
        # Normalise column names to standard schema
        if 'p_malignant' not in lr_audit.columns and 'p_mal' in lr_audit.columns:
            lr_audit = lr_audit.rename(columns={'p_mal': 'p_malignant'})
        if 'y_true' not in lr_audit.columns and 'true_label' in lr_audit.columns:
            lr_audit = lr_audit.rename(columns={'true_label': 'y_true'})
        if 'patient_id' not in lr_audit.columns:
            lr_audit['patient_id'] = -1
        lr_audit['threshold'] = 0.5
        results['lr_baseline'] = lr_audit
        print(f"  ✅ lr_baseline loaded from audit CSV  ({len(lr_audit)} images)")
    else:
        print(f"  ⚠️  lr_probs_test_image_audit.csv not found — LR baseline skipped")

    if not results:
        raise RuntimeError(
            "No models could be loaded. Check that Step 7 checkpoints exist in "
            f"{CKPT_DIR}/{{fusion,image_only,clinical_only}}/best_auc_pr.pth"
        )
    return results


def aggregate_to_image_level(df: pd.DataFrame) -> pd.DataFrame:
    """Mean-pool ROI probabilities to image level."""
    # lr_baseline is already image-level — return as-is
    if 'source_image' not in df.columns:
        return df
    agg = (
        df.groupby('source_image')
          .agg(
              p_malignant=('p_malignant', 'mean'),
              y_true=('y_true', 'first'),
              patient_id=('patient_id', 'first'),
              threshold=('threshold', 'first'),
          )
          .reset_index()
    )
    return agg


def aggregate_to_patient_level(df: pd.DataFrame) -> pd.DataFrame:
    """Mean-pool image-level probabilities to patient level."""
    img_df = aggregate_to_image_level(df)
    pat_df = (
        img_df.groupby('patient_id')
              .agg(
                  p_malignant=('p_malignant', 'mean'),
                  y_true=('y_true', 'first'),
              )
              .reset_index()
    )
    return pat_df


# =============================================================================
# SECTION A: DELONG TEST
# =============================================================================

def _delong_placement_values(y_true, y_score):
    pos_idx    = np.where(y_true == 1)[0]
    neg_idx    = np.where(y_true == 0)[0]
    n1, n0     = len(pos_idx), len(neg_idx)
    pos_scores = y_score[pos_idx]
    neg_scores = y_score[neg_idx]
    diff       = pos_scores[:, None] - neg_scores[None, :]   # (n1, n0)
    psi        = (diff > 0).astype(float) + 0.5 * (diff == 0).astype(float)
    V10        = psi.mean(axis=1)
    V01        = psi.mean(axis=0)
    theta      = psi.mean()
    return theta, V10, V01, n1, n0


def delong_roc_variance(y_true, y_score):
    theta, V10, V01, n1, n0 = _delong_placement_values(y_true, y_score)
    s10 = np.var(V10, ddof=1) / n1
    s01 = np.var(V01, ddof=1) / n0
    return theta, s10 + s01


def delong_test(y_true, y_score_a, y_score_b):
    theta_a, V10_a, V01_a, n1, n0 = _delong_placement_values(y_true, y_score_a)
    theta_b, V10_b, V01_b, _,  _  = _delong_placement_values(y_true, y_score_b)

    var_a  = np.var(V10_a, ddof=1) / n1 + np.var(V01_a, ddof=1) / n0
    var_b  = np.var(V10_b, ddof=1) / n1 + np.var(V01_b, ddof=1) / n0
    cov10  = np.cov(V10_a, V10_b, ddof=1)[0, 1] / n1
    cov01  = np.cov(V01_a, V01_b, ddof=1)[0, 1] / n0
    cov_ab = cov10 + cov01

    var_diff = var_a + var_b - 2 * cov_ab
    se_diff  = np.sqrt(max(var_diff, 1e-12))
    diff     = theta_a - theta_b
    z_stat   = diff / se_diff
    p_val    = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    z_alpha = stats.norm.ppf(1 - ALPHA / 2)
    se_a    = np.sqrt(var_a)
    se_b    = np.sqrt(var_b)
    return {
        'auc_a':     float(theta_a),
        'auc_b':     float(theta_b),
        'ci_a':      (float(theta_a - z_alpha * se_a), float(theta_a + z_alpha * se_a)),
        'ci_b':      (float(theta_b - z_alpha * se_b), float(theta_b + z_alpha * se_b)),
        'diff':      float(diff),
        'ci_diff':   (float(diff - z_alpha * se_diff), float(diff + z_alpha * se_diff)),
        'se_diff':   float(se_diff),
        'z_stat':    float(z_stat),
        'p_value':   float(p_val),
        'n_pos':     int(n1),
        'n_neg':     int(n0),
        'significant': bool(p_val < ALPHA),
    }


def run_delong_analysis(results: dict) -> dict:
    print("\n" + "=" * 70)
    print("  A. DELONG TEST — PAIRWISE AUC-ROC COMPARISONS")
    print("=" * 70)

    img_data = {mt: aggregate_to_image_level(df) for mt, df in results.items()}

    # Align all models to common images
    common = None
    for agg in img_data.values():
        imgs   = set(agg['source_image'])
        common = imgs if common is None else common & imgs
    print(f"\n  Common test images: {len(common)}")

    aligned = {}
    for mt, agg in img_data.items():
        aligned[mt] = (
            agg[agg['source_image'].isin(common)]
            .sort_values('source_image')
            .reset_index(drop=True)
        )

    ref_df = next(iter(aligned.values()))
    y_true = ref_df['y_true'].values
    n_pos  = int((y_true == 1).sum())
    n_neg  = int((y_true == 0).sum())
    print(f"  n_pos={n_pos}  n_neg={n_neg}  imbalance={n_neg/max(n_pos,1):.1f}:1")

    # Individual AUCs with DeLong CI
    print(f"\n  Individual AUC-ROC (DeLong 95% CI):")
    print(f"  {'Model':<22} {'AUC':>8}  {'95% CI':>22}")
    print(f"  {'-'*56}")
    individual = {}
    z_alpha = stats.norm.ppf(1 - ALPHA / 2)
    for mt, sub in aligned.items():
        y_score = sub['p_malignant'].values
        if len(np.unique(y_true)) < 2:
            continue
        try:
            auc = roc_auc_score(y_true, y_score)
        except Exception:
            continue
        _, var  = delong_roc_variance(y_true, y_score)
        se      = np.sqrt(var)
        ci_lo   = auc - z_alpha * se
        ci_hi   = auc + z_alpha * se
        individual[mt] = {'auc': auc, 'ci': (ci_lo, ci_hi), 'se': se}
        name = DISPLAY_NAMES.get(mt, mt)
        print(f"  {name:<22} {auc:.4f}   [{ci_lo:.4f}, {ci_hi:.4f}]")

    # Pairwise DeLong
    print(f"\n  Pairwise DeLong tests (alpha={ALPHA}):")
    model_list = list(aligned.keys())
    pairs      = list(itertools.combinations(model_list, 2))
    pairwise   = {}
    print(f"  {'Comparison':<38} {'ΔAUC':>8} {'95% CI':>22} {'z':>7} {'p':>10} {'Sig':>4}")
    print(f"  {'-'*92}")

    for mt_a, mt_b in pairs:
        ys_a = aligned[mt_a]['p_malignant'].values
        ys_b = aligned[mt_b]['p_malignant'].values
        try:
            res = delong_test(y_true, ys_a, ys_b)
        except Exception as e:
            print(f"  ⚠️  DeLong failed for {mt_a} vs {mt_b}: {e}")
            continue
        key  = f"{mt_a}_vs_{mt_b}"
        pairwise[key] = res
        sig   = "✅" if res['significant'] else "—"
        p_str = f"{res['p_value']:.4f}" if res['p_value'] >= 0.0001 else "<0.0001"
        name_a = DISPLAY_NAMES.get(mt_a, mt_a)
        name_b = DISPLAY_NAMES.get(mt_b, mt_b)
        print(f"  {name_a} vs {name_b:<16} {res['diff']:+.4f}   "
              f"[{res['ci_diff'][0]:+.4f}, {res['ci_diff'][1]:+.4f}]  "
              f"{res['z_stat']:+.3f}  {p_str:>10}   {sig}")

    delong_out = {
        'n_test_images': int(len(common)),
        'n_pos': n_pos, 'n_neg': n_neg,
        'individual_auc': {
            mt: {'auc': float(v['auc']),
                 'ci_lo': float(v['ci'][0]),
                 'ci_hi': float(v['ci'][1])}
            for mt, v in individual.items()
        },
        'pairwise': pairwise,
        'alpha': ALPHA,
    }
    with open(Path(OUTPUT_DIR) / "delong_results.json", 'w') as f:
        json.dump(delong_out, f, indent=2)
    print(f"\n  ✅ delong_results.json saved")

    _plot_auc_comparison(individual, pairwise, aligned, y_true)
    return delong_out


def _plot_auc_comparison(individual, pairwise, aligned, y_true):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("DeLong AUC-ROC Comparison", fontsize=12, fontweight='bold')

    ax = axes[0]
    models  = list(individual.keys())
    aucs    = [individual[m]['auc']    for m in models]
    ci_los  = [individual[m]['ci'][0]  for m in models]
    ci_his  = [individual[m]['ci'][1]  for m in models]
    y_pos   = np.arange(len(models))
    colors  = [COLORS.get(m, '#555555') for m in models]
    names   = [DISPLAY_NAMES.get(m, m)  for m in models]

    for i, (auc, lo, hi, col) in enumerate(zip(aucs, ci_los, ci_his, colors)):
        ax.plot([lo, hi], [i, i], '-', color=col, lw=2.5, alpha=0.8)
        ax.plot(auc, i, 'o', color=col, ms=9, zorder=5)
        ax.text(hi + 0.002, i, f"{auc:.3f}", va='center', fontsize=9, color=col)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel("AUC-ROC (95% CI, DeLong)", fontsize=10)
    ax.set_title("Individual AUC with 95% CI", fontsize=10)
    ax.axvline(0.5, ls='--', color='gray', lw=1, alpha=0.5, label='Chance')
    ax.set_xlim(max(0, min(ci_los) - 0.05), min(1.05, max(ci_his) + 0.08))
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

    ax2 = axes[1]
    for mt, sub in aligned.items():
        ys = sub['p_malignant'].values
        if len(np.unique(y_true)) < 2 or mt not in individual:
            continue
        fpr, tpr, _ = roc_curve(y_true, ys)
        auc = individual[mt]['auc']
        ci  = individual[mt]['ci']
        name = DISPLAY_NAMES.get(mt, mt)
        ax2.plot(fpr, tpr, color=COLORS.get(mt, '#555'),
                 lw=2, label=f"{name}  AUC={auc:.3f} [{ci[0]:.3f},{ci[1]:.3f}]")

    ax2.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
    ax2.set_xlabel("1 - Specificity (FPR)", fontsize=10)
    ax2.set_ylabel("Sensitivity (TPR)", fontsize=10)
    ax2.set_title("ROC Curves with DeLong CI in Legend", fontsize=10)
    ax2.legend(fontsize=8, loc='lower right')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "figA_delong_auc_comparison.png",
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ figA_delong_auc_comparison.png")


# =============================================================================
# SECTION B: CALIBRATION ANALYSIS
# =============================================================================

def compute_ece(y_true, y_prob, n_bins=10):
    bins      = np.linspace(0, 1, n_bins + 1)
    ece       = 0.0
    n         = len(y_true)
    bin_stats = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask   = (y_prob >= lo) & (y_prob < hi)
        if i == n_bins - 1:
            mask = (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0:
            bin_stats.append({'lo': lo, 'hi': hi, 'n': 0,
                               'acc': np.nan, 'conf': np.nan, 'gap': np.nan})
            continue
        n_b  = mask.sum()
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        gap  = abs(acc - conf)
        ece += (n_b / n) * gap
        bin_stats.append({'lo': lo, 'hi': hi, 'n': int(n_b),
                          'acc': float(acc), 'conf': float(conf), 'gap': float(gap)})
    return float(ece), bin_stats


def compute_mce(bin_stats):
    gaps = [b['gap'] for b in bin_stats if not np.isnan(b.get('gap', np.nan))]
    return float(max(gaps)) if gaps else float('nan')


def brier_skill_score(y_true, y_prob):
    bs     = brier_score_loss(y_true, y_prob)
    prev   = y_true.mean()
    bs_ref = brier_score_loss(y_true, np.full_like(y_prob, prev))
    bss    = 1 - bs / (bs_ref + 1e-12)
    return float(bs), float(bss)


def run_calibration_analysis(results: dict) -> dict:
    print("\n" + "=" * 70)
    print("  B. CALIBRATION ANALYSIS")
    print("=" * 70)

    img_data = {mt: aggregate_to_image_level(df) for mt, df in results.items()}
    common   = None
    for agg in img_data.values():
        imgs   = set(agg['source_image'])
        common = imgs if common is None else common & imgs

    calib_results = {}
    print(f"\n  {'Model':<22} {'ECE':>8} {'MCE':>8} {'Brier':>8} {'BSS':>8}")
    print(f"  {'-'*58}")

    for mt, agg in img_data.items():
        sub    = agg[agg['source_image'].isin(common)].sort_values('source_image')
        y_true = sub['y_true'].values
        y_prob = sub['p_malignant'].values

        ece, bin_stats = compute_ece(y_true, y_prob, n_bins=N_BINS_ECE)
        mce            = compute_mce(bin_stats)
        bs, bss        = brier_skill_score(y_true, y_prob)

        ece_boot = []
        for _ in range(N_BOOTSTRAP):
            idx = rng.integers(0, len(y_true), size=len(y_true))
            if len(np.unique(y_true[idx])) < 2:
                continue
            e, _ = compute_ece(y_true[idx], y_prob[idx], n_bins=N_BINS_ECE)
            ece_boot.append(e)
        ece_ci = (
            float(np.percentile(ece_boot, 2.5)) if ece_boot else float('nan'),
            float(np.percentile(ece_boot, 97.5)) if ece_boot else float('nan'),
        )

        calib_results[mt] = {
            'ece': ece, 'ece_ci': ece_ci, 'mce': mce,
            'brier': bs, 'bss': bss,
            'bin_stats': bin_stats,
            'y_true': y_true, 'y_prob': y_prob,
        }
        name = DISPLAY_NAMES.get(mt, mt)
        print(f"  {name:<22} {ece:.4f}   {mce:.4f}   {bs:.4f}   {bss:+.4f}")

    print(f"\n  Interpretation:")
    print(f"    ECE < 0.05 → well calibrated  |  ECE > 0.15 → poorly calibrated")
    print(f"    BSS > 0    → better than always predicting prevalence")

    metrics_out = {
        mt: {k: v for k, v in d.items() if k not in ('bin_stats', 'y_true', 'y_prob')}
        for mt, d in calib_results.items()
    }
    with open(Path(OUTPUT_DIR) / "calibration_metrics.json", 'w') as f:
        json.dump(metrics_out, f, indent=2)
    print(f"\n  ✅ calibration_metrics.json saved")

    _plot_reliability_diagrams(calib_results)
    _plot_probability_distributions(calib_results)
    return calib_results


def _plot_reliability_diagrams(calib_results: dict):
    n_models = len(calib_results)
    fig, axes = plt.subplots(1, n_models + 1,
                              figsize=(4.5 * (n_models + 1), 4.5))
    fig.suptitle("Reliability Diagrams (Calibration Curves)",
                 fontsize=12, fontweight='bold')
    model_list = list(calib_results.keys())

    for i, mt in enumerate(model_list):
        ax     = axes[i]
        d      = calib_results[mt]
        y_true = d['y_true']
        y_prob = d['y_prob']
        ece    = d['ece']
        bs     = d['brier']
        color  = COLORS.get(mt, '#555555')
        name   = DISPLAY_NAMES.get(mt, mt)

        try:
            frac_pos, mean_pred = calibration_curve(
                y_true, y_prob, n_bins=N_BINS_PLOT, strategy='uniform'
            )
        except Exception:
            ax.text(0.5, 0.5, "Calibration curve failed",
                    ha='center', va='center', transform=ax.transAxes)
            continue

        ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.6, label='Perfect')
        ax.plot(mean_pred, frac_pos, 'o-', color=color, lw=2, ms=6,
                label=f"ECE={ece:.3f}\nBrier={bs:.3f}")
        ax.fill_between(mean_pred, mean_pred, frac_pos, alpha=0.15, color=color)
        ax.set_xlabel("Mean Predicted Probability", fontsize=9)
        ax.set_ylabel("Fraction Positive (True)", fontsize=9)
        ax.set_title(name, fontsize=10, fontweight='bold')
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.legend(fontsize=8, loc='upper left'); ax.grid(alpha=0.3)

        ax_ins = ax.inset_axes([0.60, 0.07, 0.38, 0.25])
        ax_ins.hist(y_prob[y_true == 0], bins=20, alpha=0.5,
                    color='royalblue', label='Benign', density=True)
        ax_ins.hist(y_prob[y_true == 1], bins=20, alpha=0.5,
                    color='tomato', label='Malignant', density=True)
        ax_ins.set_xlabel("p", fontsize=6)
        ax_ins.tick_params(labelsize=5)
        ax_ins.set_title("Score dist.", fontsize=6)

    ax_all = axes[-1]
    ax_all.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.6, label='Perfect')
    for mt, d in calib_results.items():
        try:
            frac_pos, mean_pred = calibration_curve(
                d['y_true'], d['y_prob'], n_bins=N_BINS_PLOT, strategy='uniform'
            )
        except Exception:
            continue
        ax_all.plot(mean_pred, frac_pos, 'o-', color=COLORS.get(mt, '#555'),
                    lw=2, ms=5,
                    label=f"{DISPLAY_NAMES.get(mt,mt)} (ECE={d['ece']:.3f})")

    ax_all.set_xlabel("Mean Predicted Probability", fontsize=9)
    ax_all.set_ylabel("Fraction Positive", fontsize=9)
    ax_all.set_title("All Models Compared", fontsize=10, fontweight='bold')
    ax_all.legend(fontsize=8, loc='upper left')
    ax_all.set_xlim(-0.02, 1.02); ax_all.set_ylim(-0.02, 1.02)
    ax_all.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "figB1_reliability_diagrams.png",
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ figB1_reliability_diagrams.png")


def _plot_probability_distributions(calib_results: dict):
    n_models = len(calib_results)
    fig, axes = plt.subplots(1, n_models, figsize=(4.5 * n_models, 4))
    if n_models == 1:
        axes = [axes]
    fig.suptitle("Predicted Probability Distributions by Class",
                 fontsize=12, fontweight='bold')

    for ax, (mt, d) in zip(axes, calib_results.items()):
        y_true = d['y_true']
        y_prob = d['y_prob']
        name   = DISPLAY_NAMES.get(mt, mt)
        bins   = np.linspace(0, 1, 25)

        ax.hist(y_prob[y_true == 0], bins=bins, alpha=0.6,
                color='royalblue', label='Benign')
        ax.hist(y_prob[y_true == 1], bins=bins, alpha=0.6,
                color='tomato', label='Malignant')
        ax.axvline(y_prob[y_true == 0].mean(), color='royalblue', ls='--', lw=1.5,
                   label=f"Ben mean={y_prob[y_true==0].mean():.3f}")
        ax.axvline(y_prob[y_true == 1].mean(), color='tomato', ls='--', lw=1.5,
                   label=f"Mal mean={y_prob[y_true==1].mean():.3f}")
        ax.set_title(f"{name}\nECE={d['ece']:.3f}  Brier={d['brier']:.3f}",
                     fontsize=9)
        ax.set_xlabel("p(malignant)", fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "figB2_probability_distributions.png",
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ figB2_probability_distributions.png")


# =============================================================================
# SECTION C: ERROR ANALYSIS
# =============================================================================

def run_error_analysis(results: dict, calib_results: dict) -> dict:
    print("\n" + "=" * 70)
    print("  C. ERROR ANALYSIS — FUSION vs IMAGE-ONLY")
    print("=" * 70)

    if 'fusion' not in results or 'image_only' not in results:
        print("  ⚠️  Need both 'fusion' and 'image_only' results. Skipping.")
        return {}

    # Load clinical features for case-level interpretation
    clinical_bone_features = [
        "ulna", "radius", "humerus", "femur", "tibia",
        "fibula", "hip bone", "hand", "foot",
    ]
    paired_df = None
    if Path(PAIRED_DIR, "fusion_paired_test.csv").exists():
        paired_df = pd.read_csv(Path(PAIRED_DIR) / "fusion_paired_test.csv")
        print(f"  Clinical features loaded: {len(paired_df)} ROIs")

    # Thresholds from inference results (loaded from checkpoints)
    def get_threshold(mt):
        df = results.get(mt)
        if df is not None and 'threshold' in df.columns:
            return float(df['threshold'].iloc[0])
        return 0.5

    thr_fusion = get_threshold('fusion')
    thr_image  = get_threshold('image_only')
    print(f"  Thresholds: fusion={thr_fusion:.3f}  image_only={thr_image:.3f}")

    fus_img = aggregate_to_image_level(results['fusion'])
    img_img = aggregate_to_image_level(results['image_only'])

    common = set(fus_img['source_image']) & set(img_img['source_image'])
    fus_sub = (fus_img[fus_img['source_image'].isin(common)]
               .sort_values('source_image').reset_index(drop=True))
    img_sub = (img_img[img_img['source_image'].isin(common)]
               .sort_values('source_image').reset_index(drop=True))

    assert (fus_sub['source_image'].values == img_sub['source_image'].values).all()

    y_true       = fus_sub['y_true'].values
    p_fusion     = fus_sub['p_malignant'].values
    p_image_only = img_sub['p_malignant'].values
    source_images = fus_sub['source_image'].values

    pred_fusion = (p_fusion     >= thr_fusion).astype(int)
    pred_image  = (p_image_only >= thr_image ).astype(int)

    correct_fusion = (pred_fusion == y_true)
    correct_image  = (pred_image  == y_true)

    cat_both_correct = correct_fusion  & correct_image
    cat_corrected    = correct_fusion  & ~correct_image
    cat_worsened     = ~correct_fusion & correct_image
    cat_both_wrong   = ~correct_fusion & ~correct_image

    n_total = len(y_true)
    print(f"\n  Decision comparison (image level):")
    print(f"\n  {'Category':<42} {'N':>5} {'%':>7}")
    print(f"  {'-'*56}")
    print(f"  {'Both correct':<42} {cat_both_correct.sum():>5}  {100*cat_both_correct.sum()/n_total:>5.1f}%")
    print(f"  {'Corrected by fusion (Fusion ✅, Image ❌)':<42} {cat_corrected.sum():>5}  {100*cat_corrected.sum()/n_total:>5.1f}%")
    print(f"  {'Worsened by fusion (Fusion ❌, Image ✅)':<42} {cat_worsened.sum():>5}  {100*cat_worsened.sum()/n_total:>5.1f}%")
    print(f"  {'Both wrong':<42} {cat_both_wrong.sum():>5}  {100*cat_both_wrong.sum()/n_total:>5.1f}%")

    def describe_cases(mask, label):
        print(f"\n  ── {label} ──")
        if mask.sum() == 0:
            print(f"  (no cases)")
            return []
        cases = []
        for i in np.where(mask)[0]:
            src   = source_images[i]
            truth = "Malignant" if y_true[i] == 1 else "Benign"
            clin_row = None
            if paired_df is not None:
                rows = paired_df[paired_df['source_image'] == src]
                if len(rows) > 0:
                    clin_row = rows.iloc[0]
            case = {
                'source_image': src,
                'true_label':   truth,
                'p_fusion':     float(p_fusion[i]),
                'p_image_only': float(p_image_only[i]),
                'p_delta':      float(p_fusion[i] - p_image_only[i]),
            }
            if clin_row is not None:
                for feat in CLINICAL_FEATURES:
                    if feat in clin_row.index:
                        case[feat] = clin_row[feat]
            cases.append(case)

        cases_sorted = sorted(cases, key=lambda x: abs(x['p_delta']), reverse=True)
        for j, c in enumerate(cases_sorted[:10]):
            active_bones = [f for f in clinical_bone_features if c.get(f, 0) == 1]
            print(f"    [{j+1}] {c['source_image'][:42]:<42}  "
                  f"True={c['true_label']:<10}  "
                  f"p_fusion={c['p_fusion']:.3f}  "
                  f"p_image={c['p_image_only']:.3f}  "
                  f"Δ={c['p_delta']:+.3f}")
            if 'age' in c:
                print(f"         age={c.get('age','?')}  "
                      f"gender={c.get('gender','?')}  "
                      f"bones={active_bones}")
        return cases_sorted

    corrected_cases  = describe_cases(cat_corrected,  "CORRECTED BY FUSION")
    worsened_cases   = describe_cases(cat_worsened,   "WORSENED BY FUSION")
    both_wrong_cases = describe_cases(cat_both_wrong, "BOTH WRONG")

    # Error breakdown by true label
    print(f"\n  Error type breakdown by true label:")
    for lbl, mask in [("Corrected (Fusion ✅)", cat_corrected),
                       ("Worsened  (Fusion ❌)", cat_worsened)]:
        if mask.sum() == 0:
            continue
        n_mal = ((y_true == 1) & mask).sum()
        n_ben = ((y_true == 0) & mask).sum()
        print(f"  {lbl}: True Malignant={n_mal}  True Benign={n_ben}")

    error_out = {
        'thresholds':       {'fusion': thr_fusion, 'image_only': thr_image},
        'n_total':          int(n_total),
        'n_both_correct':   int(cat_both_correct.sum()),
        'n_corrected':      int(cat_corrected.sum()),
        'n_worsened':       int(cat_worsened.sum()),
        'n_both_wrong':     int(cat_both_wrong.sum()),
        'corrected_cases':  corrected_cases[:20],
        'worsened_cases':   worsened_cases[:20],
        'both_wrong_cases': both_wrong_cases[:20],
    }
    with open(Path(OUTPUT_DIR) / "error_analysis.json", 'w') as f:
        json.dump(error_out, f, indent=2, default=str)
    print(f"\n  ✅ error_analysis.json saved")

    _plot_error_analysis(
        y_true, p_fusion, p_image_only,
        cat_corrected, cat_worsened, cat_both_correct, cat_both_wrong,
        thr_fusion, thr_image,
    )
    return error_out


def _plot_error_analysis(y_true, p_fusion, p_image_only,
                          cat_corrected, cat_worsened,
                          cat_both_correct, cat_both_wrong,
                          thr_fusion, thr_image):
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.35)
    fig.suptitle("Error Analysis: FusionNet vs Image-Only MLP",
                 fontsize=13, fontweight='bold', y=1.01)

    cat_colors = {
        'both_correct': '#2ca02c',
        'corrected':    '#1f77b4',
        'worsened':     '#d62728',
        'both_wrong':   '#7f7f7f',
    }
    cat_labels = {
        'both_correct': f"Both correct (n={cat_both_correct.sum()})",
        'corrected':    f"Corrected by fusion (n={cat_corrected.sum()})",
        'worsened':     f"Worsened by fusion (n={cat_worsened.sum()})",
        'both_wrong':   f"Both wrong (n={cat_both_wrong.sum()})",
    }

    # Panel 1: Scatter
    ax1 = fig.add_subplot(gs[0, 0])
    for cat_key, mask in [('both_correct', cat_both_correct),
                           ('corrected', cat_corrected),
                           ('worsened', cat_worsened),
                           ('both_wrong', cat_both_wrong)]:
        if mask.sum() == 0:
            continue
        ax1.scatter(p_image_only[mask], p_fusion[mask],
                    c=cat_colors[cat_key], label=cat_labels[cat_key],
                    s=40, alpha=0.7, edgecolors='none')
    ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
    ax1.axhline(thr_fusion, ls=':', color='royalblue', lw=1.2, alpha=0.7,
                label=f"Fusion thr={thr_fusion:.3f}")
    ax1.axvline(thr_image, ls=':', color='tomato', lw=1.2, alpha=0.7,
                label=f"Image thr={thr_image:.3f}")
    ax1.set_xlabel("p(malignant) — Image-Only MLP", fontsize=10)
    ax1.set_ylabel("p(malignant) — FusionNet", fontsize=10)
    ax1.set_title("Decision Disagreement Map", fontsize=10, fontweight='bold')
    ax1.legend(fontsize=7, loc='upper left')
    ax1.set_xlim(-0.02, 1.02); ax1.set_ylim(-0.02, 1.02)
    ax1.grid(alpha=0.3)

    # Panel 2: Corrected cases
    ax2 = fig.add_subplot(gs[0, 1])
    if cat_corrected.sum() > 0:
        p_img_c = p_image_only[cat_corrected]
        p_fus_c = p_fusion[cat_corrected]
        lbls_c  = y_true[cat_corrected]
        for xi, si in enumerate(np.argsort(p_fus_c - p_img_c)[::-1]):
            col = 'tomato' if lbls_c[si] == 1 else 'royalblue'
            ax2.plot([xi, xi], [p_img_c[si], p_fus_c[si]], '-', color=col, lw=1.5, alpha=0.6)
            ax2.plot(xi, p_img_c[si], 'v', color=col, ms=5, alpha=0.5)
            ax2.plot(xi, p_fus_c[si], '^', color=col, ms=5, alpha=1.0)
        ax2.axhline(thr_fusion, ls='--', color='black', lw=1, alpha=0.5,
                    label=f"Fusion thr={thr_fusion:.3f}")
        ax2.set_title(f"CORRECTED cases (n={cat_corrected.sum()})\n"
                      "▼=Image-Only, ▲=FusionNet  (red=Malignant, blue=Benign)",
                      fontsize=9, fontweight='bold')
        ax2.set_xlabel("Case (sorted by Δp)", fontsize=9)
        ax2.set_ylabel("p(malignant)", fontsize=9)
        ax2.legend(fontsize=8); ax2.set_ylim(-0.02, 1.02); ax2.grid(alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "No corrected cases", ha='center', va='center',
                 fontsize=12, transform=ax2.transAxes)

    # Panel 3: Worsened cases
    ax3 = fig.add_subplot(gs[1, 0])
    if cat_worsened.sum() > 0:
        p_img_w = p_image_only[cat_worsened]
        p_fus_w = p_fusion[cat_worsened]
        lbls_w  = y_true[cat_worsened]
        for xi, si in enumerate(np.argsort(p_img_w - p_fus_w)[::-1]):
            col = 'tomato' if lbls_w[si] == 1 else 'royalblue'
            ax3.plot([xi, xi], [p_img_w[si], p_fus_w[si]], '-', color=col, lw=1.5, alpha=0.6)
            ax3.plot(xi, p_img_w[si], '^', color=col, ms=5, alpha=1.0)
            ax3.plot(xi, p_fus_w[si], 'v', color=col, ms=5, alpha=0.5)
        ax3.axhline(thr_fusion, ls='--', color='black', lw=1, alpha=0.5,
                    label=f"Fusion thr={thr_fusion:.3f}")
        ax3.set_title(f"WORSENED cases (n={cat_worsened.sum()})\n"
                      "▲=Image-Only, ▼=FusionNet  (red=Malignant, blue=Benign)",
                      fontsize=9, fontweight='bold')
        ax3.set_xlabel("Case (sorted by Δp)", fontsize=9)
        ax3.set_ylabel("p(malignant)", fontsize=9)
        ax3.legend(fontsize=8); ax3.set_ylim(-0.02, 1.02); ax3.grid(alpha=0.3)
    else:
        ax3.text(0.5, 0.5, "No worsened cases", ha='center', va='center',
                 fontsize=12, transform=ax3.transAxes)

    # Panel 4: Summary bar chart
    ax4    = fig.add_subplot(gs[1, 1])
    cats   = ['Both\nCorrect', 'Corrected\nby Fusion', 'Worsened\nby Fusion', 'Both\nWrong']
    counts = [cat_both_correct.sum(), cat_corrected.sum(),
              cat_worsened.sum(), cat_both_wrong.sum()]
    cols4  = [cat_colors[k] for k in
              ['both_correct', 'corrected', 'worsened', 'both_wrong']]
    bars   = ax4.bar(cats, counts, color=cols4, alpha=0.8,
                     edgecolor='white', lw=1.5)
    for bar, cnt in zip(bars, counts):
        ax4.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.2,
                 f"{cnt}\n({100*cnt/len(y_true):.1f}%)",
                 ha='center', va='bottom', fontsize=9)
    ax4.set_ylabel("Number of images", fontsize=10)
    ax4.set_title("Decision Category Summary", fontsize=10, fontweight='bold')
    ax4.set_ylim(0, max(counts) * 1.3)
    ax4.grid(axis='y', alpha=0.3)

    plt.savefig(Path(OUTPUT_DIR) / "figC1_error_analysis.png",
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✅ figC1_error_analysis.png")


# =============================================================================
# SECTION D: SUMMARY TABLE
# =============================================================================

def build_summary_table(delong_out: dict, calib_results: dict,
                         error_out: dict) -> pd.DataFrame:
    rows = []
    for mt in MODEL_TYPES:
        name   = DISPLAY_NAMES.get(mt, mt)
        ind    = delong_out.get('individual_auc', {}).get(mt, {})
        auc    = ind.get('auc',   float('nan'))
        ci_lo  = ind.get('ci_lo', float('nan'))
        ci_hi  = ind.get('ci_hi', float('nan'))
        cal    = calib_results.get(mt, {})
        ece    = cal.get('ece',   float('nan'))
        ece_ci = cal.get('ece_ci', (float('nan'), float('nan')))
        brier  = cal.get('brier', float('nan'))
        bss    = cal.get('bss',   float('nan'))
        rows.append({
            'Model':             name,
            'AUC-ROC':           f"{auc:.3f}" if not np.isnan(auc) else 'N/A',
            'AUC-ROC 95% CI':    f"[{ci_lo:.3f}, {ci_hi:.3f}]" if not np.isnan(ci_lo) else 'N/A',
            'ECE':               f"{ece:.4f}" if not np.isnan(ece) else 'N/A',
            'ECE 95% CI':        f"[{ece_ci[0]:.4f}, {ece_ci[1]:.4f}]",
            'Brier Score':       f"{brier:.4f}" if not np.isnan(brier) else 'N/A',
            'Brier Skill Score': f"{bss:+.4f}" if not np.isnan(bss) else 'N/A',
        })

    df = pd.DataFrame(rows)
    print(f"\n  Pairwise DeLong significance:")
    print(f"  {'Comparison':<40} {'p-value':>10} {'Sig':>5}")
    print(f"  {'-'*58}")
    for key, res in delong_out.get('pairwise', {}).items():
        pv    = res['p_value']
        sig   = "✅ p<0.05" if pv < 0.05 else "—"
        p_str = f"{pv:.4f}" if pv >= 0.0001 else "<0.0001"
        print(f"  {key.replace('_vs_', ' vs '):<40} {p_str:>10}   {sig}")

    return df


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 70)
    print("  STEP 10B: DEEP STATISTICAL ANALYSIS")
    print("=" * 70)
    print(f"\n  Output directory : {OUTPUT_DIR}")
    print(f"  Features         : {len(CLINICAL_FEATURES)} clinical features (CLIN_FEAT_DIM={CLIN_FEAT_DIM} ✅)")
    print(f"  N_BOOTSTRAP      : {N_BOOTSTRAP}  |  N_BINS_ECE: {N_BINS_ECE}")
    print(f"  Architecture     : ConvNeXt-Tiny-SE 768-dim → 256 → 128 (learned projection)")

    # ── Load / infer ─────────────────────────────────────────────────────────
    results = load_results()

    if not results:
        print("\n❌ No results loaded.")
        return

    # A. DeLong
    delong_out = run_delong_analysis(results)

    # B. Calibration
    calib_results = run_calibration_analysis(results)

    # C. Error analysis
    error_out = run_error_analysis(results, calib_results)

    # D. Summary table
    print("\n" + "=" * 70)
    print("  D. COMPREHENSIVE SUMMARY TABLE")
    print("=" * 70)
    summary_df = build_summary_table(delong_out, calib_results, error_out)
    print(f"\n{summary_df.to_string(index=False)}")
    summary_df.to_csv(Path(OUTPUT_DIR) / "statistical_summary_table.csv", index=False)
    print(f"\n  ✅ statistical_summary_table.csv saved")

    print(f"""
{'='*70}
  STEP 10B COMPLETE
{'='*70}

  Architecture: ConvNeXt-Tiny-SE  768 → 256 → 128 (learned) + Clinical 24 → 64 → 32
  Features    : {len(CLINICAL_FEATURES)} clinical features + 1 LR prob = 24-dim clinical input
  Fusion dim  : IMG_EMBED(128) + CLIN_EMBED(32) = 160-dim

  Outputs saved to: {OUTPUT_DIR}/
    A. DeLong:       delong_results.json + figA_delong_auc_comparison.png
    B. Calibration:  calibration_metrics.json + figB1_reliability_diagrams.png
                     + figB2_probability_distributions.png
    C. Error:        error_analysis.json + figC1_error_analysis.png
    D. Summary:      statistical_summary_table.csv
""")


if __name__ == "__main__":
    np.random.seed(RANDOM_SEED)
    main()

  STEP 10B: DEEP STATISTICAL ANALYSIS

  Output directory : fusion_outputs/step10b_statistical
  Features         : 23 clinical features (CLIN_FEAT_DIM=23 ✅)
  N_BOOTSTRAP      : 2000  |  N_BINS_ECE: 10
  Architecture     : ConvNeXt-Tiny-SE 768-dim → 256 → 128 (learned projection)

[0] Running inference from Step 7 checkpoints (no_aug features)...
  Device: cuda
  TEST ROIs   : 283
  TEST images : 241
  lr_probs shape: (283, 1)
  ✅ fusion           ROIs=283  malignant=41  threshold=0.610  val AUC-PR=0.9841
  ✅ image_only       ROIs=283  malignant=41  threshold=0.650  val AUC-PR=0.9841
  ✅ clinical_only    ROIs=283  malignant=41  threshold=0.630  val AUC-PR=0.5658
  ✅ lr_baseline loaded from audit CSV  (241 images)

  A. DELONG TEST — PAIRWISE AUC-ROC COMPARISONS

  Common test images: 241
  n_pos=41  n_neg=200  imbalance=4.9:1

  Individual AUC-ROC (DeLong 95% CI):
  Model                       AUC                  95% CI
  --------------------------------------------------------
  Fus

In [20]:
import shutil, os

src = "/kaggle/working/fusion_outputs"
zip_base = "/kaggle/working/fusion_outputs"  # no .zip here

# Creates: /kaggle/working/opt1_extra_visualizations_all.zip
shutil.make_archive(zip_base, "zip", root_dir=src)
print("Created:", zip_base + ".zip")


Created: /kaggle/working/fusion_outputs.zip
